# 🌀 Hurricane Path Prediction & ⚡Infrastructure Impact Analysis 

This notebook demonstrates the capabilities of **[Microsoft Planetary Computer Pro](https://azure.microsoft.com/en-us/products/planetary-computer-pro)** — a unified platform that combines geospatial data with enterprise AI and analytics. 

Using Planetary Computer Pro's unified STAC catalog, cloud-optimized data access, and seamless Azure integration, this workflow showcases hurricane path prediction and infrastructure impact analysis powered by Microsoft's **Aurora AI weather foundation model**.

The notebook is designed to work with **any historical or current tropical storm** from the IBTrACS database. Hurricane Helene (September 2024) is selected as the default storm for a ready-to-run experience.

### Key Planetary Computer Pro Features Demonstrated
- 🌐 **Unified STAC Catalog** - Standardized access to ECMWF weather data
- ☁️ **Cloud-Optimized Data** - Efficient retrieval of large-scale geospatial datasets
- 🔗 **Azure Integration** - Seamless connection with Microsoft Foundry for model inference
- 📊 **Scalable Analytics** - Process petabytes of weather data for predictions

## Overview

1. **Environment Setup** - Load credentials and configure Azure services
2. **Storm Data Download** - Fetch storm track data from IBTrACS (select any storm or use default: Helene)
3. **ECMWF Data Download** - Retrieve weather model input data via Microsoft Planetary Computer Pro
4. **ERA5 Static Variables** - Download static geographic data from Copernicus Climate Data Store
5. **Aurora Batch Preparation** - Format data for Aurora model inference
6. **Aurora Model Inference** - Run hurricane path predictions using the Aurora weather foundation model
7. **Cyclone Track Visualization** - Plot predicted vs actual storm paths
8. **Infrastructure Impact Analysis** - Analyze affected power grid and infrastructure using OpenStreetMap/OpenInfraMap data

## Pre-Requisites

Before running this notebook, you need to deploy the required Azure resources. The deployment includes:

- **Azure Storage Account** - For storing weather data, forecasts, and intermediate results
- **Azure AI Hub & Project** - For hosting the Aurora weather model endpoint
- **Microsoft Planetary Computer Pro (GeoCatalog)** - For STAC API access to geospatial infrastructure data

> ⚠️ **Important**: Before deploying, make sure you have GPU VM quota in one of the supported regions (`northcentralus`, `eastus`, `canadacentral`, `westeurope`, `uksouth`). You can check quota in the Azure portal under **Subscriptions > Usage + quotas**. If you don't have GPU quota, you'll need to request an increase first.

> ⚠️ **Important**: After deployment, make sure to copy `.env.example` to `.env` and populate it with your resource connection strings and API keys.

### Option 1: One-Click Deployment (Recommended)

Click the button below to deploy all required resources to your Azure subscription:

[![Deploy to Azure](https://aka.ms/deploytoazurebutton)](https://portal.azure.com/#create/Microsoft.Template/uri/https%3A%2F%2Fraw.githubusercontent.com%2FAzure%2Fmicrosoft-planetary-computer-pro%2Fmain%2Fai_workflow%2Fmpc_aurora_workflow%2Fdeploy%2Fazuredeploy.json)

> **Note**: The **Deploy to Azure** button above is a clickable link that only works when viewing this notebook on [GitHub](https://github.com/Azure/microsoft-planetary-computer-pro/blob/main/ai_workflow/mpc_aurora_workflow/hurricane_forecast_infra_impact.ipynb). If you are running this notebook locally (e.g., in VS Code or JupyterLab), please open the GitHub page to use the one-click deployment, or use the manual deployment option below.

### Option 2: Manual Deployment

If you prefer to deploy resources manually, you can use either of the following approaches:

**Using Azure CLI:**
1. Install the [Azure CLI](https://learn.microsoft.com/cli/azure/install-azure-cli) if you haven't already
2. Navigate to the `deploy/` folder and deploy the ARM template:
   ```bash
   az deployment group create --resource-group <your-resource-group> --template-file azuredeploy.json
   ```
3. Copy `.env.example` to `.env` and fill in your resource credentials

**Using the Azure Portal:**
1. Go to [portal.azure.com](https://portal.azure.com) and manually provision the required resources (Storage Account, Azure AI Hub & Project, and GeoCatalog)
2. Copy `.env.example` to `.env` and fill in your resource credentials

## 1. Environment Setup

This section installs required packages and configures authentication for the following Azure services:

- **Microsoft Planetary Computer Pro** - GeoCatalog STAC API for geospatial data access
- **Microsoft Foundry** - Aurora weather model endpoint for inference
- **Azure Blob Storage** - Data upload and retrieval
- **Copernicus Climate Data Store (CDS)** - ERA5 static variables

⚠️ **Prerequisites**: Copy `.env.example` to `.env` and fill in your credentials before running.

In [ ]:
# Environment Setup: Copy .env.example to .env if needed, then install dependencies
import shutil
import os

if not os.path.exists('.env') and os.path.exists('.env.example'):
    shutil.copy('.env.example', '.env')
    print("✅ Created .env from .env.example - Please fill in your credentials!")
elif os.path.exists('.env'):
    print("✅ .env file already exists")
else:
    print("⚠️ No .env.example found - Please create a .env file manually")

# Install required packages
%pip install -r requirements.txt -q

In [ ]:
# Load environment variables and configure Azure authentication
import logging
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential

# Configure logging (INFO level for production)
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Load environment variables
load_dotenv()

# Azure authentication
SPATIO_APP_ID = "https://geocatalog.spatio.azure.com"
credential = DefaultAzureCredential()

# Load configuration from environment
GEOCATALOG_URI = os.getenv("GEOCATALOG_URI")
AURORA_FOUNDRY_ENDPOINT = os.getenv("AURORA_FOUNDRY_ENDPOINT")
AURORA_FOUNDRY_TOKEN = os.getenv("AURORA_FOUNDRY_TOKEN")
AURORA_BLOB_STORAGE_SAS = os.getenv("AURORA_BLOB_STORAGE_SAS")
OUTPUT_BLOB_CONTAINER_URL = os.getenv("OUTPUT_BLOB_CONTAINER_URL")
STORAGE_ACCOUNT_KEY = os.getenv("STORAGE_ACCOUNT_KEY")

# Derive API URLs from the base GeoCatalog URI
STAC_API_URL = f"{GEOCATALOG_URI}/stac"

# Validate required environment variables
required_vars = [
    "GEOCATALOG_URI", "AURORA_FOUNDRY_ENDPOINT", "AURORA_FOUNDRY_TOKEN",
    "AURORA_BLOB_STORAGE_SAS", "OUTPUT_BLOB_CONTAINER_URL"
]
missing_vars = [var for var in required_vars if not os.getenv(var)]
if missing_vars:
    raise ValueError(f"Missing required environment variables: {', '.join(missing_vars)}")

print("✅ Environment configured successfully")
print(f"   GeoCatalog: {GEOCATALOG_URI}")
print(f"   STAC API:   {STAC_API_URL}")
print(f"   Aurora Endpoint: {AURORA_FOUNDRY_ENDPOINT}")

## 2. Download Storm Data from IBTrACS

This section provides an interactive storm selector that loads tropical cyclone data from:

- **IBTrACS** (International Best Track Archive for Climate Stewardship) - Comprehensive historical storm database
- **HURDAT2** - NOAA's Atlantic/East Pacific hurricane database
- **Real-time feeds** - Active storms from NHC/JTWC

Select any historical storm or use the default (Hurricane Helene 2024) to proceed. The selector automatically finds the optimal initialization point for Aurora model inference based on storm intensity and track data.

📖 **Reference**: [IBTrACS Documentation](https://ncics.org/ibtracs/) | [Tropycal Library](https://tropycal.github.io/)

In [ ]:
# =============================================================================
# SECTION 2: Interactive Storm Selector
# Reference: https://tropycal.github.io/
# =============================================================================

import tropycal.tracks as tracks
import tropycal.realtime as realtime
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import json
import os
import requests
import io
import time

# =============================================================================
# CONFIGURATION
# =============================================================================
STORM_CACHE_DIR = os.path.join(os.getcwd(), "cache", "storm_data")
os.makedirs(STORM_CACHE_DIR, exist_ok=True)

# IBTrACS URLs - default first, v04r01 as fallback for recent data
IBTRACS_V04R01_BASE = "https://www.ncei.noaa.gov/data/international-best-track-archive-for-climate-stewardship-ibtracs/v04r01/access/csv/"
IBTRACS_URL_V04R01 = "https://www.ncei.noaa.gov/data/international-best-track-archive-for-climate-stewardship-ibtracs/v04r01/access/csv/ibtracs.(basin).list.v04r01.csv"

IBTRACS_BASIN_CODES = {
    'north_atlantic': 'NA', 'east_pacific': 'EP', 'west_pacific': 'WP',
    'north_indian': 'NI', 'south_indian': 'SI', 'south_pacific': 'SP', 'south_atlantic': 'SA',
}

BASINS_CONFIG = {
    'north_atlantic': {'display': 'ATL', 'source': 'hurdat', 'full_name': 'North Atlantic'},
    'east_pacific': {'display': 'EPAC', 'source': 'hurdat', 'full_name': 'East Pacific'},
    'west_pacific': {'display': 'WPAC', 'source': 'ibtracs', 'full_name': 'West Pacific'},
    'north_indian': {'display': 'NIO', 'source': 'ibtracs', 'full_name': 'North Indian'},
    'south_indian': {'display': 'SIO', 'source': 'ibtracs', 'full_name': 'South Indian'},
    'south_pacific': {'display': 'SPAC', 'source': 'ibtracs', 'full_name': 'South Pacific'},
    'south_atlantic': {'display': 'SATL', 'source': 'ibtracs', 'full_name': 'South Atlantic'},
}

current_year = datetime.now().year
years_to_include = sorted(set([2026, 2025, 2024, current_year]), reverse=True)

# =============================================================================
# CACHE UTILITIES
# =============================================================================
def get_cache_filename(basin_id, year):
    return os.path.join(STORM_CACHE_DIR, f"storms_{basin_id}_{year}.json")

def load_cached_storm_data(basin_id, year):
    cache_file = get_cache_filename(basin_id, year)
    if os.path.exists(cache_file):
        try:
            with open(cache_file, 'r') as f:
                data = json.load(f)
                cache_date = data.get('cache_date', '')
                today = datetime.now().strftime('%Y-%m-%d')
                if year < current_year or cache_date == today:
                    return data.get('storms', {})
        except: pass
    return None

def save_storm_data_to_cache(basin_id, year, storms_data):
    try:
        with open(get_cache_filename(basin_id, year), 'w') as f:
            json.dump({'cache_date': datetime.now().strftime('%Y-%m-%d'), 'basin_id': basin_id, 
                       'year': year, 'storms': storms_data}, f, indent=2, default=str)
    except: pass

# =============================================================================
# IBTRACS DIRECT DOWNLOAD (FALLBACK FOR RECENT DATA)
# =============================================================================
def load_ibtracs_direct(basin_id, years_filter=None):
    """Load storm data directly from IBTrACS v04r01 CSV files as fallback."""
    basin_code = IBTRACS_BASIN_CODES.get(basin_id)
    if not basin_code:
        return {}
    
    urls_to_try = [
        f"{IBTRACS_V04R01_BASE}ibtracs.last3years.list.v04r01.csv",
        f"{IBTRACS_V04R01_BASE}ibtracs.{basin_code}.list.v04r01.csv",
    ]
    
    for url in urls_to_try:
        try:
            response = requests.get(url, timeout=60)
            response.raise_for_status()
            df = pd.read_csv(io.StringIO(response.text), low_memory=False, skiprows=[1])
            
            if 'last3years' in url:
                df = df[df['BASIN'] == basin_code]
            if years_filter:
                df['YEAR'] = pd.to_datetime(df['ISO_TIME']).dt.year
                df = df[df['YEAR'].isin(years_filter)]
            if df.empty:
                continue
            
            storms = {}
            for sid in df['SID'].unique():
                storm_df = df[df['SID'] == sid].sort_values('ISO_TIME')
                name = storm_df['NAME'].iloc[0] if 'NAME' in storm_df.columns else 'UNNAMED'
                if pd.isna(name) or name == 'NOT_NAMED': name = 'UNNAMED'
                year = pd.to_datetime(storm_df['ISO_TIME'].iloc[0]).year
                
                vmax = 0
                for wind_col in ['USA_WIND', 'WMO_WIND']:
                    if wind_col in storm_df.columns:
                        wind_vals = pd.to_numeric(storm_df[wind_col], errors='coerce')
                        if not wind_vals.isna().all():
                            vmax = int(wind_vals.max())
                            break
                
                storms[sid] = {
                    'name': name, 'storm_id': sid, 'year': year, 'basin_id': basin_id, 'vmax': vmax,
                    'made_landfall': 'LANDFALL' in storm_df.columns and storm_df['LANDFALL'].notna().any(),
                    'source': 'ibtracs_v04r01',
                    'track': {
                        'time': storm_df['ISO_TIME'].tolist(),
                        'lat': pd.to_numeric(storm_df['LAT'], errors='coerce').tolist(),
                        'lon': pd.to_numeric(storm_df['LON'], errors='coerce').tolist(),
                        'vmax': pd.to_numeric(storm_df.get('USA_WIND', storm_df.get('WMO_WIND', pd.Series())), errors='coerce').tolist(),
                        'mslp': pd.to_numeric(storm_df.get('USA_PRES', storm_df.get('WMO_PRES', pd.Series())), errors='coerce').tolist(),
                    }
                }
            return storms
        except: continue
    return {}

def load_active_storms_from_ibtracs():
    """Load currently active storms from IBTrACS ACTIVE file."""
    try:
        response = requests.get(f"{IBTRACS_V04R01_BASE}ibtracs.ACTIVE.list.v04r01.csv", timeout=30)
        response.raise_for_status()
        df = pd.read_csv(io.StringIO(response.text), low_memory=False, skiprows=[1])
        if df.empty: return {}
        
        storms = {}
        for sid in df['SID'].unique():
            storm_df = df[df['SID'] == sid].sort_values('ISO_TIME')
            name = storm_df['NAME'].iloc[0] if 'NAME' in storm_df.columns else 'UNNAMED'
            if pd.isna(name) or name == 'NOT_NAMED': name = 'UNNAMED'
            basin_code = storm_df['BASIN'].iloc[0]
            basin_id = next((k for k, v in IBTRACS_BASIN_CODES.items() if v == basin_code), 'unknown')
            
            vmax = 0
            for wind_col in ['USA_WIND', 'WMO_WIND']:
                if wind_col in storm_df.columns:
                    wind_vals = pd.to_numeric(storm_df[wind_col], errors='coerce')
                    if not wind_vals.isna().all():
                        vmax = int(wind_vals.max())
                        break
            
            storms[sid] = {
                'name': name, 'storm_id': sid, 'basin_id': basin_id, 'vmax': vmax, 'is_active': True,
                'source': 'ibtracs_active',
                'track': {
                    'time': storm_df['ISO_TIME'].tolist(),
                    'lat': pd.to_numeric(storm_df['LAT'], errors='coerce').tolist(),
                    'lon': pd.to_numeric(storm_df['LON'], errors='coerce').tolist(),
                    'vmax': pd.to_numeric(storm_df.get('USA_WIND', storm_df.get('WMO_WIND', pd.Series())), errors='coerce').tolist(),
                    'mslp': pd.to_numeric(storm_df.get('USA_PRES', storm_df.get('WMO_PRES', pd.Series())), errors='coerce').tolist(),
                }
            }
        return storms
    except: return {}

def load_active_storms_from_nhc_rss():
    """Load active storms from NHC RSS feeds."""
    for url in ["https://www.nhc.noaa.gov/CurrentStorms.json"]:
        try:
            response = requests.get(url, timeout=15)
            response.raise_for_status()
            data = response.json()
            if 'activeStorms' in data:
                storms = {}
                for storm in data['activeStorms']:
                    storm_id = storm.get('id', storm.get('atcfid', ''))
                    storms[storm_id] = {
                        'name': storm.get('name', 'UNNAMED'), 'storm_id': storm_id,
                        'basin_id': 'north_atlantic' if storm_id.startswith('AL') else 'east_pacific',
                        'vmax': storm.get('intensity', 0), 'is_active': True, 'source': 'nhc_rss',
                        'lat': storm.get('lat'), 'lon': storm.get('lon'),
                    }
                return storms
        except: continue
    return {}

def load_active_storms_from_jtwc_rss():
    """
    Load active storms from JTWC RSS feed.
    JTWC covers: West Pacific, North Indian, Southern Hemisphere.
    This is the official US Navy source for non-Atlantic tropical cyclones.
    """
    import xml.etree.ElementTree as ET
    import re
    
    JTWC_BASIN_MAP = {
        'WP': 'west_pacific',   # Western Pacific
        'IO': 'north_indian',   # North Indian Ocean
        'SH': 'south_indian',   # Southern Hemisphere (Indian side)
        'SP': 'south_pacific',  # South Pacific
        'EP': 'east_pacific',   # Eastern Pacific (JTWC also tracks)
        'CP': 'east_pacific',   # Central Pacific
    }
    
    storms = {}
    try:
        response = requests.get('https://www.metoc.navy.mil/jtwc/rss/jtwc.rss', timeout=20)
        response.raise_for_status()
        
        root = ET.fromstring(response.content)
        
        for item in root.findall('.//item'):
            try:
                title = item.find('title').text or ''
                description = item.find('description').text or ''
                
                # Skip non-storm items
                if 'Tropical Cyclone' not in description and 'Warning' not in description:
                    continue
                
                # Extract storm info from description HTML
                # Example: "Tropical Cyclone 19S (Fytia) Warning #03"
                storm_match = re.search(r'Tropical Cyclone\s+(\d+)([A-Z])\s+\(([^)]+)\)', description, re.IGNORECASE)
                if not storm_match:
                    # Try alternate format
                    storm_match = re.search(r'(\d+)([A-Z])\s+\(([^)]+)\)\s+Warning', description, re.IGNORECASE)
                
                if storm_match:
                    storm_num = storm_match.group(1)
                    basin_code = storm_match.group(2).upper()
                    storm_name = storm_match.group(3).strip().title()
                    
                    # Build storm ID (e.g., "SH192026" for storm 19 in SH basin, year 2026)
                    # JTWC uses format like "sh1926" in URLs
                    storm_id_short = f"{basin_code.lower()}{storm_num}{str(current_year)[2:]}"
                    storm_id = f"{basin_code}{storm_num}{current_year}"
                    
                    # Extract TCW file URL for track data
                    tcw_match = re.search(r"href='([^']*\.tcw)'", description)
                    tcw_url = tcw_match.group(1) if tcw_match else None
                    
                    # Map basin code to our basin IDs
                    basin_id = JTWC_BASIN_MAP.get(basin_code, 'unknown')
                    if basin_code == 'S':  # Southern hemisphere generic
                        basin_id = 'south_indian'
                    
                    storms[storm_id] = {
                        'name': storm_name,
                        'storm_id': storm_id,
                        'storm_id_short': storm_id_short,
                        'basin_id': basin_id,
                        'basin_code': basin_code,
                        'tcw_url': tcw_url,
                        'is_active': True,
                        'source': 'jtwc_rss',
                        'year': current_year,
                    }
            except Exception as e:
                continue
                
        return storms
        
    except Exception as e:
        print(f"   ⚠️ JTWC RSS fetch error: {e}")
        return {}

def fetch_jtwc_track_data(tcw_url=None, storm_id_short=None):
    """
    Fetch track data from JTWC TCW (JMV 3.0 Data) file.
    The TCW format contains structured track data with:
    - Current position and intensity
    - Forecast positions at T+12, T+24, T+36, T+48, T+72, T+96, T+120 hours
    
    Format example:
    T000 155S 0430E 080 R064 020 NE QD...  (Current: 15.5°S, 43.0°E, 80kt)
    T012 161S 0445E 090 ...                 (Forecast +12h)
    """
    import re
    
    # Build URL if not provided
    if not tcw_url and storm_id_short:
        tcw_url = f"https://www.metoc.navy.mil/jtwc/products/{storm_id_short}.tcw"
    
    if not tcw_url:
        return None
    
    try:
        response = requests.get(tcw_url, timeout=15)
        if response.status_code != 200:
            return None
        
        content = response.text
        lines = content.split('\n')
        
        times, lats, lons, vmaxs, mslps = [], [], [], [], []
        base_time = None
        
        # Extract base timestamp from header line
        # Format: "2026013012 19S FYTIA ..."
        for line in lines:
            time_match = re.match(r'^(\d{10})\s+', line)
            if time_match:
                try:
                    base_time = pd.to_datetime(time_match.group(1), format='%Y%m%d%H')
                    break
                except:
                    pass
        
        if not base_time:
            base_time = pd.Timestamp.utcnow()
        
        # Parse track points (T000, T012, T024, etc.)
        for line in lines:
            # Match lines like: T000 155S 0430E 080 or T012 161S 0445E 090
            track_match = re.match(r'T(\d{3})\s+(\d+)([NS])\s+(\d+)([EW])\s+(\d+)', line)
            if track_match:
                try:
                    forecast_hour = int(track_match.group(1))
                    lat_val = int(track_match.group(2)) / 10.0
                    lat_hem = track_match.group(3)
                    lon_val = int(track_match.group(4)) / 10.0
                    lon_hem = track_match.group(5)
                    vmax_val = int(track_match.group(6))
                    
                    # Apply hemisphere
                    if lat_hem == 'S':
                        lat_val = -lat_val
                    if lon_hem == 'W':
                        lon_val = -lon_val
                    
                    # Calculate timestamp
                    point_time = base_time + timedelta(hours=forecast_hour)
                    
                    times.append(str(point_time))
                    lats.append(lat_val)
                    lons.append(lon_val)
                    vmaxs.append(float(vmax_val))
                    mslps.append(np.nan)  # MSLP not in TCW track lines
                    
                except (ValueError, IndexError):
                    continue
        
        if times:
            return {'time': times, 'lat': lats, 'lon': lons, 'vmax': vmaxs, 'mslp': mslps}
            
    except Exception as e:
        pass
    
    return None

# =============================================================================
# FETCH HISTORICAL TRACK FROM IBTRACS ACTIVE FILE
# =============================================================================
def fetch_ibtracs_historical_track(storm_name, basin_id=None):
    """
    Fetch historical track data from IBTrACS ACTIVE file for a specific storm.
    
    IBTrACS ACTIVE file contains currently active storms with full track history.
    URL: https://www.ncei.noaa.gov/data/international-best-track-archive-for-climate-stewardship-ibtracs/v04r01/access/csv/ibtracs.ACTIVE.list.v04r01.csv
    
    Args:
        storm_name: Name of the storm (e.g., 'FYTIA', 'Fytia')
        basin_id: Basin ID like 'south_indian', 'west_pacific', etc. (optional, helps filter)
    
    Returns:
        dict with track data or None if not found
    """
    try:
        # Fetch IBTrACS ACTIVE file
        url = f"{IBTRACS_V04R01_BASE}ibtracs.ACTIVE.list.v04r01.csv"
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        
        df = pd.read_csv(io.StringIO(response.text), low_memory=False, skiprows=[1])
        if df.empty:
            return None
        
        # Normalize storm name for matching
        storm_name_upper = storm_name.upper().strip()
        
        # Find matching storm by name
        matches = df[df['NAME'].str.upper().str.strip() == storm_name_upper]
        
        # If basin_id provided, filter by basin
        if basin_id and not matches.empty:
            basin_code = IBTRACS_BASIN_CODES.get(basin_id, '')
            if basin_code:
                basin_matches = matches[matches['BASIN'] == basin_code]
                if not basin_matches.empty:
                    matches = basin_matches
        
        if matches.empty:
            # Try partial match
            matches = df[df['NAME'].str.upper().str.contains(storm_name_upper, na=False)]
            if basin_id and not matches.empty:
                basin_code = IBTRACS_BASIN_CODES.get(basin_id, '')
                if basin_code:
                    basin_matches = matches[matches['BASIN'] == basin_code]
                    if not basin_matches.empty:
                        matches = basin_matches
        
        if matches.empty:
            return None
        
        # Get the storm ID (use first match if multiple)
        storm_sid = matches['SID'].iloc[0]
        storm_df = df[df['SID'] == storm_sid].sort_values('ISO_TIME')
        
        # Extract track data
        times = storm_df['ISO_TIME'].tolist()
        lats = pd.to_numeric(storm_df['LAT'], errors='coerce').tolist()
        lons = pd.to_numeric(storm_df['LON'], errors='coerce').tolist()
        
        # Try USA_WIND first, then WMO_WIND
        vmaxs = pd.to_numeric(storm_df.get('USA_WIND', pd.Series()), errors='coerce').tolist()
        if all(pd.isna(v) for v in vmaxs):
            vmaxs = pd.to_numeric(storm_df.get('WMO_WIND', pd.Series()), errors='coerce').tolist()
        
        # Try USA_PRES first, then WMO_PRES
        mslps = pd.to_numeric(storm_df.get('USA_PRES', pd.Series()), errors='coerce').tolist()
        if all(pd.isna(v) for v in mslps):
            mslps = pd.to_numeric(storm_df.get('WMO_PRES', pd.Series()), errors='coerce').tolist()
        
        # Clean up NaN values
        vmaxs = [v if not pd.isna(v) else None for v in vmaxs]
        mslps = [v if not pd.isna(v) else None for v in mslps]
        lats = [v if not pd.isna(v) else None for v in lats]
        lons = [v if not pd.isna(v) else None for v in lons]
        
        if times and len(times) > 0:
            return {
                'time': times,
                'lat': lats,
                'lon': lons,
                'vmax': vmaxs,
                'mslp': mslps,
                'source': 'ibtracs_active',
                'ibtracs_sid': storm_sid,
            }
        
        return None
        
    except Exception as e:
        print(f"   ⚠️ IBTrACS fetch error: {e}")
        return None

# =============================================================================
# HELPER FUNCTIONS
# =============================================================================
def get_category(vmax):
    if vmax >= 137: return "Cat5"
    elif vmax >= 113: return "Cat4"
    elif vmax >= 96: return "Cat3"
    elif vmax >= 83: return "Cat2"
    elif vmax >= 64: return "Cat1"
    elif vmax >= 34: return "TS"
    else: return "TD"

def is_storm_completed(storm_info):
    """Check if a storm has completed/dissipated."""
    from datetime import datetime
    
    track = storm_info.get('track', {})
    if track and track.get('time'):
        try:
            last_time = track['time'][-1]
            if isinstance(last_time, str):
                last_time = pd.to_datetime(last_time)
            
            hours_since_last = (datetime.utcnow() - last_time.to_pydatetime().replace(tzinfo=None)).total_seconds() / 3600
            if hours_since_last > 48:
                return True
            
            vmaxs = track.get('vmax', [])
            if vmaxs and len(vmaxs) > 0:
                last_vmax = vmaxs[-1]
                if last_vmax is not None and last_vmax < 25 and hours_since_last > 12:
                    return True
        except:
            pass
    
    if 'storm' in storm_info:
        s = storm_info['storm']
        if hasattr(s, 'time') and len(s.time) > 0:
            try:
                last_time = s.time[-1]
                if isinstance(last_time, str):
                    last_time = pd.to_datetime(last_time)
                hours_since_last = (datetime.utcnow() - last_time.to_pydatetime().replace(tzinfo=None)).total_seconds() / 3600
                if hours_since_last > 48:
                    return True
                if hasattr(s, 'vmax') and len(s.vmax) > 0:
                    last_vmax = s.vmax[-1]
                    if not np.isnan(last_vmax) and last_vmax < 25 and hours_since_last > 12:
                        return True
            except:
                pass
    return False

def check_landfall(storm_data):
    if isinstance(storm_data, dict):
        record_types = storm_data.get('type', storm_data.get('record', []))
        if record_types:
            for rt in record_types:
                if rt and ('L' in str(rt).upper() or 'LANDFALL' in str(rt).upper()):
                    return True
    return False

class SimpleStorm:
    """Simple storm object for track data from cache/IBTrACS."""
    def __init__(self, data, track_data):
        self.name = data.get('name', 'Unknown')
        self.id = data.get('storm_id', 'Unknown')
        self.year = data.get('year', current_year)
        self.basin = data.get('basin_id', 'unknown')
        self.time = []
        if track_data.get('time'):
            for t in track_data['time']:
                try:
                    self.time.append(pd.to_datetime(t) if isinstance(t, str) else t)
                except: pass
        self.lat = [x for x in track_data.get('lat', []) if x is not None]
        self.lon = [x for x in track_data.get('lon', []) if x is not None]
        self.vmax = [x for x in track_data.get('vmax', []) if x is not None]
        self.mslp = [x for x in track_data.get('mslp', []) if x is not None]

# =============================================================================
# AURORA TRACKER OPTIMAL INITIALIZATION FINDER
# =============================================================================
def find_optimal_aurora_init(storm, is_active_storm=False):
    """
    Find the optimal initialization point for Aurora's Tracker based on tropycal data.
    
    IMPORTANT: ECMWF HRES data is only available at synoptic hours (00, 06, 12, 18 UTC).
    
    For ACTIVE storms: Use the most recent past synoptic hour (today's data) with
    the storm's current position (T+0 from track data).
    
    For HISTORICAL storms: Select optimal track point at a synoptic hour.
    """
    from datetime import timedelta
    
    MIN_WIND_THRESHOLD = 50
    IDEAL_WIND_THRESHOLD = 64
    MAX_WIND_FOR_INIT = 140
    DEFAULT_FORECAST_HOURS = 120
    VALID_ECMWF_HOURS = {0, 6, 12, 18}
    
    result = {
        'init_time': None, 'init_lat': None, 'init_lon': None, 'end_time': None,
        'init_vmax': None, 'init_mslp': None, 'score': 0, 'reason': '',
        'skipped_points': 0, 'warnings': []
    }
    
    times = storm.time if hasattr(storm, 'time') else []
    lats = storm.lat if hasattr(storm, 'lat') else []
    lons = storm.lon if hasattr(storm, 'lon') else []
    vmaxs = storm.vmax if hasattr(storm, 'vmax') else []
    mslps = storm.mslp if hasattr(storm, 'mslp') else []
    
    n_points = len(times)
    if n_points == 0:
        result['reason'] = "No track data available"
        result['warnings'].append("Storm has no track points - cannot initialize")
        return result
    
    # ==========================================================================
    # ACTIVE STORM: Use TODAY's most recent synoptic time with current position
    # ==========================================================================
    if is_active_storm:
        now_utc = pd.Timestamp.utcnow()
        current_hour = now_utc.hour
        
        # Find the most recent past synoptic hour (0, 6, 12, 18)
        synoptic_hours = [0, 6, 12, 18]
        past_synoptic = [h for h in synoptic_hours if h <= current_hour]
        if past_synoptic:
            best_hour = max(past_synoptic)
            init_time = now_utc.replace(hour=best_hour, minute=0, second=0, microsecond=0)
        else:
            # Current hour is before 6 UTC, use previous day's 18Z
            best_hour = 18
            init_time = (now_utc - timedelta(days=1)).replace(hour=best_hour, minute=0, second=0, microsecond=0)
        
        # Use the FIRST track point (T+0 = current position) for location/intensity
        init_lat = lats[0] if len(lats) > 0 else None
        init_lon = lons[0] if len(lons) > 0 else None
        init_vmax = vmaxs[0] if len(vmaxs) > 0 else None
        init_mslp = mslps[0] if len(mslps) > 0 else None
        
        result['init_time'] = init_time
        result['init_lat'] = init_lat
        result['init_lon'] = init_lon
        result['init_vmax'] = init_vmax
        result['init_mslp'] = init_mslp
        result['score'] = 80
        result['reason'] = f"Active storm | Current position | {best_hour:02d}Z (today)"
        result['end_time'] = init_time + timedelta(hours=DEFAULT_FORECAST_HOURS)
        
        return result
    
    scored_points = []
    skipped_non_synoptic = 0
    
    for i in range(n_points):
        score = 0
        reasons = []
        
        vmax = vmaxs[i] if i < len(vmaxs) else np.nan
        mslp = mslps[i] if i < len(mslps) else np.nan
        
        if np.isnan(vmax):
            continue
        
        track_time = times[i]
        if hasattr(track_time, 'hour'):
            track_hour = track_time.hour
        else:
            try:
                track_hour = pd.to_datetime(track_time).hour
            except:
                track_hour = None
        
        if track_hour is not None and track_hour not in VALID_ECMWF_HOURS:
            skipped_non_synoptic += 1
            continue
        
        if vmax >= IDEAL_WIND_THRESHOLD:
            score += 50
            reasons.append(f"Hurricane strength ({vmax} kt)")
        elif vmax >= MIN_WIND_THRESHOLD:
            score += 30
            reasons.append(f"Strong TS ({vmax} kt)")
        elif vmax >= 34:
            score += 10
            reasons.append(f"Tropical storm ({vmax} kt)")
        else:
            score -= 20
            reasons.append(f"Weak system ({vmax} kt)")
        
        if vmax > MAX_WIND_FOR_INIT:
            score -= 10
            reasons.append("Past peak intensity")
        
        if not np.isnan(mslp) and mslp < 1010:
            score += 20
            if mslp < 980:
                score += 10
            reasons.append(f"Good pressure ({mslp} mb)")
        elif np.isnan(mslp):
            score -= 5
            reasons.append("No pressure data")
        
        if i == 0:
            score -= 15
            reasons.append("Genesis point")
        elif i == 1:
            score -= 5
            reasons.append("Near genesis")
        
        remaining_points = n_points - i - 1
        if is_active_storm:
            if i == n_points - 1:
                score += 20
                reasons.append("Latest observation (real-time)")
            elif i >= n_points - 3:
                score += 10
                reasons.append("Recent observation")
            hours_old = (n_points - 1 - i) * 6
            if hours_old > 48:
                score -= 5
                reasons.append(f"Observation {hours_old}h old")
        else:
            # Historical storms: adjust scoring based on track length
            is_short_track = n_points <= 6  # 36h or less of track data
            if is_short_track:
                # Short-track historical storms: prefer the first point to
                # maximize forecast coverage through the track. Don't penalize
                # for limited forecast window since the track is inherently short.
                if i == 0:
                    score += 10
                    reasons.append("First point (short track - max coverage)")
                elif i == 1:
                    score += 5
                    reasons.append("Early point (short track)")
                # No penalty for remaining_points < 3 on short tracks
            else:
                # Normal historical storm scoring
                if remaining_points >= 10:
                    score += 15
                    reasons.append(f"{remaining_points * 6}h+ forecast window")
                elif remaining_points >= 5:
                    score += 10
                elif remaining_points < 3:
                    score -= 10
                    reasons.append("Limited forecast window")
        
        scored_points.append({
            'index': i, 'time': times[i], 'lat': lats[i], 'lon': lons[i],
            'vmax': vmax, 'mslp': mslp, 'score': score, 'reasons': reasons
        })
    
    if not scored_points:
        result['warnings'].append(f"No track points at valid ECMWF hours. Skipped {skipped_non_synoptic} non-synoptic points.")
        
        fallback_idx = 0
        for i in range(n_points):
            if i < len(vmaxs) and not np.isnan(vmaxs[i]):
                fallback_idx = i
                break
        
        fallback_time = times[fallback_idx]
        if hasattr(fallback_time, 'hour'):
            current_hour = fallback_time.hour
            synoptic_hours = [0, 6, 12, 18, 24]
            nearest_hour = min(synoptic_hours, key=lambda h: abs(h - current_hour) if h != 24 else abs(24 - current_hour))
            if nearest_hour == 24:
                nearest_hour = 0
                fallback_time = fallback_time.replace(hour=0, minute=0, second=0, microsecond=0) + timedelta(days=1)
            else:
                fallback_time = fallback_time.replace(hour=nearest_hour, minute=0, second=0, microsecond=0)
            result['warnings'].append(f"Snapped init time from {current_hour:02d}Z to {nearest_hour:02d}Z")
        
        result['init_time'] = fallback_time
        result['init_lat'] = lats[fallback_idx]
        result['init_lon'] = lons[fallback_idx]
        result['init_vmax'] = vmaxs[fallback_idx] if len(vmaxs) > fallback_idx else np.nan
        result['init_mslp'] = mslps[fallback_idx] if len(mslps) > fallback_idx else np.nan
        result['reason'] = "Fallback to nearest synoptic hour (no exact matches)"
    else:
        scored_points.sort(key=lambda x: (-x['score'], x['index']))
        best = scored_points[0]
        
        result['skipped_points'] = best['index']
        result['init_time'] = best['time']
        result['init_lat'] = best['lat']
        result['init_lon'] = best['lon']
        result['init_vmax'] = best['vmax']
        result['init_mslp'] = best['mslp']
        result['score'] = min(100, max(0, best['score']))
        result['reason'] = " | ".join(best['reasons'][:3])
        
        init_hour = best['time'].hour if hasattr(best['time'], 'hour') else None
        if init_hour is not None:
            result['reason'] += f" | {init_hour:02d}Z (synoptic)"
        
        if best['vmax'] < MIN_WIND_THRESHOLD:
            result['warnings'].append(f"Storm never reached {MIN_WIND_THRESHOLD} kt - tracker may fail.")
        if best['index'] == 0:
            result['warnings'].append("Using genesis point - storm structure may not be well-defined.")
    
    if is_active_storm:
        result['end_time'] = result['init_time'] + timedelta(hours=DEFAULT_FORECAST_HOURS)
    else:
        result['end_time'] = times[-1]
    
    return result

# =============================================================================
# LOAD STORM DATA
# =============================================================================
print("=" * 70)
print("🌀 LOADING STORM DATABASES")
print("=" * 70)
basin_datasets = {}
failed_basins = []

for basin_id, config in BASINS_CONFIG.items():
    try:
        try:
            if config['source'] == 'ibtracs':
                ds = tracks.TrackDataset(basin=basin_id, source=config['source'], include_btk=False)
            else:
                ds = tracks.TrackDataset(basin=basin_id, source=config['source'], include_btk=True)
            basin_datasets[basin_id] = ds
            print(f"   ✓ {config['full_name']} ({config['source']})")
            time.sleep(1)
            continue
        except Exception as e1:
            if config['source'] == 'ibtracs':
                try:
                    ds = tracks.TrackDataset(basin=basin_id, source=config['source'], 
                                           include_btk=False, ibtracs_url=IBTRACS_URL_V04R01)
                    basin_datasets[basin_id] = ds
                    print(f"   ✓ {config['full_name']} (v04r01 fallback)")
                    time.sleep(1)
                    continue
                except: pass
            raise e1
    except Exception as e:
        failed_basins.append(basin_id)
        print(f"   ⚠️ {config['full_name']}: failed")

print(f"\n   Loaded: {len(basin_datasets)}/{len(BASINS_CONFIG)} basins")

# =============================================================================
# LOAD ACTIVE STORMS
# =============================================================================
print("\n🔴 Checking for active storms...")
realtime_storms = {}
realtime_loaded = False

realtime_sources = [
    {'jtwc': True, 'jtwc_source': 'jtwc'},
    {'jtwc': True, 'jtwc_source': 'ucar'},
    {'jtwc': True, 'jtwc_source': 'noaa'},
    {'jtwc': False, 'jtwc_source': None},
]

for source_config in realtime_sources:
    try:
        realtime_obj = realtime.Realtime(jtwc=source_config['jtwc'], 
                                          jtwc_source=source_config['jtwc_source']) if source_config['jtwc'] else realtime.Realtime(jtwc=False)
        
        if realtime_obj and hasattr(realtime_obj, 'list_active_storms'):
            active_ids = realtime_obj.list_active_storms(basin='all')
            if active_ids:
                for storm_id in active_ids:
                    try:
                        storm = realtime_obj.get_storm(storm_id)
                        if storm and hasattr(storm, 'name'):
                            basin = getattr(storm, 'basin', 'unknown')
                            realtime_storms[storm_id] = {
                                'storm': storm, 'name': storm.name, 'basin': basin,
                                'source': 'realtime', 'is_active': True
                            }
                    except: continue
                if realtime_storms:
                    realtime_loaded = True
                    jtwc_label = f" (JTWC/{source_config['jtwc_source']})" if source_config['jtwc'] else ""
                    print(f"   ✓ Tropycal{jtwc_label}: Found {len(realtime_storms)} active storm(s)")
                    break
    except Exception as e:
        continue

# NHC RSS - Atlantic and Eastern Pacific
try:
    nhc_active = load_active_storms_from_nhc_rss()
    if nhc_active:
        new_storms = {k: v for k, v in nhc_active.items() if k not in realtime_storms}
        if new_storms:
            realtime_storms.update(new_storms)
            realtime_loaded = True
            print(f"   📡 NHC RSS: Found {len(new_storms)} additional storm(s)")
except Exception as e:
    print(f"   ⚠️ NHC RSS check failed: {e}")

# JTWC RSS - Primary source for non-Atlantic storms (covers WestPac, Indian, Southern Hemisphere)
try:
    jtwc_active = load_active_storms_from_jtwc_rss()
    if jtwc_active:
        new_storms = {k: v for k, v in jtwc_active.items() if k not in realtime_storms}
        if new_storms:
            realtime_storms.update(new_storms)
            realtime_loaded = True
            print(f"   📡 JTWC RSS: Found {len(new_storms)} storm(s) with track data")
except Exception as e:
    print(f"   ⚠️ JTWC RSS check failed: {e}")

# IBTrACS ACTIVE - Authoritative source with vmax and full track data (prefetch)
# This enriches the JTWC/NHC/tropycal storms with accurate intensity and track data
try:
    ibtracs_active = load_active_storms_from_ibtracs()
    if ibtracs_active:
        # Match IBTrACS storms with realtime_storms by name and enrich them
        enriched_count = 0
        for ibt_sid, ibt_data in ibtracs_active.items():
            ibt_name = ibt_data.get('name', '').upper()
            
            # Find matching storm in realtime_storms by name
            for rt_id, rt_data in list(realtime_storms.items()):
                rt_name = rt_data.get('name', '').upper()
                if ibt_name and rt_name and ibt_name == rt_name:
                    # Enrich with IBTrACS data (vmax, track, sid)
                    realtime_storms[rt_id]['vmax'] = ibt_data.get('vmax', rt_data.get('vmax', 0))
                    realtime_storms[rt_id]['track'] = ibt_data.get('track')  # Prefetched track!
                    realtime_storms[rt_id]['ibtracs_sid'] = ibt_sid
                    realtime_storms[rt_id]['source'] = 'ibtracs_active'
                    enriched_count += 1
                    break
            else:
                # New storm not in realtime_storms - add it
                if ibt_name not in [rs.get('name', '').upper() for rs in realtime_storms.values()]:
                    realtime_storms[ibt_sid] = ibt_data
                    enriched_count += 1
        
        if enriched_count > 0:
            print(f"   📊 IBTrACS ACTIVE: Enriched/added {enriched_count} storm(s) with intensity & track data")
except Exception as e:
    print(f"   ⚠️ IBTrACS ACTIVE check failed: {e}")

if realtime_storms:
    print(f"   ✓ Found {len(realtime_storms)} active storm(s)")
else:
    print("   No active storms currently")

# ============================================================================
# CROSS-REFERENCE: Reclassify "active" storms that exist in tropycal as historical
# ============================================================================
# If tropycal already has this storm in its historical dataset, it means the storm
# is completed. IBTrACS active feed is sometimes late to remove finished storms,
# so we trust tropycal as the authoritative source for completion status.
reclassified_storms = []
for rt_id, rt_info in list(realtime_storms.items()):
    rt_name = rt_info.get('name', '').upper()
    if not rt_name:
        continue
    # Check if this storm exists in any tropycal basin dataset for the current year
    for basin_id, ds in basin_datasets.items():
        try:
            season = ds.get_season(current_year)
            if hasattr(season, 'dict') and season.dict:
                for sid, sdata in season.dict.items():
                    hist_name = (sdata.get('name', '') if isinstance(sdata, dict)
                                 else getattr(sdata, 'name', '')).upper()
                    if hist_name and hist_name == rt_name:
                        reclassified_storms.append(rt_name)
                        del realtime_storms[rt_id]
                        raise StopIteration  # Break out of both inner loops
        except StopIteration:
            break
        except Exception:
            continue

if reclassified_storms:
    print(f"   📜 Reclassified {len(reclassified_storms)} storm(s) as historical "
          f"(found in tropycal): {', '.join(reclassified_storms)}")

# ============================================================================
# BUILD STORM OPTIONS
# ============================================================================
print(f"\n📊 Loading historical storms for {years_to_include}...")

storm_options = []
storm_data_cache = {}
loaded_from_cache, loaded_from_server, loaded_from_ibtracs_direct = {}, {}, {}

# Add active storms first (filter out completed storms, but trust IBTrACS ACTIVE)
completed_storms = []
for storm_id, storm_info in realtime_storms.items():
    # Storms from IBTrACS ACTIVE file are definitionally active - don't filter them
    source = storm_info.get('source', '')
    if source != 'ibtracs_active' and is_storm_completed(storm_info):
        completed_storms.append(storm_info.get('name', storm_id))
        continue
    
    if 'storm' in storm_info:
        s = storm_info['storm']
        name, vmax = storm_info['name'], s.vmax[-1] if hasattr(s, 'vmax') and len(s.vmax) > 0 else 0
        basin = storm_info.get('basin', 'unknown')
    else:
        name, vmax, basin = storm_info.get('name', storm_id), storm_info.get('vmax', 0), storm_info.get('basin_id', 'unknown')
    
    basin_short = BASINS_CONFIG.get(basin, {}).get('display', basin[:4].upper() if basin else 'UNK')
    option_text = f"🔴 {name} ({current_year}) - {basin_short} - ACTIVE - {vmax} kt"
    option_value = f"realtime_{storm_id}"
    storm_options.append((option_text, option_value))
    storm_data_cache[option_value] = {
        'storm': storm_info.get('storm'), 'storm_id': storm_id, 'name': name,
        'year': current_year, 'source': storm_info.get('source', 'realtime'),
        'basin_id': basin, 'is_active': True, 'track': storm_info.get('track'),
        'tcw_url': storm_info.get('tcw_url'), 'storm_id_short': storm_info.get('storm_id_short'),
        'ibtracs_sid': storm_info.get('ibtracs_sid')
    }

if completed_storms:
    print(f"   ℹ️ Filtered out {len(completed_storms)} completed storm(s): {', '.join(completed_storms)}")

# Load historical storms
for year in years_to_include:
    year_total = 0
    for basin_id, ds in basin_datasets.items():
        basin_short = BASINS_CONFIG[basin_id]['display']
        storms_added = 0
        
        cached_storms = load_cached_storm_data(basin_id, year)
        if cached_storms:
            for storm_id, storm_info in cached_storms.items():
                storm_name = storm_info.get('name', storm_id)
                if storm_name in ['UNNAMED', ''] or not storm_name: continue
                
                vmax = storm_info.get('vmax', 0)
                made_landfall = storm_info.get('made_landfall', False)
                marker = "🟢" if year == current_year else "⚪"
                landfall_marker = "🏠" if made_landfall else ""
                
                option_value = f"historical_{basin_id}_{year}_{storm_id}"
                if option_value not in storm_data_cache:
                    storm_options.append((f"{marker}{landfall_marker} {storm_name} ({year}) - {basin_short} - {get_category(vmax)} - {vmax} kt", option_value))
                    storm_data_cache[option_value] = {
                        'name': storm_name, 'storm_id': storm_id, 'year': year, 'basin_id': basin_id,
                        'source': 'historical', 'is_active': False, 'made_landfall': made_landfall,
                        'cached': True, 'track': storm_info.get('track')
                    }
                    storms_added += 1
            
            if storms_added > 0:
                loaded_from_cache[f"{basin_short}_{year}"] = storms_added
                year_total += storms_added
        
        try:
            season = ds.get_season(year)
            if hasattr(season, 'dict') and season.dict:
                server_storms_data = {}
                server_added = 0
                
                for storm_id, storm_data in season.dict.items():
                    storm_name = storm_data.get('name', storm_id) if isinstance(storm_data, dict) else getattr(storm_data, 'name', storm_id)
                    if storm_name in ['UNNAMED', ''] or not storm_name: continue
                    
                    if isinstance(storm_data, dict):
                        vmax_list = storm_data.get('vmax', [])
                        vmax = int(np.nanmax(vmax_list)) if vmax_list and len(vmax_list) > 0 else 0
                    else:
                        vmax = int(np.nanmax(storm_data.vmax)) if hasattr(storm_data, 'vmax') else 0
                    
                    made_landfall = check_landfall(storm_data)
                    marker = "🟢" if year == current_year else "⚪"
                    landfall_marker = "🏠" if made_landfall else ""
                    option_value = f"historical_{basin_id}_{year}_{storm_id}"
                    
                    if isinstance(storm_data, dict):
                        track_data = {
                            'time': [str(t) for t in storm_data.get('time', [])],
                            'lat': [float(x) if not np.isnan(x) else None for x in storm_data.get('lat', [])],
                            'lon': [float(x) if not np.isnan(x) else None for x in storm_data.get('lon', [])],
                            'vmax': [float(x) if not np.isnan(x) else None for x in storm_data.get('vmax', [])],
                            'mslp': [float(x) if not np.isnan(x) else None for x in storm_data.get('mslp', [])],
                        }
                    else:
                        track_data = {
                            'time': [str(t) for t in (storm_data.time if hasattr(storm_data, 'time') else [])],
                            'lat': [float(x) if not np.isnan(x) else None for x in (storm_data.lat if hasattr(storm_data, 'lat') else [])],
                            'lon': [float(x) if not np.isnan(x) else None for x in (storm_data.lon if hasattr(storm_data, 'lon') else [])],
                            'vmax': [float(x) if not np.isnan(x) else None for x in (storm_data.vmax if hasattr(storm_data, 'vmax') else [])],
                            'mslp': [float(x) if not np.isnan(x) else None for x in (storm_data.mslp if hasattr(storm_data, 'mslp') else [])],
                        }
                    
                    server_storms_data[storm_id] = {
                        'name': storm_name, 'storm_id': storm_id, 'year': year, 'basin_id': basin_id,
                        'vmax': vmax, 'made_landfall': made_landfall, 'track': track_data
                    }
                    
                    if option_value not in storm_data_cache:
                        storm_options.append((f"{marker}{landfall_marker} {storm_name} ({year}) - {basin_short} - {get_category(vmax)} - {vmax} kt", option_value))
                        storm_data_cache[option_value] = {
                            'name': storm_name, 'storm_id': storm_id, 'year': year, 'basin_id': basin_id,
                            'source': 'historical', 'is_active': False, 'made_landfall': made_landfall, 'track': track_data
                        }
                        server_added += 1
                
                if server_storms_data:
                    save_storm_data_to_cache(basin_id, year, server_storms_data)
                    if server_added > 0:
                        loaded_from_server[f"{basin_short}_{year}"] = server_added
                        year_total += server_added
        except: pass
    
    if year_total == 0 and year >= 2024:
        for basin_id in basin_datasets.keys():
            direct_storms = load_ibtracs_direct(basin_id, years_filter=[year])
            for storm_id, storm_info in direct_storms.items():
                storm_name = storm_info.get('name', storm_id)
                if storm_name in ['UNNAMED', ''] or not storm_name: continue
                
                vmax = storm_info.get('vmax', 0)
                made_landfall = storm_info.get('made_landfall', False)
                marker = "🟢" if year == current_year else "⚪"
                landfall_marker = "🏠" if made_landfall else ""
                basin_short = BASINS_CONFIG[basin_id]['display']
                option_value = f"historical_{basin_id}_{year}_{storm_id}"
                
                if option_value not in storm_data_cache:
                    storm_options.append((f"{marker}{landfall_marker} {storm_name} ({year}) - {basin_short} - {get_category(vmax)} - {vmax} kt", option_value))
                    storm_data_cache[option_value] = {
                        'name': storm_name, 'storm_id': storm_id, 'year': year, 'basin_id': basin_id,
                        'source': 'ibtracs_direct', 'is_active': False, 'made_landfall': made_landfall, 'track': storm_info.get('track')
                    }
            
            if direct_storms:
                loaded_from_ibtracs_direct[f"{BASINS_CONFIG[basin_id]['display']}_{year}"] = len(direct_storms)
                save_storm_data_to_cache(basin_id, year, direct_storms)

print(f"\n   📦 From cache: {sum(loaded_from_cache.values())} storms")
print(f"   🌐 From server: {sum(loaded_from_server.values())} storms")
print(f"   📡 From IBTrACS direct: {sum(loaded_from_ibtracs_direct.values())} storms")
print(f"   📋 Total: {len(storm_options)} storms available")

# =============================================================================
# SET DEFAULT STORM (HELENE 2024)
# =============================================================================
default_storm_value = None
for i, (opt_text, opt_value) in enumerate(storm_options):
    if 'helene' in opt_text.lower() and '2024' in opt_text and 'north_atlantic' in opt_value:
        storm_options[i] = (opt_text + " ⭐ DEFAULT", opt_value)
        default_storm_value = opt_value
        break

if not default_storm_value:
    for opt_text, opt_value in storm_options:
        if '2024' in opt_text and not opt_text.startswith('🔴'):
            default_storm_value = opt_value
            break

if not default_storm_value and storm_options:
    default_storm_value = storm_options[0][1]

print(f"\n🎯 Hurricane Helene (2024) selected as default.")
print("   Feel free to choose a different storm from the widget below.")

# =============================================================================
# GLOBAL STORM VARIABLES
# =============================================================================
storm_id = None
storm_name = None
storm_year = None
storm_basin = None
selected_storm = None
selected_storm_df = None
storm_init_time = None
storm_end_time = None
storm_init_lat = None
storm_init_lon = None

# =============================================================================
# WIDGET FUNCTIONS
# =============================================================================
def display_storm_details(change):
    """Display detailed information about the selected storm"""
    with details_output:
        clear_output(wait=True)
        selected_value = storm_dropdown.value
        if not selected_value:
            print("Please select a storm")
            return
        
        cache_entry = storm_data_cache.get(selected_value)
        if not cache_entry:
            print(f"Error: Could not find data for {selected_value}")
            return
        
        source = cache_entry.get('source', 'unknown')
        is_realtime = source == 'realtime'
        
        try:
            if is_realtime and 'storm' in cache_entry and cache_entry['storm']:
                storm = cache_entry['storm']
            elif cache_entry.get('track'):
                storm = SimpleStorm(cache_entry, cache_entry.get('track', {}))
            elif source in ['jtwc_rss', 'ibtracs_active', 'ibtracs_direct', 'nhc_rss']:
                print(f"   ⏳ Fetching track data for {cache_entry.get('name', 'storm')}...")
                
                # For active storms, try IBTrACS ACTIVE first (most reliable historical data)
                storm_name_temp = cache_entry.get('name', '')
                basin_id_temp = cache_entry.get('basin_id')
                
                track_found = False
                
                # 1. Try IBTrACS ACTIVE file first (best source for historical observations)
                ibtracs_track = fetch_ibtracs_historical_track(storm_name_temp, basin_id_temp)
                if ibtracs_track and len(ibtracs_track.get('time', [])) > 0:
                    cache_entry['track'] = ibtracs_track
                    cache_entry['track_source'] = 'ibtracs'
                    storm = SimpleStorm(cache_entry, ibtracs_track)
                    print(f"   ✓ Historical track from IBTrACS: {len(ibtracs_track.get('time', []))} observation points")
                    track_found = True
                
                # 2. Fall back to JTWC TCW data (forecasts/current position)
                if not track_found:
                    print(f"   ⏳ IBTrACS not available, trying JTWC TCW...")
                    tcw_url = cache_entry.get('tcw_url')
                    storm_id_short = cache_entry.get('storm_id_short')
                    jtwc_track = fetch_jtwc_track_data(tcw_url=tcw_url, storm_id_short=storm_id_short)
                    if jtwc_track:
                        cache_entry['track'] = jtwc_track
                        cache_entry['track_source'] = 'jtwc'
                        storm = SimpleStorm(cache_entry, jtwc_track)
                        print(f"   ✓ Track from JTWC: {len(jtwc_track.get('time', []))} points (current + forecast)")
                        print(f"   ⚠️ Note: Historical observation data not yet available")
                        track_found = True
                
                # 3. Last resort - no track data
                if not track_found:
                    print(f"   ⚠️ No track data available from IBTrACS or JTWC")
                    print(f"   Try confirming selection anyway - forecast will start from estimated position.")
                    storm = SimpleStorm(cache_entry, {'time': [], 'lat': [], 'lon': [], 'vmax': [], 'mslp': []})
            else:
                basin_id = cache_entry['basin_id']
                ds = basin_datasets.get(basin_id)
                if ds:
                    try:
                        storm = ds.get_storm(cache_entry['storm_id'])
                    except KeyError:
                        print(f"   ⚠️ Storm not in tropycal, trying JTWC...")
                        tcw_url = cache_entry.get('tcw_url')
                        storm_id_short = cache_entry.get('storm_id_short')
                        jtwc_track = fetch_jtwc_track_data(tcw_url=tcw_url, storm_id_short=storm_id_short)
                        if jtwc_track:
                            cache_entry['track'] = jtwc_track
                            print(f"   ✓ Track loaded from JTWC: {len(jtwc_track.get('time', []))} points")
                        storm = SimpleStorm(cache_entry, cache_entry.get('track', {}))
                else:
                    print(f"Error: Basin {basin_id} not loaded")
                    return
            
            print(f"🌀 {storm.name} ({storm.year if hasattr(storm, 'year') else cache_entry.get('year', 'N/A')})")
            print("=" * 50)
            
            basin_display = BASINS_CONFIG.get(cache_entry['basin_id'], {}).get('full_name', cache_entry['basin_id'])
            print(f"\n📍 Basin: {basin_display}")
            print(f"🆔 Storm ID: {storm.id if hasattr(storm, 'id') else cache_entry.get('storm_id', 'N/A')}")
            print(f"📡 Source: {source}")
            
            if hasattr(storm, 'time') and len(storm.time) > 0:
                print(f"\n📊 Track: {storm.time[0]} → {storm.time[-1]} ({len(storm.time)} points)")
            
            if hasattr(storm, 'vmax') and len(storm.vmax) > 0:
                max_wind = np.nanmax(storm.vmax)
                print(f"💨 Peak: {max_wind:.0f} kt ({get_category(max_wind)})")
            
            if cache_entry.get('made_landfall'):
                print(f"🏠 Made landfall")
            
            aurora_init = find_optimal_aurora_init(storm, is_realtime or source == 'jtwc_rss')
            if aurora_init['init_time']:
                print(f"\n🎯 Aurora Init: {aurora_init['init_time']}")
                print(f"   Location: {aurora_init['init_lat']:.1f}°N, {aurora_init['init_lon']:.1f}°E")
                if aurora_init['init_vmax']:
                    print(f"   Intensity: {aurora_init['init_vmax']:.0f} kt | {aurora_init['reason']} | score: {aurora_init['score']}/100")
            print(f"\n✅ Click 'Confirm Selection' to use this storm")
            
        except Exception as e:
            print(f"Error loading storm details: {e}")
            import traceback
            traceback.print_exc()

def confirm_selection(button):
    """Confirm storm selection and set global variables"""
    global storm_id, storm_name, storm_year, storm_basin, selected_storm, selected_storm_df
    global storm_init_time, storm_end_time, storm_init_lat, storm_init_lon
    global is_active_storm
    
    with status_output:
        clear_output(wait=True)
        selected_value = storm_dropdown.value
        if not selected_value:
            print("⚠️ Please select a storm first")
            return
        
        cache_entry = storm_data_cache.get(selected_value)
        if not cache_entry:
            print(f"⚠️ Error: Could not find data for {selected_value}")
            return
        
        try:
            source = cache_entry.get('source', '')
            is_realtime = source == 'realtime'
            is_direct = source in ['ibtracs_direct', 'ibtracs_active']
            
            if is_realtime and 'storm' in cache_entry and cache_entry['storm']:
                storm = cache_entry['storm']
                storm_id = storm.id if hasattr(storm, 'id') else selected_value.replace('realtime_', '')
                storm_name = storm.name if hasattr(storm, 'name') else cache_entry.get('name', 'Unknown')
                storm_year = storm.year if hasattr(storm, 'year') else current_year
                storm_basin = cache_entry['basin_id']
            elif is_direct or cache_entry.get('track') or source in ['jtwc_rss', 'nhc_rss']:
                storm_id = cache_entry['storm_id']
                storm_name = cache_entry.get('name', 'Unknown')
                storm_year = cache_entry.get('year', current_year)
                storm_basin = cache_entry['basin_id']
                if not cache_entry.get('track') and source == 'jtwc_rss':
                    print(f"   ⏳ Fetching track data for {storm_name}...")
                    
                    track_found = False
                    
                    # 1. Try IBTrACS ACTIVE first (best historical data)
                    ibtracs_track = fetch_ibtracs_historical_track(storm_name, storm_basin)
                    if ibtracs_track and len(ibtracs_track.get('time', [])) > 0:
                        cache_entry['track'] = ibtracs_track
                        cache_entry['track_source'] = 'ibtracs'
                        print(f"   ✓ Historical track from IBTrACS: {len(ibtracs_track.get('time', []))} points")
                        track_found = True
                    
                    # 2. Fall back to JTWC TCW
                    if not track_found:
                        print(f"   ⏳ IBTrACS not available, trying JTWC TCW...")
                        tcw_url = cache_entry.get('tcw_url')
                        storm_id_short = cache_entry.get('storm_id_short')
                        jtwc_track = fetch_jtwc_track_data(tcw_url=tcw_url, storm_id_short=storm_id_short)
                        if jtwc_track:
                            cache_entry['track'] = jtwc_track
                            cache_entry['track_source'] = 'jtwc'
                            print(f"   ✓ Track from JTWC: {len(jtwc_track.get('time', []))} points")
                            track_found = True
                    
                    if not track_found:
                        print(f"   ⚠️ No track data available - using current position only")
                storm = SimpleStorm(cache_entry, cache_entry.get('track', {}))
            else:
                basin_id = cache_entry['basin_id']
                ds = basin_datasets.get(basin_id)
                if ds:
                    try:
                        storm = ds.get_storm(cache_entry['storm_id'])
                    except KeyError:
                        print(f"   ⚠️ Storm not in tropycal, trying JTWC...")
                        tcw_url = cache_entry.get('tcw_url')
                        storm_id_short = cache_entry.get('storm_id_short')
                        jtwc_track = fetch_jtwc_track_data(tcw_url=tcw_url, storm_id_short=storm_id_short)
                        if jtwc_track:
                            cache_entry['track'] = jtwc_track
                            print(f"   ✓ Track loaded from JTWC: {len(jtwc_track.get('time', []))} points")
                        storm = SimpleStorm(cache_entry, cache_entry.get('track', {}))
                    storm_id = cache_entry['storm_id']
                    storm_name = storm.name if hasattr(storm, 'name') else cache_entry.get('name', 'Unknown')
                    storm_year = cache_entry.get('year', current_year)
                    storm_basin = basin_id
                else:
                    print(f"⚠️ Error: Basin {basin_id} not loaded")
                    return
            
            selected_storm = storm
            
            if hasattr(storm, 'time') and len(storm.time) > 0:
                n_times = len(storm.time)
                def get_array(arr, length):
                    if arr is None: return [None] * length
                    arr_list = list(arr) if hasattr(arr, '__iter__') else [arr]
                    return (arr_list + [None] * length)[:length]
                
                selected_storm_df = pd.DataFrame({
                    'time': list(storm.time)[:n_times],
                    'lat': get_array(storm.lat if hasattr(storm, 'lat') else None, n_times),
                    'lon': get_array(storm.lon if hasattr(storm, 'lon') else None, n_times),
                    'vmax': get_array(storm.vmax if hasattr(storm, 'vmax') else None, n_times),
                    'mslp': get_array(storm.mslp if hasattr(storm, 'mslp') else None, n_times),
                })
            
            is_active_storm = cache_entry.get('is_active', False) or source in ['realtime', 'ibtracs_active', 'jtwc_rss']
            aurora_init = find_optimal_aurora_init(storm, is_active_storm)
            storm_init_time = aurora_init['init_time']
            storm_end_time = aurora_init['end_time']
            storm_init_lat = aurora_init['init_lat']
            storm_init_lon = aurora_init['init_lon']
            
            print("=" * 50)
            print(f"✅ SELECTED: {storm_name} ({storm_year})")
            if is_active_storm:
                print(f"   🔴 ACTIVE STORM - Intelligent forecast termination enabled")
            print("=" * 50)
            print(f"\n📋 Variables set:")
            print(f"   storm_id = '{storm_id}' | storm_name = '{storm_name}'")
            print(f"   storm_year = {storm_year} | storm_basin = '{storm_basin}'")
            print(f"   storm_init_time = {storm_init_time}")
            print(f"   storm_end_time = {storm_end_time}")
            print(f"\n🚀 Ready to proceed with Aurora inference!")
            
            # Backward compatibility aliases
            global HELENE_STORM_ID, HELENE_YEAR, HELENE_BASIN
            HELENE_STORM_ID = storm_id
            HELENE_YEAR = storm_year
            HELENE_BASIN = storm_basin
            
        except Exception as e:
            print(f"⚠️ Error: {e}")
            import traceback
            traceback.print_exc()


# CREATE WIDGETS
# =============================================================================
storm_dropdown = widgets.Dropdown(
    options=storm_options, value=default_storm_value, description='Storm:',
    style={'description_width': 'initial'}, layout=widgets.Layout(width='650px')
)

confirm_button = widgets.Button(
    description='Confirm Selection', button_style='success', icon='check',
    layout=widgets.Layout(width='200px')
)
details_output = widgets.Output()
status_output = widgets.Output()

legend_html = widgets.HTML(value="""
<div style="margin: 10px 0; padding: 10px; background: #f8f9fa; border-radius: 5px; font-size: 12px;">
    <b>Legend:</b> 🔴 Active | 🟢 Current Year | ⚪ Past Year | 🏠 Landfall | ⭐ DEFAULT<br>
    <b>Categories:</b> TD | TS | Cat1-5<br>
    <b>Basins:</b> ATL | EPAC | WPAC | NIO | SIO | SPAC | SATL
</div>
""")
confirm_button.on_click(confirm_selection)
storm_dropdown.observe(display_storm_details, names='value')

widget_box = widgets.VBox([
    legend_html,
    widgets.HBox([storm_dropdown, confirm_button]),
    details_output,
    status_output
])

print("\n" + "=" * 70)
print("🌀 STORM SELECTOR")
print("=" * 70)
display(widget_box)
display_storm_details(None)


# AUTO-SELECT DEFAULT STORM (HELENE 2024)
# =============================================================================
if default_storm_value:
    cache_entry = storm_data_cache.get(default_storm_value)
    if cache_entry:
        try:
            source = cache_entry.get('source', '')
            is_realtime = source == 'realtime'
            
            if is_realtime and 'storm' in cache_entry and cache_entry['storm']:
                storm = cache_entry['storm']
                storm_id = storm.id if hasattr(storm, 'id') else default_storm_value.replace('realtime_', '')
                storm_name = storm.name if hasattr(storm, 'name') else cache_entry.get('name', 'Unknown')
                storm_year = storm.year if hasattr(storm, 'year') else current_year
                storm_basin = cache_entry['basin_id']
            elif cache_entry.get('track'):
                storm_id = cache_entry['storm_id']
                storm_name = cache_entry.get('name', 'Unknown')
                storm_year = cache_entry['year']
                storm_basin = cache_entry['basin_id']
                storm = SimpleStorm(cache_entry, cache_entry.get('track', {}))
            else:
                basin_id = cache_entry['basin_id']
                ds = basin_datasets.get(basin_id)
                if ds:
                    storm = ds.get_storm(cache_entry['storm_id'])
                    storm_id = cache_entry['storm_id']
                    storm_name = storm.name if hasattr(storm, 'name') else cache_entry.get('name', 'Unknown')
                    storm_year = cache_entry['year']
                    storm_basin = basin_id
            
            selected_storm = storm
            
            if hasattr(storm, 'time') and len(storm.time) > 0:
                n_times = len(storm.time)
                def get_array(arr, length):
                    if arr is None: return [None] * length
                    arr_list = list(arr) if hasattr(arr, '__iter__') else [arr]
                    return (arr_list + [None] * length)[:length]
                
                selected_storm_df = pd.DataFrame({
                    'time': list(storm.time)[:n_times],
                    'lat': get_array(storm.lat if hasattr(storm, 'lat') else None, n_times),
                    'lon': get_array(storm.lon if hasattr(storm, 'lon') else None, n_times),
                    'vmax': get_array(storm.vmax if hasattr(storm, 'vmax') else None, n_times),
                    'mslp': get_array(storm.mslp if hasattr(storm, 'mslp') else None, n_times),
                })
            
            is_active_storm = cache_entry.get('is_active', False) or source in ['realtime', 'ibtracs_active']
            aurora_init = find_optimal_aurora_init(storm, is_active_storm)
            storm_init_time = aurora_init['init_time']
            storm_end_time = aurora_init['end_time']
            storm_init_lat = aurora_init['init_lat']
            storm_init_lon = aurora_init['init_lon']
            
            # Backward compatibility aliases
            HELENE_STORM_ID = storm_id
            HELENE_YEAR = storm_year
            HELENE_BASIN = storm_basin

        except Exception as e:
            print(f"⚠️ Auto-select error: {e}")

## 3. Download Model Input Data (ECMWF HRES) via Planetary Computer Pro

Aurora requires two timesteps of ECMWF High-Resolution (HRES) forecast data, 6 hours apart. This data is accessed through **Microsoft Planetary Computer Pro's** STAC catalog and Azure Blob Storage.

**Data Sources**
| Stream | Availability | Source |
|--------|--------------|--------|
| **OPER** | 00Z, 12Z UTC | STAC API (Planetary Computer) |
| **SCDA** | 06Z, 18Z UTC | Azure Blob Storage |

The notebook automatically selects the correct stream based on your storm's initialization time and downloads both timesteps required for Aurora inference.

### 3.1 Data Availability Check

Verify ECMWF data availability and ensure track data covers the initialization time. For active storms, adjusts T0 if there's a gap between track observations and available ECMWF data.

In [ ]:
# =============================================================================
# SECTION 3.1: ECMWF DATA AVAILABILITY CHECK & DETERMINE DOWNLOAD PLAN
# =============================================================================
# This section:
# 1. Checks ECMWF data availability via Planetary Computer STAC
# 2. Determines the optimal Aurora initialization time (T0 and T-6h)
# 3. Aligns track data with available forecast initialization times
# 4. Creates download plan for Section 3.2 (OPER) and 3.3 (SCDA)
# =============================================================================

from datetime import datetime, timedelta
from pathlib import Path
import pystac_client
import planetary_computer
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def make_tz_naive(ts):
    """Convert timestamp to timezone-naive for comparison."""
    if ts is None:
        return None
    if hasattr(ts, 'tzinfo') and ts.tzinfo is not None:
        return ts.tz_localize(None) if hasattr(ts, 'tz_localize') else ts.replace(tzinfo=None)
    return ts

def find_ecmwf_data_near_time(catalog, target_time, stream):
    """Find ECMWF data at or before the target time for a specific stream.
    
    IMPORTANT: Filters for 0.25° resolution required by aurora-0.25-finetuned model.
    """
    target_naive = make_tz_naive(target_time)
    
    # Search window: 7 days before target
    search_start = target_naive - timedelta(days=7)
    search_end = target_naive + timedelta(hours=6)  # Small buffer for timezone issues
    
    search = catalog.search(
        collections=["ecmwf-forecast"],
        datetime=f"{search_start.isoformat()}/{search_end.isoformat()}",
        query={
            "ecmwf:stream": {"eq": stream},
            "ecmwf:type": {"eq": "fc"},
            "ecmwf:step": {"eq": "0h"},
        },
        sortby=[{"field": "datetime", "direction": "desc"}],
        limit=100,  # Increased to ensure we find 0.25° items
    )
    
    items = list(search.items())
    
    # Filter for 0.25° resolution (required for aurora-0.25-finetuned model)
    # Item IDs contain resolution: e.g., ecmwf-2024-09-28T12-oper-fc-0h-0.25
    items_025 = [item for item in items if '0.25' in item.id]
    if items_025:
        items = items_025
    
    # Find the item closest to (but not after) target_time
    best_item = None
    best_time = None
    
    for item in items:
        item_time = make_tz_naive(pd.to_datetime(item.datetime))
        if item_time <= target_naive:
            if best_time is None or item_time > best_time:
                best_time = item_time
                best_item = item
    
    return best_time, best_item

def find_best_forecast_time_for_observation(catalog, last_obs_time):
    """
    Find the best available ECMWF forecast time closest to the last observation.
    
    For Aurora, we want T0 to be as close as possible to the last real storm observation,
    so that the atmospheric state matches when we have actual storm position data.
    
    This searches for OPER (00Z, 12Z) and SCDA (06Z, 18Z) data at or before last_obs_time.
    """
    last_obs_naive = make_tz_naive(last_obs_time)
    
    # Find best available OPER data (00Z, 12Z)
    oper_time, oper_item = find_ecmwf_data_near_time(catalog, last_obs_time, "oper")
    
    # Find best available SCDA data (06Z, 18Z)
    scda_time, scda_item = find_ecmwf_data_near_time(catalog, last_obs_time, "scda")
    
    results = {
        'oper_time': oper_time,
        'scda_time': scda_time,
        'target_time': last_obs_naive,
    }
    
    # Determine which is closer to (but not after) the last observation
    if oper_time is None and scda_time is None:
        return None, None, results
    elif oper_time is None:
        return scda_time, 'scda', results
    elif scda_time is None:
        return oper_time, 'oper', results
    else:
        # Both available - pick the one closer to last_obs_time
        if scda_time > oper_time:
            return scda_time, 'scda', results
        else:
            return oper_time, 'oper', results

def display_storm_not_ready_warning(storm_name, track_times, forecast_time, data_gap_hours, has_position_at_t0):
    """Display a prominent warning that the storm may not be ready for accurate forecasting."""
    from IPython.display import display, HTML
    
    first_obs = track_times.min()
    last_obs = track_times.max()
    first_obs_str = first_obs.strftime('%Y-%m-%d %H:%M UTC')
    last_obs_str = last_obs.strftime('%Y-%m-%d %H:%M UTC')
    forecast_str = forecast_time.strftime('%Y-%m-%d %H:%M UTC') if hasattr(forecast_time, 'strftime') else str(forecast_time)
    
    # Check if T0 is before track starts
    t0_before_track = forecast_time < first_obs
    if t0_before_track:
        gap_from_first = (first_obs - forecast_time).total_seconds() / 3600
        data_gap_hours = gap_from_first
    
    # Determine severity based on gap
    if data_gap_hours > 24 or t0_before_track:
        border_color = "#d32f2f"
        bg_color = "#ffebee"
        icon = "⚠️"
    elif data_gap_hours > 12:
        border_color = "#f57c00"
        bg_color = "#fff3e0"
        icon = "🕐"
    else:
        border_color = "#fbc02d"
        bg_color = "#fffde7"
        icon = "📡"
    
    position_warning = ""
    if t0_before_track:
        position_warning = f"""
        <div style="margin-top: 8px; padding: 10px; background: rgba(211,47,47,0.1); border-radius: 4px; border: 1px solid #d32f2f;">
            <strong>📍 No storm position data at forecast initialization time (T0)</strong><br>
            <span style="font-size: 12px;">
                T0 ({forecast_str}) is <strong>{data_gap_hours:.1f} hours BEFORE</strong> the first track observation.<br>
                The storm's lat/lon at T0 will be <strong>back-extrapolated</strong> from the earliest available position ({first_obs_str}).<br>
                <em style="color: #666;">All storm sources checked: JTWC, IBTrACS, NHC</em>
            </span>
        </div>
        """
    elif not has_position_at_t0:
        position_warning = f"""
        <div style="margin-top: 8px; padding: 8px; background: rgba(0,0,0,0.05); border-radius: 4px;">
            <strong>📍 No storm position available at forecast time</strong><br>
            Position will be extrapolated from the nearest observation.
        </div>
        """
    
    html = f"""
    <div style="background-color: {bg_color}; border-left: 5px solid {border_color}; padding: 15px 20px; 
                margin: 15px 0; font-size: 14px; color: #333; border-radius: 4px; box-shadow: 0 2px 4px rgba(0,0,0,0.1);">
        <div style="font-size: 16px; font-weight: bold; margin-bottom: 10px;">
            {icon} Storm "{storm_name}" May Not Be Ready for Accurate Forecasting
        </div>
        
        <div style="margin-bottom: 12px;">
            <strong>Storm Track Data Availability:</strong>
            <table style="margin-top: 5px; font-size: 13px; border-collapse: collapse;">
                <tr><td style="padding: 3px 10px 3px 0;">First observation:</td><td><code>{first_obs_str}</code></td></tr>
                <tr><td style="padding: 3px 10px 3px 0;">Last observation:</td><td><code>{last_obs_str}</code></td></tr>
                <tr><td style="padding: 3px 10px 3px 0;">Forecast init time (T0):</td><td><code>{forecast_str}</code></td></tr>
                <tr><td style="padding: 3px 10px 3px 0;"><strong>{'Gap to first obs:' if t0_before_track else 'Data gap:'}</strong></td>
                    <td><strong style="color: {border_color};">{data_gap_hours:.1f} hours</strong></td></tr>
            </table>
        </div>
        
        {position_warning}
        
        <div style="margin-top: 12px; padding: 10px; background: rgba(255,255,255,0.7); border-radius: 4px;">
            <strong>💡 Recommendation:</strong><br>
            <span style="font-size: 13px;">
                For more accurate predictions, consider waiting <strong>6-12 hours</strong> for additional storm track 
                observations to become available.
            </span>
        </div>
    </div>
    """
    display(HTML(html))

# =============================================================================
# STAC CONNECTION AND CONFIGURATION
# =============================================================================
print("="*70)
print("🌐 ECMWF DATA AVAILABILITY CHECK")
print("="*70)

# Connect to Planetary Computer STAC
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)
print("✓ Connected to Planetary Computer STAC")

# Set up download directory
downloads_dir = Path("downloads")
downloads_dir.mkdir(exist_ok=True)

# =============================================================================
# ANALYZE STORM TRACK DATA AVAILABILITY
# =============================================================================
print(f"\n📍 Storm Track Analysis: {storm_name}")
print("="*70)

# Get track times
storm_times = pd.to_datetime(selected_storm_df['time'])
first_observation = make_tz_naive(storm_times.min())

# For real-time storms (e.g., from JTWC), track data may include FORECAST positions
# with timestamps in the future. We need the ACTUAL last observation, not forecasts.
# Filter to only include times that are <= current time
now_utc = pd.Timestamp.utcnow()
now_naive = make_tz_naive(now_utc)

# Get only actual observations (times in the past or present)
actual_observations = storm_times[storm_times.apply(lambda t: make_tz_naive(t) <= now_naive)]

if len(actual_observations) > 0:
    last_observation = make_tz_naive(actual_observations.max())
    num_actual_obs = len(actual_observations)
    num_forecast_points = len(storm_times) - num_actual_obs
else:
    # Fallback: if all times are in the future (shouldn't happen), use the earliest
    last_observation = first_observation
    num_actual_obs = 0
    num_forecast_points = len(storm_times)

num_observations = len(storm_times)

print(f"   Track data points:  {num_observations} total")
if num_forecast_points > 0:
    print(f"      - Observations:  {num_actual_obs} (past/present)")
    print(f"      - Forecasts:     {num_forecast_points} (future - excluded)")
print(f"   First observation:  {first_observation}")
print(f"   Last observation:   {last_observation}")
print(f"   Requested init (T0): {storm_init_time}")

# Calculate time span of track data
track_duration = (last_observation - first_observation).total_seconds() / 3600
print(f"   Track duration:     {track_duration:.1f} hours")

# =============================================================================
# CHECK DATA AVAILABILITY - ONLY FOR ACTIVE STORMS
# =============================================================================
# HISTORICAL STORMS: 
#   - Use the T0 from find_optimal_aurora_init directly (no availability check needed)
#   - ECMWF historical data is always available on Planetary Computer
#   - Forecast from T0 through the end of the track
#
# ACTIVE STORMS:
#   - Search for ECMWF data closest to the last observation  
#   - This determines what data is actually available for download
#   - Forecast forward with smart termination algorithm
# =============================================================================

# Get the T0 from find_optimal_aurora_init (set in cell 8)
time_t0 = make_tz_naive(storm_init_time)

if is_active_storm:
    # ACTIVE STORMS: Need to check what data is actually available
    print(f"\n🔍 ACTIVE STORM: Searching for available ECMWF data near last observation ({last_observation})...")
    
    best_time, best_stream, availability_info = find_best_forecast_time_for_observation(catalog, last_observation)
    
    if best_time is None:
        raise ValueError("Cannot find any ECMWF forecast data near the target time!")
    
    # Display availability info
    print(f"\n   📊 Data Availability:")
    print(f"      Target time:     {last_observation}")
    if availability_info['oper_time']:
        gap_oper = (last_observation - availability_info['oper_time']).total_seconds() / 3600
        print(f"      OPER (00z/12z):  {availability_info['oper_time']} ({gap_oper:.1f}h gap)")
    else:
        print(f"      OPER (00z/12z):  Not available")
        
    if availability_info['scda_time']:
        gap_scda = (last_observation - availability_info['scda_time']).total_seconds() / 3600
        print(f"      SCDA (06z/18z):  {availability_info['scda_time']} ({gap_scda:.1f}h gap)")
    else:
        print(f"      SCDA (06z/18z):  Not available")
    
    gap_best = abs((last_observation - best_time).total_seconds() / 3600)
    print(f"\n   ✓ Best match: {best_time} ({best_stream.upper()}) - {gap_best:.1f}h from last observation")
    
    # Update T0 to the best available ECMWF time
    storm_init_time = best_time
    time_t0 = best_time
    print(f"\n   📍 Using T0 = {best_time} (closest available ECMWF data)")
    
    recent_time = best_time
    recent_stream = best_stream
    
else:
    # HISTORICAL STORMS: No availability check needed - data always exists
    print(f"\n📜 HISTORICAL STORM: Using optimal init time from find_optimal_aurora_init")
    print(f"   T0 = {time_t0} (ECMWF historical data is always available)")
    
    # Determine stream based on hour
    init_hour = time_t0.hour
    if init_hour in [0, 12]:
        recent_stream = "oper"
    else:
        recent_stream = "scda"
    recent_time = time_t0

# =============================================================================
# CALCULATE DATA GAP AND CHECK POSITION AVAILABILITY (ACTIVE STORMS ONLY)
# =============================================================================

# Check if we have position data at T0
has_position_at_t0 = (first_observation <= time_t0 <= last_observation)

# Check how far T0 is from the closest track observation
time_diffs = abs((storm_times - time_t0).dt.total_seconds() / 3600)
closest_time_diff = time_diffs.min()
has_close_position = closest_time_diff <= 6

if is_active_storm:
    # For active storms, calculate gap between last observation and T0
    data_gap = time_t0 - last_observation
    data_gap_hours = data_gap.total_seconds() / 3600
    
    print(f"\n📊 Data Alignment Analysis:")
    print(f"   Last storm observation: {last_observation}")
    print(f"   Forecast init time (T0): {time_t0}")
    
    if data_gap_hours > 0:
        print(f"   ⚠️  Data gap: {data_gap_hours:.1f} hours (track data is BEHIND forecast time)")
    elif data_gap_hours < 0:
        print(f"   ✓  Data overlap: {abs(data_gap_hours):.1f} hours (track data extends past forecast time)")
    else:
        print(f"   ✓  Perfect alignment: Track data matches forecast time")
    
    # Calculate gap from first observation if T0 is before track starts
    if time_t0 < first_observation:
        gap_from_first = (first_observation - time_t0).total_seconds() / 3600
        print(f"   ⚠️  T0 is {gap_from_first:.1f} hours BEFORE first track observation!")
        print(f"      First track position: {first_observation}")
        print(f"      NO lat/lon data available at T0 - position will be back-extrapolated")
    elif has_position_at_t0:
        print(f"   ✓  Storm position available at T0 (closest obs: {closest_time_diff:.1f}h away)")
    else:
        print(f"   ⚠️  NO storm position at T0 - will use extrapolated position")
    
    # Display warning if data gap is significant
    if data_gap_hours > 3 or not has_position_at_t0:
        display_storm_not_ready_warning(
            storm_name=storm_name,
            track_times=storm_times,
            forecast_time=time_t0,
            data_gap_hours=max(0, data_gap_hours),
            has_position_at_t0=has_position_at_t0
        )
else:
    # For historical storms, T0 is from find_optimal_aurora_init - no gap warning needed
    data_gap_hours = 0  # Not relevant for historical
    print(f"\n📊 Position Check:")
    print(f"   T0: {time_t0}")
    print(f"   Track covers: {first_observation} to {last_observation}")
    if has_position_at_t0:
        print(f"   ✓  Storm position available at T0 (closest obs: {closest_time_diff:.1f}h away)")
    else:
        print(f"   ⚠️  Storm position at T0 will be interpolated")

# =============================================================================
# DETERMINE STREAMS FOR T0 AND T-6h
# =============================================================================

init_hour = time_t0.hour

if init_hour in [0, 12]:
    stream_t0 = "oper"
    source_t0 = "Planetary Computer STAC"
else:
    stream_t0 = "scda"
    source_t0 = "Planetary Computer STAC"

print(f"\n   T0 hour: {init_hour}:00 UTC → {stream_t0.upper()} stream")

# For T-6h (6 hours before T0)
time_t_minus_6 = time_t0 - timedelta(hours=6)
hour_t_minus_6 = time_t_minus_6.hour

if hour_t_minus_6 in [0, 12]:
    stream_t_minus_6 = "oper"
    source_t_minus_6 = "Planetary Computer STAC"
else:
    stream_t_minus_6 = "scda"
    source_t_minus_6 = "Planetary Computer STAC"

print(f"\n📅 Required timesteps:")
print(f"   T0:   {time_t0} → {stream_t0.upper()} ({source_t0})")
print(f"   T-6h: {time_t_minus_6} → {stream_t_minus_6.upper()} ({source_t_minus_6})")

# =============================================================================
# STORE AURORA INITIALIZATION PARAMETERS
# =============================================================================
data_availability_warning_issued = data_gap_hours > 3 or not has_position_at_t0

aurora_init = {
    'time_t0': time_t0,
    'time_t_minus_6': time_t_minus_6,
    'stream_t0': stream_t0,
    'stream_t_minus_6': stream_t_minus_6,
    'source_t0': source_t0,
    'source_t_minus_6': source_t_minus_6,
    'storm_name': storm_name,
    'storm_init_time': storm_init_time,
    'data_gap_hours': data_gap_hours,
    'has_position_at_t0': has_position_at_t0,
}

print(f"\n✅ Aurora initialization configured:")
print(f"   Storm: {storm_name}")
print(f"   T0: {time_t0} ({stream_t0.upper()})")
print(f"   T-6h: {time_t_minus_6} ({stream_t_minus_6.upper()})")

# =============================================================================
# CREATE DOWNLOAD PLAN FOR SECTIONS 3.2 AND 3.3
# =============================================================================
print(f"\n" + "="*70)
print(f"📋 DOWNLOAD PLAN (for Sections 3.2 and 3.3)")
print(f"="*70)

oper_downloads = []
scda_downloads = []

# Add T0 to the appropriate list
if stream_t0 == "oper":
    oper_downloads.append(("T0", time_t0))
else:
    scda_downloads.append(("T0", time_t0))

# Add T-6h to the appropriate list  
if stream_t_minus_6 == "oper":
    oper_downloads.append(("T-6h", time_t_minus_6))
else:
    scda_downloads.append(("T-6h", time_t_minus_6))

print(f"\n   OPER downloads (Section 3.2): {len(oper_downloads)} file(s)")
for label, dt in oper_downloads:
    print(f"      - {label}: {dt}")

print(f"\n   SCDA downloads (Section 3.3): {len(scda_downloads)} file(s)")
for label, dt in scda_downloads:
    print(f"      - {label}: {dt}")

print(f"\n✅ Data availability check complete - proceed to Sections 3.2 and 3.3 for downloads")

### 3.2 Download OPER Stream Data (STAC API)

Query the Planetary Computer STAC catalog for operational forecast data at 00Z or 12Z UTC.

In [ ]:
# =============================================================================
# SECTION 3.2: DOWNLOAD OPER STREAM DATA (STAC/Planetary Computer)
# This stream is available at 0h and 12h UTC
# =============================================================================

from pathlib import Path
import urllib.request

# Create downloads directory if it doesn't exist
downloads_dir = Path("downloads")
downloads_dir.mkdir(exist_ok=True)

print("="*60)
print("📥 DOWNLOADING OPER STREAM DATA (STAC/Planetary Computer)")
print("="*60)

# Initialize file paths dict for OPER downloads
oper_file_paths = {}

if oper_downloads:
    for label, download_time in oper_downloads:
        print(f"\n🔍 Searching for OPER data: {label} @ {download_time}")
        
        # Search for 0h step from oper stream
        search_oper = catalog.search(
            collections=["ecmwf-forecast"],
            datetime=download_time,
            query={
                "ecmwf:stream": {"eq": "oper"},
                "ecmwf:type": {"eq": "fc"},
                "ecmwf:step": {"eq": "0h"},
            },
        )
        
        items_oper = list(search_oper.items())
        print(f"   Found {len(items_oper)} item(s)")
        
        # Filter for 0.25° resolution (required for aurora-0.25-finetuned model)
        # Item IDs contain resolution: e.g., ecmwf-2024-09-28T12-oper-fc-0h-0.25
        items_oper_025 = [item for item in items_oper if '0.25' in item.id]
        if items_oper_025:
            print(f"   Filtered to {len(items_oper_025)} item(s) at 0.25° resolution")
            items_oper = items_oper_025
        else:
            print(f"   ⚠️  No 0.25° items found, using available: {[i.id for i in items_oper]}")
        
        if items_oper:
            item_oper = items_oper[0]
            print(f"   Selected item: {item_oper.id}")
            
            # Download the file
            url_oper = item_oper.assets["data"].href
            local_filename = f"{download_time.strftime('%Y%m%d%H%M%S')}_0h_oper_{label.replace('-', '_')}.grib2"
            local_path = downloads_dir / local_filename
            
            if local_path.exists():
                print(f"   ⏭️  Skipping {local_filename} (already exists)")
            else:
                print(f"   ⬇️  Downloading {local_filename}...")
                urllib.request.urlretrieve(url_oper, local_path)
                file_size_mb = local_path.stat().st_size / 1024 / 1024
                print(f"      ✅ Saved to {local_path} ({file_size_mb:.1f} MB)")
            
            oper_file_paths[label] = local_path
        else:
            print(f"   ⚠️  No OPER data found for {download_time}")
            oper_file_paths[label] = None
else:
    print("\n   ℹ️  No OPER downloads needed for this init time")
    print(f"      (T0 and T-6h both use SCDA stream)")

print(f"\n✅ OPER downloads complete")
if oper_file_paths:
    for label, path in oper_file_paths.items():
        print(f"   {label}: {path}")


### 3.3 Download SCDA Stream Data (STAC API)

Download short cut-off high-resolution (SCDA) forecast data at 06Z or 18Z UTC from the Planetary Computer STAC API.

In [ ]:
# =============================================================================
# SECTION 3.3: DOWNLOAD SCDA STREAM DATA (STAC/Planetary Computer)
# This stream is available at 6h and 18h UTC
# =============================================================================

from pathlib import Path
import urllib.request

# Create downloads directory if it doesn't exist
downloads_dir = Path("downloads")
downloads_dir.mkdir(exist_ok=True)

print("="*60)
print("📥 DOWNLOADING SCDA STREAM DATA (STAC/Planetary Computer)")
print("="*60)

# Initialize file paths dict for SCDA downloads
scda_file_paths = {}

if scda_downloads:
    for label, download_time in scda_downloads:
        print(f"\n🔍 Searching for SCDA data: {label} @ {download_time}")
        
        # Search for 0h step from scda stream
        search_scda = catalog.search(
            collections=["ecmwf-forecast"],
            datetime=download_time,
            query={
                "ecmwf:stream": {"eq": "scda"},
                "ecmwf:type": {"eq": "fc"},
                "ecmwf:step": {"eq": "0h"},
            },
        )
        
        items_scda = list(search_scda.items())
        print(f"   Found {len(items_scda)} item(s)")
        
        # Filter for 0.25° resolution (required for aurora-0.25-finetuned model)
        # Item IDs contain resolution: e.g., ecmwf-2024-09-28T18-scda-fc-0h-0.25
        items_scda_025 = [item for item in items_scda if '0.25' in item.id]
        if items_scda_025:
            print(f"   Filtered to {len(items_scda_025)} item(s) at 0.25° resolution")
            items_scda = items_scda_025
        else:
            print(f"   ⚠️  No 0.25° items found, using available: {[i.id for i in items_scda]}")
        
        if items_scda:
            item_scda = items_scda[0]
            print(f"   Selected item: {item_scda.id}")
            
            # Download the file
            url_scda = item_scda.assets["data"].href
            local_filename = f"{download_time.strftime('%Y%m%d%H%M%S')}_0h_scda_{label.replace('-', '_')}.grib2"
            local_path = downloads_dir / local_filename
            
            if local_path.exists():
                print(f"   ⏭️  Skipping {local_filename} (already exists)")
            else:
                print(f"   ⬇️  Downloading {local_filename}...")
                urllib.request.urlretrieve(url_scda, local_path)
                file_size_mb = local_path.stat().st_size / 1024 / 1024
                print(f"      ✅ Saved to {local_path} ({file_size_mb:.1f} MB)")
            
            scda_file_paths[label] = local_path
        else:
            print(f"   ⚠️  No SCDA data found for {download_time}")
            scda_file_paths[label] = None
else:
    print("\n   ℹ️  No SCDA downloads needed for this init time")
    print(f"      (T0 and T-6h both use OPER stream)")

print(f"\n✅ SCDA downloads complete")
if scda_file_paths:
    for label, path in scda_file_paths.items():
        print(f"   {label}: {path}")


In [ ]:
# =============================================================================
# DOWNLOAD SUMMARY - COMBINE FILE PATHS FOR MERGE STEP (3.4)
# =============================================================================

print("="*60)
print("📁 DOWNLOAD SUMMARY")
print("="*60)

# Determine which file is T-6h and which is T0
# Files could come from either OPER or SCDA depending on the init hour
file_t_minus_6 = oper_file_paths.get('T-6h') or scda_file_paths.get('T-6h')
file_t0 = oper_file_paths.get('T0') or scda_file_paths.get('T0')

print(f"\n   T-6h file ({time_t_minus_6}):")
print(f"      Stream: {stream_t_minus_6.upper()}")
print(f"      Path:   {file_t_minus_6}")

print(f"\n   T0 file ({time_t0}):")
print(f"      Stream: {stream_t0.upper()}")
print(f"      Path:   {file_t0}")

# Validate both files exist
if file_t_minus_6 and file_t0:
    # Convert to Path objects if they're strings
    file_t_minus_6 = Path(file_t_minus_6) if isinstance(file_t_minus_6, str) else file_t_minus_6
    file_t0 = Path(file_t0) if isinstance(file_t0, str) else file_t0
    
    if file_t_minus_6.exists() and file_t0.exists():
        print(f"\n✅ Both timesteps downloaded successfully!")
        print(f"   Ready for merge in Section 3.4")
    else:
        missing = []
        if not file_t_minus_6.exists():
            missing.append(f"T-6h ({file_t_minus_6})")
        if not file_t0.exists():
            missing.append(f"T0 ({file_t0})")
        print(f"\n⚠️  File(s) not found on disk: {missing}")
else:
    missing = []
    if not file_t_minus_6:
        missing.append("T-6h")
    if not file_t0:
        missing.append("T0")
    print(f"\n❌ Missing downloads: {missing}")
    print(f"   Please check the download logs above for errors.")

### 3.4 Merge GRIB Files with xarray

Combine the two GRIB files (T-6h and T0) into a single xarray Dataset with 2 timesteps. This step also:
- Extracts surface variables (2m temperature, 10m winds, mean sea level pressure)
- Converts longitude from [-180°, 180°] to [0°, 360°] format (Aurora requirement)

In [ ]:
# Open GRIB files with surface variable handling

import xarray as xr
from pathlib import Path

def open_grib_file(filepath):
    """Open and process a GRIB2 file with surface variable handling."""
    
    # Open without filter for atmospheric variables
    ds_unfiltered = xr.open_dataset(filepath, engine="cfgrib")
    
    datasets_to_merge = [ds_unfiltered]
    
    # Extract t2m using shortName filter (more reliable than typeOfLevel + level)
    # The GRIB file uses shortName='2t' which maps to variable name 't2m'
    try:
        ds_2m = xr.open_dataset(
            filepath, 
            engine="cfgrib",
            backend_kwargs={'filter_by_keys': {'shortName': '2t'}}
        )
        if 't2m' in ds_2m.data_vars:
            datasets_to_merge.append(ds_2m)
            print(f"      ✓ Extracted t2m using shortName='2t'")
    except Exception as e:
        print(f"      ⚠️ Could not extract t2m: {e}")
    
    # Extract u10, v10 at 10m height using shortName filters
    try:
        ds_u10 = xr.open_dataset(
            filepath, 
            engine="cfgrib",
            backend_kwargs={'filter_by_keys': {'shortName': '10u'}}
        )
        datasets_to_merge.append(ds_u10)
        print(f"      ✓ Extracted u10 using shortName='10u'")
    except Exception as e:
        print(f"      ⚠️ Could not extract u10: {e}")
    
    try:
        ds_v10 = xr.open_dataset(
            filepath, 
            engine="cfgrib",
            backend_kwargs={'filter_by_keys': {'shortName': '10v'}}
        )
        datasets_to_merge.append(ds_v10)
        print(f"      ✓ Extracted v10 using shortName='10v'")
    except Exception as e:
        print(f"      ⚠️ Could not extract v10: {e}")
    
    # Combine all datasets
    ds_combined = xr.merge(datasets_to_merge, compat='override')
    
    return ds_combined

# =============================================================================
# Open both GRIB files using the smart stream selection
# file_t_minus_6 = first timestep (T-6h), file_t0 = second timestep (T0)
# =============================================================================

if file_t_minus_6 and file_t0: 
    # Open T-6h file (first timestep)
    print(f"\n📄 Opening T-6h file ({time_t_minus_6}):")
    print(f"   Source: {file_t_minus_6}")
    ds_t_minus_6 = open_grib_file(file_t_minus_6)
    print(f"   Variables: {list(ds_t_minus_6.data_vars)}")
    
    # Open T0 file (second timestep)
    print(f"\n📄 Opening T0 file ({time_t0}):")
    print(f"   Source: {file_t0}")
    ds_t0_file = open_grib_file(file_t0)
    print(f"   Variables: {list(ds_t0_file.data_vars)}")
else:
    missing = []
    if not file_t_minus_6:
        missing.append("T-6h")
    if not file_t0:
        missing.append("T0")
    raise ValueError(f"Missing GRIB files: {missing}. Please check the download steps.")

In [ ]:
# Combine datasets along time dimension

print("Merging datasets...")

# Check the coordinates in each dataset
print(f"\n   ds_t_minus_6 coords: {list(ds_t_minus_6.coords)}")
print(f"   ds_t0_file coords: {list(ds_t0_file.coords)}")

# Drop conflicting scalar coordinates that aren't needed for the merge
# valid_time and step are scalar coordinates that differ between the two files
coords_to_drop = ['valid_time', 'step', 'surface', 'meanSea', 'entireAtmosphere', 'depthBelowLandLayer']
ds_t_minus_6_clean = ds_t_minus_6.drop_vars([c for c in coords_to_drop if c in ds_t_minus_6.coords], errors='ignore')
ds_t0_clean = ds_t0_file.drop_vars([c for c in coords_to_drop if c in ds_t0_file.coords], errors='ignore')

print(f"\n   After cleanup - ds_t_minus_6 coords: {list(ds_t_minus_6_clean.coords)}")
print(f"   After cleanup - ds_t0 coords: {list(ds_t0_clean.coords)}")

# Combine the two datasets: T-6h (first) + T0 (second)
# IMPORTANT: Order matters! Aurora expects [T-6h, T0] along time dimension
ds = xr.concat([ds_t_minus_6_clean, ds_t0_clean], dim='time', compat='override', coords='minimal')

print(f"\n✅ Combined datasets with {len(ds.time)} timesteps")
print(f"   Time dimension: {ds.time.values}")
print(f"   Expected: T-6h={time_t_minus_6}, T0={time_t0}")
print(f"   Variables: {list(ds.data_vars)}")

# =============================================================================
# Convert longitudes from -180/180 to 0/360 format (Aurora requires 0-360)
# IMPORTANT: Use roll() to properly shift BOTH data AND coordinates together!
# =============================================================================
print(f"\n📊 Converting longitude from -180/180 to 0/360 format...")
print(f"   Original longitude range: {ds.longitude.values.min():.1f} to {ds.longitude.values.max():.1f}")

# Get original longitude values
lon_orig = ds.longitude.values

# Count negative longitude points (they need to be rolled to the end)
n_negative = np.sum(lon_orig < 0)
print(f"   Negative longitude points to roll: {n_negative}")

# Roll the dataset along longitude - this shifts BOTH coordinates AND data together
ds = ds.roll(longitude=-n_negative, roll_coords=True)

# Now convert the coordinate labels from -180/180 to 0/360
lon_rolled = ds.longitude.values
lon_360 = np.where(lon_rolled < 0, lon_rolled + 360, lon_rolled)
ds = ds.assign_coords(longitude=lon_360)

print(f"   Converted longitude range: {ds.longitude.values.min():.1f} to {ds.longitude.values.max():.1f}")
print(f"   Longitude is monotonic increasing: {np.all(np.diff(ds.longitude.values) > 0)}")

### 3.5 Upload to Azure GeoCatalog

Upload the processed storm data to Microsoft Planetary Computer Pro's GeoCatalog for visualization and sharing. This creates a STAC collection with render options for interactive exploration in the GeoCatalog Explorer.

#### 3.5.1 Upload Data to Azure Blob Storage

Save the processed dataset as NetCDF and upload to Azure Blob Storage for GeoCatalog ingestion.

In [ ]:
# Upload ds dataset to blob storage as NetCDF

import random
import string
import asyncio
import aiofiles
from pathlib import Path
from typing import Tuple
from urllib.parse import urlparse
from datetime import datetime, timedelta, timezone
from azure.identity import DefaultAzureCredential
from azure.storage.blob import BlobServiceClient, generate_blob_sas, BlobSasPermissions

# Configuration - use environment variables
BLOB_STORAGE_SAS = os.getenv("AURORA_BLOB_STORAGE_SAS")
UPLOAD_CONTAINER = os.getenv("UPLOAD_CONTAINER_NAME", "hurricane-data")
STORAGE_ACCOUNT_KEY = os.getenv("STORAGE_ACCOUNT_KEY")
credential = DefaultAzureCredential()

def generate_unique_suffix() -> str:
    """Generate a unique 5-character alphanumeric suffix."""
    return ''.join(random.choices(string.ascii_lowercase + string.digits, k=5))

async def upload_to_blob_storage(file_path: Path, blob_name: str) -> Tuple[str, str]:
    """Upload file to blob storage and return (blob_url, sas_url)."""
    print(f"Uploading {blob_name}...")
    
    parsed_url = urlparse(BLOB_STORAGE_SAS)
    
    if not parsed_url.hostname:
        raise ValueError(
            f"Invalid BLOB_STORAGE_SAS URL: missing hostname.\n"
            f"Expected format: https://accountname.blob.core.windows.net/?sv=..."
        )
    
    account_url = f"{parsed_url.scheme}://{parsed_url.netloc}"
    container_name = UPLOAD_CONTAINER
    
    print(f"  Container: {container_name}")
    print(f"  Account: {account_url}")
    
    blob_service_client = BlobServiceClient(account_url=account_url, credential=credential)
    
    container_client = blob_service_client.get_container_client(container_name)
    try:
        await asyncio.get_event_loop().run_in_executor(
            None, lambda: container_client.get_container_properties()
        )
        print(f"  ✓ Container '{container_name}' exists")
    except Exception as e:
        if "ContainerNotFound" in str(e):
            print(f"  Creating container '{container_name}'...")
            try:
                await asyncio.get_event_loop().run_in_executor(
                    None, lambda: container_client.create_container()
                )
                print(f"  ✓ Container '{container_name}' created")
            except Exception as create_error:
                print(f"  ❌ Failed to create container: {create_error}")
                raise
        else:
            print(f"  ⚠️  Error checking container: {e}")
    
    blob_client = blob_service_client.get_blob_client(container=container_name, blob=blob_name)
    
    async with aiofiles.open(file_path, 'rb') as data:
        file_content = await data.read()
        await asyncio.get_event_loop().run_in_executor(
            None, lambda: blob_client.upload_blob(file_content, overwrite=True)
        )
    
    blob_url = f"{account_url}/{container_name}/{blob_name}"
    
    account_name = parsed_url.hostname.split('.')[0]
    sas_token = generate_blob_sas(
        account_name=account_name,
        container_name=container_name,
        blob_name=blob_name,
        account_key=STORAGE_ACCOUNT_KEY,
        permission=BlobSasPermissions(read=True),
        expiry=datetime.now(timezone.utc) + timedelta(days=30),
        start=datetime.now(timezone.utc)
    )
    sas_url = f"{blob_url}?{sas_token}"
    
    print(f"✓ Uploaded to {blob_url}")
    return blob_url, sas_url

# Save ds to NetCDF and upload to blob storage
print("="*80)
print("UPLOADING DS TO BLOB STORAGE (Separate files per timestamp)")
print("="*80)

# ============================================================================
# Create separate datasets for each timestamp:
# 1. Surface variables (2D) - for direct rendering
# 2. Atmospheric variables (3D with level) - use 'sel' query for rendering
# ============================================================================

print("\n🔄 Creating separate 2D surface datasets for each timestamp...")

# Identify the pressure level dimension name
level_dim = None
for dim_name in ['isobaricInhPa', 'level', 'pressure_level']:
    if dim_name in ds.dims:
        level_dim = dim_name
        break

# Identify surface variables (those WITHOUT the level dimension)
if level_dim:
    print(f"   Found pressure level dimension: '{level_dim}'")
    surface_vars = [var for var in ds.data_vars if level_dim not in ds[var].dims]
    pressure_vars = [var for var in ds.data_vars if level_dim in ds[var].dims]
    print(f"   Surface variables: {surface_vars}")
    print(f"   Pressure level variables (excluded): {pressure_vars}")
else:
    print("   No pressure level dimension found")
    surface_vars = list(ds.data_vars)

# Create base surface dataset (still has time dimension)
ds_surface_base = ds[surface_vars].copy()

# ============================================================================
# USE THE ACTUAL KNOWN TIMESTAMPS (time1, time2) instead of dataset time values
# The dataset's time dimension may have incorrect values (e.g., epoch 1970)
# ============================================================================
from datetime import timedelta

# Define the actual timestamps we downloaded data for
actual_timestamps = [
    storm_init_time,                      # time1 - first timestep
    storm_init_time + timedelta(hours=6)  # time2 - second timestep (6 hours later)
]

# Get number of time steps in dataset
if 'time' in ds_surface_base.dims:
    num_timesteps = len(ds_surface_base['time'])
    print(f"   Dataset has {num_timesteps} time steps")
    # Make sure we have matching number of actual timestamps
    actual_timestamps = actual_timestamps[:num_timesteps]
else:
    num_timesteps = 1
    actual_timestamps = [storm_init_time]

print(f"   Using ACTUAL timestamps: {actual_timestamps}")

# Generate unique suffix for this upload batch
unique_suffix = generate_unique_suffix()

# Store upload results for each timestamp
uploaded_files = []  # List of (timestamp, blob_url, sas_url, ds_slice)

print(f"\n📤 Uploading {len(actual_timestamps)} separate NetCDF files...")

for i, ts in enumerate(actual_timestamps):
    print(f"\n--- Timestamp {i+1}/{len(actual_timestamps)}: {ts} ---")
    
    # Select this time step and drop time dimension to make 2D
    if 'time' in ds_surface_base.dims:
        ds_slice = ds_surface_base.isel(time=i, drop=True)
    else:
        ds_slice = ds_surface_base
    
    # =========================================================================
    # ADD CF-COMPLIANT ATTRIBUTES FOR GEOCATALOG RENDERING
    # cf_xarray requires axis attributes to identify X/Y dimensions
    # =========================================================================
    # Check for latitude coordinate (could be 'latitude' or 'lat')
    lat_coord = 'latitude' if 'latitude' in ds_slice.coords else ('lat' if 'lat' in ds_slice.coords else None)
    lon_coord = 'longitude' if 'longitude' in ds_slice.coords else ('lon' if 'lon' in ds_slice.coords else None)
    
    # =========================================================================
    # CONVERT LONGITUDE BACK TO [-180, 180] FOR GEOCATALOG RENDERING
    # Aurora uses [0, 360] but GIS tools/titiler expect [-180, 180]
    # We need to: 1) roll data so 180-360 portion comes first, 2) relabel coords
    # =========================================================================
    if lon_coord:
        lon_vals = ds_slice[lon_coord].values
        if lon_vals.max() > 180:
            print(f"   🔄 Converting longitude from [0,360] to [-180,180]...")
            print(f"      Original lon range: {lon_vals.min():.1f} to {lon_vals.max():.1f}")
            
            # Sanity check: capture a data value at a known location BEFORE roll
            # Pick a point: lon=270 (which becomes -90)
            # Find a variable that has the expected 2D (lat, lon) shape
            test_lon_idx = np.abs(lon_vals - 270).argmin()  # ~270 degrees
            lat_size = len(ds_slice[lat_coord])
            test_lat_idx = min(lat_size // 2, lat_size - 1)  # middle latitude, bounded
            
            # Find a variable with standard 2D shape (lat, lon) for sanity check
            first_var = None
            for var_name in ds_slice.data_vars:
                var_shape = ds_slice[var_name].shape
                if len(var_shape) == 2 and var_shape[0] == lat_size:
                    first_var = var_name
                    break
            
            if first_var is not None:
                val_before = float(ds_slice[first_var].values[test_lat_idx, test_lon_idx])
                lon_before = float(lon_vals[test_lon_idx])
                print(f"      Sanity check - {first_var} at lon={lon_before:.1f}°: {val_before:.2f}")
            else:
                val_before = None
                lon_before = float(lon_vals[test_lon_idx])
                print(f"      Skipping sanity check - no standard 2D variable found")
            
            # Find split point: index where longitude >= 180
            # Data at indices >= split_idx will move to the beginning (become negative lons)
            split_idx = np.searchsorted(lon_vals, 180.0)
            n_to_roll = len(lon_vals) - split_idx  # number of points with lon >= 180
            
            print(f"      Split at index {split_idx}, rolling {n_to_roll} points to front")
            
            # Roll dataset: move lon>=180 portion to the front
            # roll_coords=True ensures BOTH data arrays AND coordinates shift together
            ds_slice = ds_slice.roll({lon_coord: n_to_roll}, roll_coords=True)
            
            # Now relabel: values that were >= 180 are now at front, subtract 360
            new_lon = ds_slice[lon_coord].values.copy()
            new_lon = np.where(new_lon >= 180, new_lon - 360, new_lon)
            ds_slice = ds_slice.assign_coords({lon_coord: new_lon})
            
            # Verify the data rolled correctly: same value should now be at lon=-90
            if first_var is not None:
                test_lon_after = -90.0
                test_lon_idx_after = np.abs(ds_slice[lon_coord].values - test_lon_after).argmin()
                val_after = float(ds_slice[first_var].values[test_lat_idx, test_lon_idx_after])
                lon_after = float(ds_slice[lon_coord].values[test_lon_idx_after])
                print(f"      After roll - {first_var} at lon={lon_after:.1f}°: {val_after:.2f}")
                
                if val_before is not None and abs(val_before - val_after) < 0.01:
                    print(f"      ✓ Data integrity verified: values match!")
                elif val_before is not None:
                    print(f"      ⚠️  WARNING: Values don't match! Roll may be incorrect.")
            else:
                print(f"      Skipping post-roll verification - no standard 2D variable")
            
            print(f"      Converted lon range: {ds_slice[lon_coord].values.min():.1f} to {ds_slice[lon_coord].values.max():.1f}")
            print(f"      Longitude monotonic increasing: {np.all(np.diff(ds_slice[lon_coord].values) > 0)}")
    
    if lat_coord:
        ds_slice[lat_coord].attrs['axis'] = 'Y'
        ds_slice[lat_coord].attrs['standard_name'] = 'latitude'
        ds_slice[lat_coord].attrs['units'] = 'degrees_north'
        ds_slice[lat_coord].attrs['long_name'] = 'latitude'
    
    if lon_coord:
        ds_slice[lon_coord].attrs['axis'] = 'X'
        ds_slice[lon_coord].attrs['standard_name'] = 'longitude'
        ds_slice[lon_coord].attrs['units'] = 'degrees_east'
        ds_slice[lon_coord].attrs['long_name'] = 'longitude'
    
    print(f"   ✓ Added CF-compliant coordinate attributes (X={lon_coord}, Y={lat_coord})")
    
    # ts is already the actual timestamp (datetime) - no conversion needed
    print(f"   📅 Using timestamp: {ts} (type: {type(ts).__name__})")
    
    # Print shape info
    print(f"   Variables: {list(ds_slice.data_vars)}")
    print(f"   Dimensions: {dict(ds_slice.dims)}")
    for var in list(ds_slice.data_vars)[:2]:  # Show first 2
        print(f"   {var}: shape={ds_slice[var].shape}")
    
    # Generate filename with timestamp
    date_str = ts.strftime("%Y%m%d")
    time_str = ts.strftime("%H%M")
    netcdf_filename = f"hurricane_{storm_name.lower()}_{date_str}_{time_str}_{unique_suffix}_t{i}.nc"
    netcdf_path = downloads_dir / netcdf_filename
    
    # Save to NetCDF
    print(f"   Saving to {netcdf_filename}...")
    ds_slice.to_netcdf(netcdf_path, engine='netcdf4')
    print(f"   ✓ Saved ({netcdf_path.stat().st_size / (1024*1024):.2f} MB)")
    
    # Upload to blob storage
    blob_url, sas_url = await upload_to_blob_storage(netcdf_path, netcdf_filename)
    
    # Store results
    uploaded_files.append({
        'index': i,
        'timestamp': ts,
        'blob_url': blob_url,
        'sas_url': sas_url,
        'ds_slice': ds_slice,
        'filename': netcdf_filename,
        'type': 'surface'
    })
    
    print(f"   ✅ Uploaded: {blob_url}")

print(f"\n✅ All {len(uploaded_files)} surface files uploaded successfully!")
for f in uploaded_files:
    print(f"   {f['index']+1}. {f['timestamp']} -> {f['filename']}")

# ============================================================================
# UPLOAD ATMOSPHERIC VARIABLES (3D with level dimension)
# These will use 'sel' parameter in render options to slice to 2D
# ============================================================================
print("\n" + "="*80)
print("UPLOADING ATMOSPHERIC VARIABLES (3D with level dimension)")
print("="*80)

if level_dim and pressure_vars:
    print(f"\n🌡️ Atmospheric variables to upload: {pressure_vars}")
    print(f"   Level dimension: '{level_dim}'")
    print(f"   Levels: {list(ds[level_dim].values)}")
    
    # Create atmospheric dataset base (with level dimension)
    ds_atmos_base = ds[pressure_vars].copy()
    
    # Store atmospheric upload results
    uploaded_atmos_files = []
    
    for i, ts in enumerate(actual_timestamps):
        print(f"\n--- Atmospheric Timestamp {i+1}/{len(actual_timestamps)}: {ts} ---")
        
        # Select this time step but KEEP the level dimension (makes it 3D: level, lat, lon)
        if 'time' in ds_atmos_base.dims:
            ds_atmos_slice = ds_atmos_base.isel(time=i, drop=True)
        else:
            ds_atmos_slice = ds_atmos_base
        
        # Get coordinate names
        lat_coord = 'latitude' if 'latitude' in ds_atmos_slice.coords else ('lat' if 'lat' in ds_atmos_slice.coords else None)
        lon_coord = 'longitude' if 'longitude' in ds_atmos_slice.coords else ('lon' if 'lon' in ds_atmos_slice.coords else None)
        
        # Convert longitude back to [-180, 180] for GeoCatalog
        if lon_coord:
            lon_vals = ds_atmos_slice[lon_coord].values
            if lon_vals.max() > 180:
                print(f"   🔄 Converting longitude from [0,360] to [-180,180]...")
                split_idx = np.searchsorted(lon_vals, 180.0)
                n_to_roll = len(lon_vals) - split_idx
                ds_atmos_slice = ds_atmos_slice.roll({lon_coord: n_to_roll}, roll_coords=True)
                new_lon = ds_atmos_slice[lon_coord].values.copy()
                new_lon = np.where(new_lon >= 180, new_lon - 360, new_lon)
                ds_atmos_slice = ds_atmos_slice.assign_coords({lon_coord: new_lon})
                print(f"      Converted range: {ds_atmos_slice[lon_coord].values.min():.1f} to {ds_atmos_slice[lon_coord].values.max():.1f}")
        
        # Add CF-compliant attributes
        if lat_coord:
            ds_atmos_slice[lat_coord].attrs['axis'] = 'Y'
            ds_atmos_slice[lat_coord].attrs['standard_name'] = 'latitude'
            ds_atmos_slice[lat_coord].attrs['units'] = 'degrees_north'
        if lon_coord:
            ds_atmos_slice[lon_coord].attrs['axis'] = 'X'
            ds_atmos_slice[lon_coord].attrs['standard_name'] = 'longitude'
            ds_atmos_slice[lon_coord].attrs['units'] = 'degrees_east'
        
        # Add attributes to level dimension for CF compliance
        if level_dim in ds_atmos_slice.coords:
            ds_atmos_slice[level_dim].attrs['axis'] = 'Z'
            ds_atmos_slice[level_dim].attrs['units'] = 'hPa'
            ds_atmos_slice[level_dim].attrs['positive'] = 'down'
            ds_atmos_slice[level_dim].attrs['standard_name'] = 'air_pressure'
        
        # Print shape info
        print(f"   Variables: {list(ds_atmos_slice.data_vars)}")
        print(f"   Dimensions: {dict(ds_atmos_slice.dims)}")
        for var in list(ds_atmos_slice.data_vars)[:2]:
            print(f"   {var}: shape={ds_atmos_slice[var].shape}")
        
        # Generate filename
        date_str = ts.strftime("%Y%m%d")
        time_str = ts.strftime("%H%M")
        atmos_filename = f"hurricane_{storm_name.lower()}_{date_str}_{time_str}_{unique_suffix}_atmos_t{i}.nc"
        atmos_path = downloads_dir / atmos_filename
        
        # Save to NetCDF
        print(f"   Saving to {atmos_filename}...")
        ds_atmos_slice.to_netcdf(atmos_path, engine='netcdf4')
        print(f"   ✓ Saved ({atmos_path.stat().st_size / (1024*1024):.2f} MB)")
        
        # Upload to blob storage
        blob_url, sas_url = await upload_to_blob_storage(atmos_path, atmos_filename)
        
        # Store results
        uploaded_atmos_files.append({
            'index': i,
            'timestamp': ts,
            'blob_url': blob_url,
            'sas_url': sas_url,
            'ds_slice': ds_atmos_slice,
            'filename': atmos_filename,
            'type': 'atmospheric',
            'level_dim': level_dim,
            'levels': list(ds_atmos_slice[level_dim].values)
        })
        
        print(f"   ✅ Uploaded: {blob_url}")
    
    print(f"\n✅ All {len(uploaded_atmos_files)} atmospheric files uploaded successfully!")
    for f in uploaded_atmos_files:
        print(f"   {f['index']+1}. {f['timestamp']} -> {f['filename']} (levels: {len(f['levels'])})")
else:
    print("\n⚠️ No atmospheric variables found to upload")
    uploaded_atmos_files = []

#### 3.5.2 Ingest Data to GeoCatalog STAC Collection

Create a STAC collection and ingest items with datacube metadata for visualization in GeoCatalog Explorer.

In [ ]:
# Ingest the data to MPC STAC collection

import aiohttp
import pystac
from typing import List
import json

# Collection configuration for Hurricane data
INPUT_COLLECTION = f"hurricane-{storm_name.lower()}-{storm_year}-Input-Data"
API_VERSION = "2025-04-30-preview"

def join_url(base: str, *paths: str) -> str:
    """Properly join URL parts, avoiding double slashes."""
    url = base.rstrip('/')
    for path in paths:
        path = path.lstrip('/')
        url = f"{url}/{path}"
    return url

def get_bearer_token():
    """Get bearer token for GeoCatalog API."""
    access_token = credential.get_token(f"{SPATIO_APP_ID}/.default")
    return {"Authorization": f"Bearer {access_token.token}"}

async def ensure_collection_exists(collection_id: str, title: str, description: str):
    """Create collection if it doesn't exist."""
    print(f"Checking if collection '{collection_id}' exists...")
    
    headers = get_bearer_token()
    params = {"api-version": API_VERSION}
    
    timeout = aiohttp.ClientTimeout(total=300)
    async with aiohttp.ClientSession(timeout=timeout) as session:
        print(f"  Testing connection to: {STAC_API_URL}")
        try:
            async with session.get(STAC_API_URL, headers=headers, params=params) as response:
                if response.status == 404:
                    print(f"  ⚠️  Catalog root not found (404)")
                elif response.status >= 400:
                    error_text = await response.text()
                    print(f"  ⚠️  Catalog root returned {response.status}: {error_text}")
                else:
                    print(f"  ✓ Catalog root accessible")
        except Exception as e:
            print(f"  ❌ Error accessing catalog: {e}")
        
        # Check if collection exists
        collection_url = join_url(STAC_API_URL, "collections", collection_id)
        print(f"  Checking: {collection_url}")
        
        async with session.get(collection_url, headers=headers, params=params) as response:
            if response.status == 200:
                print(f"✓ Collection '{collection_id}' already exists")
                return
            elif response.status == 404:
                print(f"  Collection not found, will create it")
            else:
                error_text = await response.text()
                print(f"⚠️  Unexpected response: {response.status} - {error_text}")
        
        # Create collection
        print(f"Creating collection '{collection_id}'...")
        
        # Define item_assets for the collection, this is useful for rendering in mpc 
        item_assets = {
            "data": {
                "type": "application/x-netcdf",
                "roles": ["data"],
                "title": f"Hurricane {storm_name} ECMWF Data",
                "description": f"Combined ECMWF atmospheric data for Hurricane {storm_name} in NetCDF format",
                "xarray:open_kwargs": {
                    "engine": "h5netcdf"
                }
            },
            "forecast_surface": {
                "type": "application/x-netcdf",
                "title": "Aurora Surface Weather Forecast NetCDF",
                "description": "Aurora model surface weather forecasts including 2m temperature, 10m winds, and mean sea level pressure",
                "roles": ["data"],
                "xarray:open_kwargs": {
                    "engine": "h5netcdf"
                }
            },
            "forecast_atmospheric": {
                "type": "application/x-netcdf",
                "title": "Aurora Atmospheric Weather Forecast NetCDF",
                "description": "Aurora model atmospheric weather forecasts on pressure levels",
                "roles": ["data"],
                "xarray:open_kwargs": {
                    "engine": "h5netcdf"
                }
            }
        }
        
        collection_dict = {
            "id": collection_id,
            "type": "Collection",
            "stac_version": "1.0.0",
            "title": title,
            "description": description,
            "license": "proprietary",
            "extent": {
                "spatial": {
                    "bbox": [[-180, -90, 180, 90]]
                },
                "temporal": {
                    "interval": [[
                        storm_init_time.strftime("%Y-%m-%dT%H:%M:%SZ"),
                        storm_end_time.strftime("%Y-%m-%dT%H:%M:%SZ")
                    ]]
                }
            },
            "item_assets": item_assets,
            "links": []
        }
        
        collections_url = join_url(STAC_API_URL, "collections")
        print(f"  Posting to: {collections_url}")
        
        async with session.post(
            collections_url,
            json=collection_dict,
            headers={**headers, "Content-Type": "application/json"},
            params=params
        ) as response:
            if response.status >= 400:
                error_text = await response.text()
                print(f"❌ Failed to create collection: {response.status}")
                print(f"   Response: {error_text}")
                raise Exception(f"Failed to create collection: {response.status}")
            print(f"✓ Collection '{collection_id}' created successfully (status {response.status})")
            
            # Wait for collection to be available (202 = accepted, needs time to propagate)
            if response.status == 202:
                print(f"  ⏳ Waiting for collection to become available...")
                max_retries = 10
                retry_delay = 3  # seconds
                
                for attempt in range(max_retries):
                    await asyncio.sleep(retry_delay)
                    async with session.get(collection_url, headers=headers, params=params) as check_response:
                        if check_response.status == 200:
                            print(f"  ✓ Collection is now available (attempt {attempt + 1})")
                            break
                        else:
                            print(f"  ... still waiting (attempt {attempt + 1}/{max_retries})")
                else:
                    print(f"  ⚠️  Collection may not be fully available yet, proceeding anyway")

async def create_stac_item(item_id: str, date_time: datetime, bbox: List[float], 
                          data_url: str, dataset=None) -> pystac.Item:
    """Create a STAC item for the uploaded dataset with proper metadata for rendering.
    
    Args:
        item_id: Unique identifier for the item
        date_time: Datetime of the data
        bbox: Bounding box [west, south, east, north]
        data_url: URL to the NetCDF asset
        dataset: Optional xarray Dataset to extract variable metadata
    """
    
    geometry = {
        "type": "Polygon",
        "coordinates": [[
            [bbox[0], bbox[1]],
            [bbox[2], bbox[1]],
            [bbox[2], bbox[3]],
            [bbox[0], bbox[3]],
            [bbox[0], bbox[1]]
        ]]
    }
    
    # Build properties - keep it simple to avoid validation issues
    properties = {
        "title": f"Hurricane {storm_name} {storm_year} Input Data",
        "description": f"Combined ECMWF data for Hurricane {storm_name} at {date_time.isoformat()}",
        "data_source": "ECMWF",
        "event": f"Hurricane {storm_name} {storm_year}",
        "spatial_coverage": "global",
    }
    
    # Add datacube extension properties if dataset provided
    # Note: Removed reference_system to avoid projjson schema validation issues
    if dataset is not None:
        # Extract variable names from the dataset
        var_names = list(dataset.data_vars)
        coord_names = list(dataset.coords)
        
        # Build cube:dimensions (without reference_system to avoid projjson schema validation)
        cube_dimensions = {}
        if 'latitude' in dataset.coords:
            lat_vals = dataset.coords['latitude'].values
            cube_dimensions['latitude'] = {
                "type": "spatial",
                "axis": "y",
                "extent": [float(lat_vals.min()), float(lat_vals.max())]
            }
        if 'longitude' in dataset.coords:
            lon_vals = dataset.coords['longitude'].values
            cube_dimensions['longitude'] = {
                "type": "spatial",
                "axis": "x", 
                "extent": [float(lon_vals.min()), float(lon_vals.max())]
            }
        if 'time' in dataset.coords:
            time_vals = dataset.coords['time'].values
            cube_dimensions['time'] = {
                "type": "temporal",
                "extent": [str(time_vals.min()), str(time_vals.max())]
            }
        if 'level' in dataset.coords:
            level_vals = dataset.coords['level'].values
            cube_dimensions['level'] = {
                "type": "vertical",
                "values": [int(v) for v in level_vals]
            }
        
        # Build cube:variables
        cube_variables = {}
        for var_name in var_names:
            var_data = dataset[var_name]
            cube_variables[var_name] = {
                "type": "data",
                "dimensions": list(var_data.dims),
                "unit": str(var_data.attrs.get('units', 'unknown')),
                "description": str(var_data.attrs.get('long_name', var_name))
            }
        
        properties["cube:dimensions"] = cube_dimensions
        properties["cube:variables"] = cube_variables
        
        # Add list of variables for easy reference
        properties["variables"] = var_names
        
        # Build raster:bands for the asset (helps GeoCatalog understand raster structure)
        raster_bands = []
        for i, var_name in enumerate(var_names):
            var_data = dataset[var_name]
            band_info = {
                "name": var_name,
                "description": str(var_data.attrs.get('long_name', var_name)),
                "data_type": str(var_data.dtype),
                "unit": str(var_data.attrs.get('units', 'unknown')),
            }
            # Add statistics if we can compute them (skip NaN values)
            try:
                import numpy as np
                min_val = float(var_data.min().values)
                max_val = float(var_data.max().values)
                # Only add statistics if they are valid (not NaN or Inf)
                if np.isfinite(min_val) and np.isfinite(max_val):
                    band_info["statistics"] = {
                        "minimum": min_val,
                        "maximum": max_val,
                    }
            except:
                pass  # Skip statistics if computation fails
            raster_bands.append(band_info)
    
    # Define STAC extensions used (removed projection extension to avoid validation issues)
    stac_extensions = [
        "https://stac-extensions.github.io/datacube/v2.2.0/schema.json",
        "https://stac-extensions.github.io/raster/v1.1.0/schema.json"
    ]
    
    item = pystac.Item(
        id=item_id,
        geometry=geometry,
        bbox=bbox,
        datetime=date_time,
        properties=properties,
        stac_extensions=stac_extensions
    )
    
    # Build asset extra_fields with raster:bands
    asset_extra_fields = {
        "type": "application/x-netcdf",
        "xarray:open_kwargs": {
            "engine": "h5netcdf"
        }
    }
    
    # Add raster:bands if we have dataset info
    if dataset is not None:
        asset_extra_fields["raster:bands"] = raster_bands
    
    # Add the data asset with proper extra_fields for NetCDF
    item.add_asset(
        "data",
        pystac.Asset(
            href=data_url,
            media_type="application/x-netcdf",  # Correct media type for NetCDF
            title=f"Hurricane {storm_name} Data (NetCDF)",
            roles=["data"],
            extra_fields=asset_extra_fields
        )
    )
    
    return item

async def check_item_exists(item_id: str, collection_id: str) -> bool:
    """Check if a STAC item already exists in the collection."""
    headers = get_bearer_token()
    params = {"api-version": API_VERSION}
    item_url = join_url(STAC_API_URL, "collections", collection_id, "items", item_id)
    
    timeout = aiohttp.ClientTimeout(total=30)
    async with aiohttp.ClientSession(timeout=timeout) as session:
        async with session.get(item_url, headers=headers, params=params) as response:
            return response.status == 200

async def ingest_to_geocatalog(item: pystac.Item, collection_id: str, skip_if_exists: bool = True):
    """Ingest STAC item into GeoCatalog collection."""
    if skip_if_exists and await check_item_exists(item.id, collection_id):
        print(f"✓ Item {item.id} already exists in collection {collection_id}, skipping")
        return
    
    print(f"Ingesting item {item.id} to collection {collection_id}...")
    
    headers = get_bearer_token()
    params = {"api-version": API_VERSION}
    item_dict = item.to_dict()
    
    collection_items_url = join_url(STAC_API_URL, "collections", collection_id, "items")
    print(f"  Posting to: {collection_items_url}")
    
    timeout = aiohttp.ClientTimeout(total=300)
    async with aiohttp.ClientSession(timeout=timeout) as session:
        async with session.post(
            collection_items_url,
            json=item_dict,
            headers={**headers, "Content-Type": "application/json"},
            params=params
        ) as response:
            if response.status >= 400:
                error_text = await response.text()
                print(f"❌ Failed to ingest item: {response.status}")
                print(f"   Response: {error_text}")
                raise Exception(f"Failed to ingest item: {response.status}")
            
            print(f"✓ Successfully ingested item {item.id} (status {response.status})")

# ============================================================================
# INGEST MULTIPLE STAC ITEMS TO GEOCATALOG (one per timestamp)
# Each item has TWO assets: surface (2D) and atmospheric (3D with level)
# ============================================================================
print("="*80)
print("INGESTING STORM DATA TO GEOCATALOG STAC COLLECTION")
print("="*80)

# Ensure collection exists
await ensure_collection_exists(
    INPUT_COLLECTION,
    f"Hurricane {storm_name} {storm_year} Input Data",
    f"Input data for Hurricane {selected_storm.name} ({selected_storm.year})"
)

# Ingest a STAC item for each uploaded file (one per timestamp)
print(f"\n📥 Ingesting {len(uploaded_files)} STAC items (one per timestamp)...")
print(f"   Each item will have: surface data + atmospheric data assets")

ingested_items = []

for file_info in uploaded_files:
    ts = file_info['timestamp']
    sas_url = file_info['sas_url']
    ds_slice = file_info['ds_slice']
    idx = file_info['index']
    
    print(f"\n--- Item {idx+1}/{len(uploaded_files)} ---")
    print(f"   📅 Timestamp: {ts} (type: {type(ts).__name__})")
    
    # Ensure timestamp is a proper datetime object
    from datetime import datetime, timezone
    if hasattr(ts, 'tzinfo') and ts.tzinfo is None:
        # Add UTC timezone if missing
        ts = ts.replace(tzinfo=timezone.utc)
        print(f"   📅 Added UTC timezone: {ts}")
    
    # Create unique item ID with timestamp
    item_id = f"hurricane-{storm_name.lower()}-{ts.strftime('%Y%m%d')}-{ts.strftime('%H%M')}-{unique_suffix}-t{idx}"
    print(f"   🆔 Item ID: {item_id}")
    
    # Create STAC item for this timestamp (with surface data as primary asset)
    stac_item = await create_stac_item(
        item_id=item_id,
        date_time=ts,
        bbox=[-180, -90, 180, 90],  # Global extent
        data_url=sas_url,
        dataset=ds_slice  # Pass the 2D dataset slice
    )
    
    # Add atmospheric asset if available
    if uploaded_atmos_files and idx < len(uploaded_atmos_files):
        atmos_info = uploaded_atmos_files[idx]
        atmos_sas_url = atmos_info['sas_url']
        
        # Build atmospheric asset extra_fields
        atmos_extra_fields = {
            "type": "application/x-netcdf",
            "xarray:open_kwargs": {
                "engine": "h5netcdf"
            }
        }
        
        # Add atmospheric asset
        stac_item.add_asset(
            "atmospheric",
            pystac.Asset(
                href=atmos_sas_url,
                media_type="application/x-netcdf",
                title=f"Hurricane {storm_name} Atmospheric Data (NetCDF)",
                description=f"3D atmospheric variables on pressure levels ({len(atmos_info['levels'])} levels)",
                roles=["data"],
                extra_fields=atmos_extra_fields
            )
        )
        print(f"   🌡️ Added atmospheric asset with {len(atmos_info['levels'])} levels")
    
    # Ingest to GeoCatalog
    await ingest_to_geocatalog(stac_item, INPUT_COLLECTION)
    
    ingested_items.append({
        'item_id': item_id,
        'timestamp': ts,
        'sas_url': sas_url,
        'has_atmospheric': idx < len(uploaded_atmos_files) if uploaded_atmos_files else False
    })
    
    print(f"   ✅ Ingested: {item_id}")

print(f"\n✅ All {len(ingested_items)} STAC items ingested to GeoCatalog!")
print(f"  Collection: {INPUT_COLLECTION}")
for item in ingested_items:
    atmos_str = " (+ atmospheric)" if item.get('has_atmospheric') else ""
    print(f"  - {item['item_id']} ({item['timestamp']}){atmos_str}")

# Store reference to first dataset slice for render config
ds_surface = uploaded_files[0]['ds_slice'] if uploaded_files else None
# Store reference to first atmospheric dataset for render config
ds_atmos = uploaded_atmos_files[0]['ds_slice'] if uploaded_atmos_files else None
atmos_levels = uploaded_atmos_files[0]['levels'] if uploaded_atmos_files else []

#### 3.5.3 Configure Render Options via SDK

Use the Azure Planetary Computer Pro SDK to configure visualization settings:

**Surface Variables (2D)**
- Direct rendering with colormap and rescaling

**Atmospheric Variables (3D with pressure levels)**
- Uses `sel` query parameter to slice 3D data at specific pressure levels
- Key levels: 850 hPa (boundary layer), 500 hPa (steering flow), 200 hPa (jet stream)
- Variables: Temperature, Geopotential, U/V Wind, Humidity

In [ ]:
# ============================================================================
# Configure Render and Mosaic Options via Azure Planetary Computer SDK
# ============================================================================
# This uses the official SDK to configure visualization settings programmatically
# instead of manually through the portal UI.

from azure.planetarycomputer import PlanetaryComputerProClient
from azure.planetarycomputer.models import RenderOption, RenderOptionType, StacMosaic
from azure.identity import DefaultAzureCredential
import logging

# Reduce SDK logging noise
logging.getLogger("azure.core.pipeline.policies.http_logging_policy").setLevel(logging.WARNING)

# ============================================================================
# Initialize Planetary Computer Pro Client
# ============================================================================

print(f"📍 GeoCatalog URI: {GEOCATALOG_URI}")
print(f"📁 Collection ID: {INPUT_COLLECTION}")

# Create the client
credential = DefaultAzureCredential()
pc_client = PlanetaryComputerProClient(
    endpoint=GEOCATALOG_URI,
    credential=credential
)
print("✅ Planetary Computer Pro Client initialized")

# ============================================================================
# Define Render Configuration for SURFACE VARIABLES (2D)
# ============================================================================
# Surface variables only (2D - these will render correctly)
variable_config = {
    "msl": {"id": "msl-pressure", "name": "Mean Sea Level Pressure", "rescale": "95000,105000", "colormap": "rdylbu_r"},
    "t2m": {"id": "t2m-temp", "name": "2m Temperature", "rescale": "270,310", "colormap": "plasma"},
    "u10": {"id": "u10-wind", "name": "10m U Wind Component", "rescale": "-50,50", "colormap": "coolwarm"},
    "v10": {"id": "v10-wind", "name": "10m V Wind Component", "rescale": "-50,50", "colormap": "coolwarm"},
}

print(f"\n📊 Surface variables for visualization: {list(variable_config.keys())}")

# ============================================================================
# Define Render Configuration for ATMOSPHERIC VARIABLES (3D with level)
# These use 'sel' query parameter to select specific pressure levels
# ============================================================================

# Key pressure levels for hurricane analysis:
# - 850 hPa: Boundary layer (low-level winds, moisture)
# - 500 hPa: Mid-troposphere (steering flow, vorticity)
# - 200 hPa: Upper-troposphere (outflow, jet stream)

atmospheric_render_config = {
    # Temperature at key levels
    "t-850hpa": {"var": "t", "level": 850, "name": "Temperature at 850 hPa", "rescale": "260,300", "colormap": "plasma"},
    "t-500hpa": {"var": "t", "level": 500, "name": "Temperature at 500 hPa", "rescale": "230,270", "colormap": "plasma"},
    "t-200hpa": {"var": "t", "level": 200, "name": "Temperature at 200 hPa", "rescale": "200,240", "colormap": "plasma"},
    
    # Geopotential height at key levels
    "z-850hpa": {"var": "z", "level": 850, "name": "Geopotential at 850 hPa", "rescale": "10000,17000", "colormap": "viridis"},
    "z-500hpa": {"var": "z", "level": 500, "name": "Geopotential at 500 hPa", "rescale": "50000,60000", "colormap": "viridis"},
    
    # U-wind component
    "u-850hpa": {"var": "u", "level": 850, "name": "U Wind at 850 hPa", "rescale": "-50,50", "colormap": "coolwarm"},
    "u-200hpa": {"var": "u", "level": 200, "name": "U Wind at 200 hPa (Jet)", "rescale": "-80,80", "colormap": "coolwarm"},
    
    # V-wind component
    "v-850hpa": {"var": "v", "level": 850, "name": "V Wind at 850 hPa", "rescale": "-50,50", "colormap": "coolwarm"},
    "v-200hpa": {"var": "v", "level": 200, "name": "V Wind at 200 hPa (Jet)", "rescale": "-80,80", "colormap": "coolwarm"},
    
    # Specific humidity (moisture)
    "q-850hpa": {"var": "q", "level": 850, "name": "Humidity at 850 hPa", "rescale": "0,0.02", "colormap": "blues"},
}

print(f"🌡️ Atmospheric variables with level selection: {len(atmospheric_render_config)} render options")
if atmos_levels:
    print(f"   Available levels: {atmos_levels}")
else:
    print(f"   ⚠️ No atmospheric levels available")

# ============================================================================
# Function: Update Collection Tile Settings
# ============================================================================

async def update_collection_tile_settings(collection_id: str, min_zoom: int = 0):
    """Update the collection tile settings to allow lower zoom levels via API."""
    
    print("\n" + "="*70)
    print("⚙️ UPDATING COLLECTION TILE SETTINGS")
    print("="*70)
    
    print(f"   📝 Setting minZoom to: {min_zoom}")
    
    headers = get_bearer_token()
    headers["Content-Type"] = "application/json"
    params = {"api-version": API_VERSION}
    
    payload = {
        "tileSettings": {
            "minZoom": min_zoom,
            "maxItemsPerTile": 100
        }
    }
    
    # Try multiple endpoint patterns
    endpoints_to_try = [
        f"{GEOCATALOG_URI}/stac/collections/{collection_id}/configuration",
        f"{GEOCATALOG_URI}/collections/{collection_id}/configuration",
        f"{GEOCATALOG_URI}/stac/collections/{collection_id}",
    ]
    
    timeout = aiohttp.ClientTimeout(total=60)
    
    for config_url in endpoints_to_try:
        print(f"   📍 Trying: {config_url}")
        
        async with aiohttp.ClientSession(timeout=timeout) as session:
            # Try PATCH first
            async with session.patch(
                config_url,
                json=payload,
                headers=headers,
                params=params
            ) as response:
                if response.status < 400:
                    print(f"   ✅ Tile settings updated successfully (PATCH, status {response.status})")
                    return True
                    
            # Try PUT if PATCH failed
            async with session.put(
                config_url,
                json=payload,
                headers=headers,
                params=params
            ) as response:
                if response.status < 400:
                    print(f"   ✅ Tile settings updated successfully (PUT, status {response.status})")
                    return True
    
    # If all attempts failed, tile settings on render options should still work
    print(f"   ⚠️ Could not update collection-level tile settings")
    print(f"   Note: Individual render options will still have minZoom=0")
    return False

# ============================================================================
# Function: Configure Render Options via API
# ============================================================================

async def configure_render_options(client: PlanetaryComputerProClient, collection_id: str, dataset, atmos_dataset=None, levels=None):
    """Configure render options for surface AND atmospheric variables using the SDK.
    
    Surface variables (2D): Use standard rendering
    Atmospheric variables (3D with level): Use 'sel' parameter to slice to specific pressure levels
    
    IMPORTANT: This function first DELETES ALL existing render options, then creates
    new ones for both surface and atmospheric variables.
    """
    
    print("\n" + "="*70)
    print("🎨 CONFIGURING RENDER OPTIONS VIA API")
    print("="*70)
    
    # Get existing render options
    try:
        existing_renders = client.stac.list_render_options(collection_id=collection_id)
        existing_ids = [r.id for r in existing_renders] if existing_renders else []
        print(f"📋 Found {len(existing_ids)} existing render options")
    except Exception as e:
        print(f"⚠️ Could not list existing render options: {e}")
        existing_ids = []
    
    # STEP 1: Delete ALL existing render options to clean up stale configs
    if existing_ids:
        print(f"\n🗑️ Deleting ALL {len(existing_ids)} existing render options...")
        for render_id in existing_ids:
            try:
                client.stac.delete_render_option(
                    collection_id=collection_id,
                    render_option_id=render_id
                )
                print(f"   ✓ Deleted: {render_id}")
            except Exception as e:
                print(f"   ⚠️ Failed to delete {render_id}: {e}")
        print(f"   ✅ Cleanup complete")
    
    success_count = 0
    failed_count = 0
    
    # =========================================================================
    # STEP 2: Create SURFACE variable render options (2D)
    # =========================================================================
    print(f"\n➕ Creating {len(variable_config)} surface variable render options...")
    
    for var_name, config in variable_config.items():
        render_id = config.get("id", f"{var_name}-data")
        
        # Check if this variable exists in the dataset
        if dataset is None or var_name not in dataset.data_vars:
            print(f"   ⏭️ Skipping '{var_name}' - not in dataset")
            continue
        
        # Build the options string for surface variables (asset=data)
        options_str = f"assets=data&subdataset_name={var_name}&rescale={config['rescale']}&colormap_name={config['colormap']}"
        
        render_option = RenderOption(
            id=render_id,
            name=config["name"],
            type=RenderOptionType.RASTER_TILE,
            options=options_str,
            min_zoom=0
        )
        
        try:
            print(f"   ➕ Creating '{render_id}' for surface variable '{var_name}'...")
            client.stac.create_render_option(
                collection_id=collection_id,
                body=render_option
            )
            print(f"      ✅ Created")
            success_count += 1
        except Exception as e:
            print(f"   ❌ Failed to create render option '{render_id}': {e}")
            failed_count += 1
    
    # =========================================================================
    # STEP 3: Create ATMOSPHERIC variable render options (3D with sel parameter)
    # =========================================================================
    if atmos_dataset is not None and levels is not None and len(levels) > 0:
        print(f"\n🌡️ Creating atmospheric variable render options with level selection...")
        print(f"   Available levels: {levels}")
        
        # Get the level dimension name from the atmospheric dataset
        level_dim_name = None
        for dim_name in ['isobaricInhPa', 'level', 'pressure_level']:
            if dim_name in atmos_dataset.dims:
                level_dim_name = dim_name
                break
        
        if level_dim_name is None:
            print(f"   ⚠️ Could not find level dimension in atmospheric dataset")
        else:
            print(f"   Level dimension: '{level_dim_name}'")
            
            for render_id, config in atmospheric_render_config.items():
                var_name = config["var"]
                level = config["level"]
                
                # Check if variable exists and level is available
                if var_name not in atmos_dataset.data_vars:
                    print(f"   ⏭️ Skipping '{render_id}' - variable '{var_name}' not in dataset")
                    continue
                
                if level not in levels:
                    print(f"   ⏭️ Skipping '{render_id}' - level {level} not available")
                    continue
                
                # Build options string WITH sel parameter for level subsetting
                # The 'sel' parameter tells titiler to select a specific level from the 3D data
                # Format: sel={dim_name}={value} (uses equals sign, not colon!)
                options_str = (
                    f"assets=atmospheric"  # Use atmospheric asset
                    f"&subdataset_name={var_name}"
                    f"&sel={level_dim_name}={level}"  # KEY: Select specific pressure level (dim=value format)
                    f"&rescale={config['rescale']}"
                    f"&colormap_name={config['colormap']}"
                )
                
                render_option = RenderOption(
                    id=render_id,
                    name=config["name"],
                    type=RenderOptionType.RASTER_TILE,
                    options=options_str,
                    min_zoom=0
                )
                
                try:
                    print(f"   ➕ Creating '{render_id}' for {var_name} at {level} hPa...")
                    client.stac.create_render_option(
                        collection_id=collection_id,
                        body=render_option
                    )
                    print(f"      ✅ Created (sel={level_dim_name}={level})")
                    success_count += 1
                except Exception as e:
                    print(f"   ❌ Failed to create render option '{render_id}': {e}")
                    failed_count += 1
    else:
        print(f"\n⚠️ Skipping atmospheric render options - no atmospheric data available")
    
    print(f"\n📊 Render Configuration Summary:")
    print(f"   ✅ Created: {success_count}")
    print(f"   ❌ Failed: {failed_count}")
    print(f"   📋 Surface renders: {list(variable_config.keys())}")
    if atmos_dataset is not None:
        print(f"   📋 Atmospheric renders: {list(atmospheric_render_config.keys())}")
    
    return success_count > 0

# ============================================================================
# Function: Configure Mosaics via API
# ============================================================================

async def configure_mosaics(client: PlanetaryComputerProClient, collection_id: str):
    """Configure mosaic definitions for the collection using the SDK."""
    
    print("\n" + "="*70)
    print("🧩 CONFIGURING MOSAICS VIA API")
    print("="*70)
    
    # Define mosaics
    mosaics_to_create = [
        StacMosaic(
            id="most-recent",
            name="Most Recent Available",
            description="Most recently ingested data for the collection",
            cql=[]  # Empty CQL = all items, sorted by datetime desc
        )
    ]
    
    # Check existing mosaics
    try:
        existing_mosaics = client.stac.list_mosaics(collection_id=collection_id)
        existing_ids = [m.id for m in existing_mosaics] if existing_mosaics else []
        print(f"📋 Existing mosaics: {existing_ids}")
    except Exception as e:
        print(f"⚠️ Could not list existing mosaics: {e}")
        existing_ids = []
    
    success_count = 0
    
    for mosaic in mosaics_to_create:
        try:
            # Delete existing if present
            if mosaic.id in existing_ids:
                print(f"   🗑️ Deleting existing mosaic '{mosaic.id}'...")
                client.stac.delete_mosaic(
                    collection_id=collection_id,
                    mosaic_id=mosaic.id
                )
            
            # Create the mosaic
            print(f"   ➕ Creating mosaic '{mosaic.id}'...")
            client.stac.add_mosaic(
                collection_id=collection_id,
                body=mosaic
            )
            print(f"   ✅ Successfully created mosaic '{mosaic.id}'")
            success_count += 1
            
        except Exception as e:
            print(f"   ❌ Failed to create mosaic '{mosaic.id}': {e}")
    
    return success_count > 0

# ============================================================================
# Function: Get Current Collection Configuration
# ============================================================================

async def show_collection_configuration(client: PlanetaryComputerProClient, collection_id: str):
    """Display the current collection configuration."""
    
    print("\n" + "="*70)
    print("📋 CURRENT COLLECTION CONFIGURATION")
    print("="*70)
    
    try:
        config = client.stac.get_collection_configuration(collection_id=collection_id)
        print(f"\n{config}")
    except Exception as e:
        print(f"⚠️ Could not retrieve configuration: {e}")
    
    # List render options
    try:
        renders = client.stac.list_render_options(collection_id=collection_id)
        print(f"\n🎨 Render Options ({len(renders) if renders else 0}):")
        if renders:
            for r in renders:
                print(f"   - {r.id}: {r.name} ({r.type}) [minZoom: {r.min_zoom}]")
    except Exception as e:
        print(f"⚠️ Could not list render options: {e}")
    
    # List mosaics
    try:
        mosaics = client.stac.list_mosaics(collection_id=collection_id)
        print(f"\n🧩 Mosaics ({len(mosaics) if mosaics else 0}):")
        if mosaics:
            for m in mosaics:
                print(f"   - {m.id}: {m.name}")
    except Exception as e:
        print(f"⚠️ Could not list mosaics: {e}")

print("\n✅ API configuration functions defined")
print("\n⚡ Run the next cell to apply render and mosaic configuration via API")

In [ ]:
# Fallback: ensure URL variables are defined (empty string if earlier cells didn't run)
explorer_url = locals().get('explorer_url', '')
forecast_explorer_url = locals().get('forecast_explorer_url', '')
track_explorer_url = locals().get('track_explorer_url', '')
infra_explorer_url_safe = locals().get('infra_explorer_url_safe', locals().get('infra_explorer_url', ''))

# ============================================================================
# APPLY RENDER AND MOSAIC CONFIGURATION VIA API
# ============================================================================
# This cell executes the configuration - run after items are successfully ingested

print("="*70)
print("🚀 APPLYING RENDER & MOSAIC CONFIGURATION VIA API")
print("="*70)
print(f"\nCollection: {INPUT_COLLECTION}")
print(f"Endpoint: {GEOCATALOG_URI}")

# Show current configuration first
await show_collection_configuration(pc_client, INPUT_COLLECTION)

# Configure render options for BOTH surface AND atmospheric variables
# - Surface variables: standard 2D rendering (asset=data)
# - Atmospheric variables: use 'sel' parameter to slice 3D data to specific levels (asset=atmospheric)
# Note: minZoom=0 is set on each render option individually
render_success = await configure_render_options(
    pc_client, 
    INPUT_COLLECTION, 
    ds_surface,                     # Surface dataset (2D)
    atmos_dataset=ds_atmos,         # Atmospheric dataset (3D with level)
    levels=atmos_levels             # Available pressure levels
)

# Configure mosaics
mosaic_success = await configure_mosaics(pc_client, INPUT_COLLECTION)

# Show updated configuration
await show_collection_configuration(pc_client, INPUT_COLLECTION)

# Summary
print("\n" + "="*70)
print("📊 CONFIGURATION SUMMARY")
print("="*70)
print(f"✅ Render Options: {'Configured' if render_success else 'Failed'}")
print(f"✅ Mosaics: {'Configured' if mosaic_success else 'Failed'}")

# Open Explorer UI in browser to visualize data
import webbrowser
from urllib.parse import quote

# Build Explorer URL with default parameters
# Format: c=lon%2Clat (URL-encoded comma), z=zoom, v=view version, d=dataset, m=mosaic, r=render, s=settings, sr=sort, ae=animation
explorer_url = (
    f"{GEOCATALOG_URI}/explorer"
    f"?c=0.0000%2C0.0000"             # Center coordinates (lon,lat URL-encoded)
    f"&z=2.00"                        # Zoom level
    f"&v=2"                           # View version
    f"&d={INPUT_COLLECTION}"          # Dataset (collection)
    f"&m=default"                     # Mosaic
    f"&r=msl-pressure"                # Default render option (mean sea level pressure)
    f"&s=false%3A%3A100%3A%3Atrue"    # Settings (URL-encoded)
    f"&sr=desc"                       # Sort order
    f"&ae=0"                          # Animation enabled
)

print(f"\n🌐 Explorer URL: {explorer_url}")

# Wait for backend to propagate render configuration before opening browser
import time
print(f"\n⏳ Waiting 30 seconds for render configuration to propagate on backend...")
for remaining in range(30, 0, -10):
    print(f"   {remaining} seconds remaining...")
    time.sleep(10)
print(f"   Done waiting!")

print(f"\n📝 Opening GeoCatalog Explorer to visualize data...")
webbrowser.open(explorer_url)

## 4. Download ERA5 Static Variables

Download time-invariant geographic data from the Microsoft Planetary Computer ERA5 collection:

| Variable | Description |
|----------|-------------|
| **Geopotential** | Surface elevation reference |
| **Land-Sea Mask** | Ocean/land classification |
| **Soil Type** | Surface soil classification |

These static variables are required by Aurora for accurate atmospheric modeling and only need to be downloaded once.

In [ ]:
# Download ERA5 static variables from Microsoft Planetary Computer

import xarray as xr
import planetary_computer
from pystac_client import Client
from pathlib import Path
import requests

# Open Planetary Computer STAC catalog with authentication
catalog = Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

# Get ERA5 collection and access static variables asset
collection = catalog.get_collection("era5-pds")
static_asset = collection.assets["era5-static-var"]

print(f"📥 Loading ERA5 static variables from Planetary Computer...")

# Download to local file for netcdf4 compatibility
# Ensure downloads directory exists
downloads_dir = Path("downloads")
downloads_dir.mkdir(exist_ok=True)
static_file = downloads_dir / "era5_static_pc.nc"

if not static_file.exists():
    print(f"   Downloading from: {static_asset.href[:80]}...")
    response = requests.get(static_asset.href, stream=True)
    response.raise_for_status()
    with open(static_file, 'wb') as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
    print(f"   ✓ Downloaded to {static_file}")
else:
    print(f"   ✓ Using cached file: {static_file}")

# Open with netcdf4 engine
ds_static = xr.open_dataset(static_file, engine="netcdf4")

print(f"\n✅ ERA5 static variables loaded from Planetary Computer")
print(f"   Variables: {list(ds_static.data_vars)}")
print(f"   Dimensions: {dict(ds_static.dims)}")
ds_static

## 5. Prepare Aurora Batch Object

Aurora model input consists of four components:

| Component | Variables | Shape |
|-----------|-----------|-------|
| **Surface** | 2m temp, 10m winds, MSL pressure | [batch, 2, lat, lon] |
| **Atmospheric** | Temperature, winds, humidity, geopotential (13 pressure levels) | [batch, 2, level, lat, lon] |
| **Static** | Geopotential, land-sea mask, soil type | [lat, lon] |
| **Metadata** | Coordinates, pressure levels, timestamp | - |

All variables are provided unnormalized; the model handles normalization internally.

### 8.2 Build Unified Impact Zone from Swath & Coastal Tiers

Combine the outer swath polygon and the flood‐tier coastal polygon from Section 7.1b into a single **impact zone** used for all infrastructure queries.  This replaces the old cone‐of‐uncertainty approach.

In [ ]:
# Surface Variables

import numpy as np
import torch

# Extract surface variables from the combined ECMWF dataset
# Aurora needs: 2t (2m temp), 10u (10m u-wind), 10v (10m v-wind), msl (mean sea level pressure)
# Aurora requires shape [batch, 2, lat, lon] - we now have 2 timesteps (0h and 6h)

# Helper function to extract and clean data
# ERA5 CDS data may have extra dimensions (e.g., 'expver') that need to be squeezed
def extract_surface_var(ds, var_name):
    """Extract surface variable and ensure correct shape [lat, lon] per timestep."""
    data = ds[var_name].to_numpy()
    # Squeeze out any singleton dimensions except batch and time
    # Expected input: (time, ..., lat, lon) with possible extra dims
    # Expected output: (time, lat, lon)
    while data.ndim > 3:  # More than [time, lat, lon]
        # Find and squeeze singleton dimensions (but not time which is dim 0)
        for i in range(1, data.ndim - 2):  # Skip time (0) and lat/lon (-2, -1)
            if data.shape[i] == 1:
                data = np.squeeze(data, axis=i)
                break
        else:
            break  # No more singleton dims to squeeze
    return data

surf_vars = {
    "2t": torch.from_numpy(extract_surface_var(ds, "t2m")[None]),      # 2m temperature [1, 2, lat, lon]
    "10u": torch.from_numpy(extract_surface_var(ds, "u10")[None]),     # 10m U wind component [1, 2, lat, lon]
    "10v": torch.from_numpy(extract_surface_var(ds, "v10")[None]),     # 10m V wind component [1, 2, lat, lon]
    "msl": torch.from_numpy(extract_surface_var(ds, "msl")[None]),     # Mean sea level pressure [1, 2, lat, lon]
}

print("✅ Extracted surface variables")
print(f"   2t shape: {surf_vars['2t'].shape}")
print(f"   10u shape: {surf_vars['10u'].shape}")
print(f"   10v shape: {surf_vars['10v'].shape}")
print(f"   msl shape: {surf_vars['msl'].shape}")

### 5.2 Prepare Atmospheric Variables

Extract temperature, wind components, specific humidity, and geopotential at 13 pressure levels (50-1000 hPa).

In [ ]:
# Atmospheric Variables

# Extract atmospheric variables from ECMWF dataset at pressure levels
# Aurora needs: t (temperature), u (u-wind), v (v-wind), q (specific humidity), z (geopotential)
# Note: ECMWF provides gh (geopotential height in m), convert to z (geopotential in m²/s²) by multiplying by g=9.80665
# Aurora requires shape [batch, 2, level, lat, lon] - we now have 2 timesteps (0h and 6h)

# Helper function to extract and clean atmospheric data
# ERA5 CDS data may have extra dimensions (e.g., 'expver') that need to be squeezed
def extract_atmos_var(ds, var_name, scale_factor=1.0):
    """Extract atmospheric variable and ensure correct shape [time, level, lat, lon]."""
    data = ds[var_name].to_numpy() * scale_factor
    # Expected final shape: (time, level, lat, lon) = (2, 13, 721, 1440)
    # ERA5 CDS might have (time, expver, level, lat, lon) or other extra dims
    
    # Find dimensions: time should be 2, level should be 13
    target_ndim = 4  # [time, level, lat, lon]
    
    while data.ndim > target_ndim:
        # Look for singleton dimensions to squeeze (skip time at 0)
        squeezed = False
        for i in range(1, data.ndim - 2):  # Skip time (0) and lat/lon (-2, -1)
            if data.shape[i] == 1:
                data = np.squeeze(data, axis=i)
                squeezed = True
                break
        if not squeezed:
            break  # No more singleton dims to squeeze
    
    return data

atmos_vars = {
    "t": torch.from_numpy(extract_atmos_var(ds, "t")[None]),        # Temperature [1, 2, level, lat, lon]
    "u": torch.from_numpy(extract_atmos_var(ds, "u")[None]),        # U wind component [1, 2, level, lat, lon]
    "v": torch.from_numpy(extract_atmos_var(ds, "v")[None]),        # V wind component [1, 2, level, lat, lon]
    "q": torch.from_numpy(extract_atmos_var(ds, "q")[None]),        # Specific humidity [1, 2, level, lat, lon]
    "z": torch.from_numpy(extract_atmos_var(ds, "gh", scale_factor=9.80665)[None]),  # Geopotential [1, 2, level, lat, lon]
}

print("✅ Extracted atmospheric variables")
print(f"   t shape: {atmos_vars['t'].shape}")
print(f"   u shape: {atmos_vars['u'].shape}")
print(f"   v shape: {atmos_vars['v'].shape}")
print(f"   q shape: {atmos_vars['q'].shape}")
print(f"   z shape: {atmos_vars['z'].shape}")

### 5.3 Prepare Static Variables

Extract geopotential, land-sea mask, and soil type from ERA5 data, converting longitude to match Aurora's [0°, 360°] convention.

In [ ]:
# Static Variables

# Extract static variables from ds_static (Planetary Computer ERA5 collection)
# IMPORTANT: Keep static data in the SAME coordinate system as atmospheric data!
# The batch creation cell handles the [-180,180] -> [0,360] conversion for ALL variables
# together using sort_idx. If we pre-convert static data, the sorting will be wrong.

import numpy as np
import xarray as xr

# Get coordinate names for static dataset
static_lat_coord = 'latitude' if 'latitude' in ds_static.coords else 'lat'
static_lon_coord = 'longitude' if 'longitude' in ds_static.coords else 'lon'
static_lon = ds_static[static_lon_coord].values
static_lat = ds_static[static_lat_coord].values

print(f"📊 Static data (Planetary Computer ERA5):")
print(f"   Latitude: {len(static_lat)} points ({static_lat.min():.1f} to {static_lat.max():.1f})")
print(f"   Longitude: {len(static_lon)} points ({static_lon.min():.1f} to {static_lon.max():.1f})")

# Get target grid from atmospheric data (keep in ORIGINAL coordinate system)
target_lat = ds.latitude.values
target_lon = ds.longitude.values  # This is in [-180, 180] from ECMWF

print(f"\n📊 Atmospheric data grid (target):")
print(f"   Latitude: {len(target_lat)} points ({target_lat.min():.1f} to {target_lat.max():.1f})")
print(f"   Longitude: {len(target_lon)} points ({target_lon.min():.1f} to {target_lon.max():.1f})")

# Check if interpolation is needed (grid size mismatch)
need_interpolation = (
    len(static_lat) != len(target_lat) or
    len(static_lon) != len(target_lon)
)

if need_interpolation:
    print(f"\n🔄 Interpolating static variables to match atmospheric grid...")
    print(f"   Source: {len(static_lat)} x {len(static_lon)}")
    print(f"   Target: {len(target_lat)} x {len(target_lon)}")
    
    # Both datasets should be in [-180, 180] - interpolate directly
    # The Planetary Computer ERA5 static data is in [-180, 180] format
    interp_coords = {
        static_lat_coord: target_lat,
        static_lon_coord: target_lon
    }
    
    # Interpolate static dataset to match atmospheric grid
    ds_static_interp = ds_static.interp(
        interp_coords,
        method='linear',
        kwargs={'fill_value': 'extrapolate'}
    )
    
    print(f"   ✓ Interpolated to: {len(ds_static_interp[static_lat_coord])} x {len(ds_static_interp[static_lon_coord])}")
else:
    print(f"\n✓ Static and atmospheric grids already match - no interpolation needed")
    ds_static_interp = ds_static

# Extract the data arrays (keep in original coordinate system)
z_data = ds_static_interp["z"].squeeze().values
lsm_data = ds_static_interp["lsm"].squeeze().values
slt_data = ds_static_interp["slt"].squeeze().values

# Verify alignment
print(f"\n📊 Final static variable shapes:")
print(f"   z: {z_data.shape}")
print(f"   lsm: {lsm_data.shape}")
print(f"   slt: {slt_data.shape}")

# Expected shape should match [n_lat, n_lon] from atmospheric data
expected_shape = (len(target_lat), len(target_lon))
if z_data.shape == expected_shape:
    print(f"\n✓ Static variable shapes match atmospheric grid: {expected_shape}")
else:
    print(f"\n⚠️ Shape mismatch! Expected {expected_shape}, got {z_data.shape}")

# NOTE: Static vars are in [-180, 180] longitude format here.
# The batch creation cell will convert ALL variables to [0, 360] together
# using the same sort_idx, ensuring consistent alignment.
static_vars = {
    "z": torch.from_numpy(z_data.astype(np.float32)),
    "lsm": torch.from_numpy(lsm_data.astype(np.float32)),
    "slt": torch.from_numpy(slt_data.astype(np.float32)),
}

print(f"\n✅ Static vars ready (will be converted to [0,360] in batch creation)")
print(f"   {{{', '.join(f'{k}: {v.shape}' for k, v in static_vars.items())}}}")

### 5.4 Create Aurora Batch

Combine all variables into an Aurora `Batch` object with proper metadata (coordinates, pressure levels, timestamp).

In [ ]:
# Create Batch 

from aurora import Batch, Metadata
from datetime import datetime

# Debug: Print the actual dimensions of the atmospheric variables
print("📊 Checking atmospheric variable dimensions...")
for name, tensor in atmos_vars.items():
    print(f"   {name}: {tensor.shape} (ndim={tensor.ndim})")

# Get metadata from the ECMWF dataset
# Convert to torch tensors with proper dtype
lat = torch.tensor(ds.latitude.values, dtype=torch.float32)
lon = torch.tensor(ds.longitude.values, dtype=torch.float32)

# Convert longitudes from [-180, 180] to [0, 360) for Aurora
lon = torch.where(lon < 0, lon + 360, lon)

# Sort longitudes to ensure strictly increasing order
# This is needed because converting [-180,180] to [0,360) breaks monotonicity
sort_idx = torch.argsort(lon)
lon = lon[sort_idx]

# Reorder all data arrays along the longitude dimension
# Surface vars: [batch, time, lat, lon]
surf_vars_sorted = {k: v[..., sort_idx] for k, v in surf_vars.items()}

# Atmos vars: [batch, time, level, lat, lon]
atmos_vars_sorted = {k: v[..., sort_idx] for k, v in atmos_vars.items()}

# Static vars: [lat, lon]
static_vars_sorted = {k: v[..., sort_idx] for k, v in static_vars.items()}

# Extract and sort pressure levels (Aurora expects ascending order: 50, 100, ... 1000)
# Handle different coordinate names
if 'isobaricInhPa' in ds.coords:
    level_coord = ds.isobaricInhPa.values
elif 'pressure_level' in ds.coords:
    level_coord = ds.pressure_level.values
elif 'level' in ds.coords:
    level_coord = ds.level.values
else:
    raise ValueError(f"Cannot find pressure level coordinate. Available coords: {list(ds.coords)}")

original_levels = tuple(int(p) for p in level_coord)
pressure_levels = tuple(sorted(original_levels))

# Get the indices to reorder from original to sorted order
level_sort_idx = [original_levels.index(p) for p in pressure_levels]
level_sort_idx_tensor = torch.tensor(level_sort_idx)

print(f"\nSorted pressure levels: {pressure_levels}")
print(f"Original levels: {original_levels}")
print(f"Reorder indices: {level_sort_idx}")

# Reorder atmospheric variables along the level dimension
# Dynamically determine which dimension is the level dimension based on tensor shape
# Expected final shape: [batch, time, level, lat, lon] = [1, 2, 13, 721, 1440] for global 0.25° data
for k, v in atmos_vars_sorted.items():
    print(f"\n   Processing {k}: shape={v.shape}")
    
    # Determine level dimension index by finding dimension with size matching number of levels
    num_levels = len(original_levels)
    level_dim = None
    for dim_idx, dim_size in enumerate(v.shape):
        if dim_size == num_levels:
            level_dim = dim_idx
            break
    
    if level_dim is not None:
        print(f"      Found level dimension at index {level_dim} (size={num_levels})")
        # Use index_select to reorder along the correct dimension
        atmos_vars_sorted[k] = torch.index_select(v, level_dim, level_sort_idx_tensor)
        print(f"      New shape: {atmos_vars_sorted[k].shape}")
    else:
        print(f"      ⚠️ Could not find level dimension with size {num_levels}")

# Time - reference time (SECOND timestep per Aurora convention)
# Aurora assigns "issue time" to the second of the two input times
time_val = ds.time.values[1]
time_dt = datetime.utcfromtimestamp(time_val.astype("datetime64[s]").astype("int"))

# Create metadata
metadata = Metadata(
    lat=lat,
    lon=lon,
    time=(time_dt,),
    atmos_levels=pressure_levels
)

# Create the Aurora Batch object
batch = Batch(
    surf_vars=surf_vars_sorted,
    static_vars=static_vars_sorted,
    atmos_vars=atmos_vars_sorted,
    metadata=metadata
)

print("\n✅ Aurora Batch created")
print(f"   Lat range: {lat.min():.1f} to {lat.max():.1f}")
print(f"   Lon range: {lon.min():.1f} to {lon.max():.1f}")
print(f"   Pressure levels: {pressure_levels}")
print(f"   Time: {time_dt}")

### 5.5 Verify Data Dimensions

Validate that all tensors have the correct shapes and the grid dimensions are compatible with Aurora's patch-based processing.

In [ ]:
# Check final batch data dimensions for Aurora inference
print("=" * 60)
print("AURORA BATCH DATA DIMENSION CHECK")
print("=" * 60)

# Check surface variables - expected shape: [batch, 2, lat, lon]
print("\n📊 Surface Variables (expected: [batch, 2, lat, lon]):")
for name, tensor in batch.surf_vars.items():
    print(f"   {name}: {tensor.shape}")

# Check atmospheric variables - expected shape: [batch, 2, level, lat, lon]
print("\n📊 Atmospheric Variables (expected: [batch, 2, level, lat, lon]):")
for name, tensor in batch.atmos_vars.items():
    print(f"   {name}: {tensor.shape}")

# Check static variables - expected shape: [lat, lon]
print("\n📊 Static Variables (expected: [lat, lon]):")
for name, tensor in batch.static_vars.items():
    print(f"   {name}: {tensor.shape}")

# Check metadata
print("\n📊 Metadata:")
print(f"   Latitude points: {len(batch.metadata.lat)}")
print(f"   Longitude points: {len(batch.metadata.lon)}")
print(f"   Lat range: {batch.metadata.lat.min():.2f} to {batch.metadata.lat.max():.2f}")
print(f"   Lon range: {batch.metadata.lon.min():.2f} to {batch.metadata.lon.max():.2f}")
print(f"   Pressure levels: {batch.metadata.atmos_levels}")
print(f"   Time: {batch.metadata.time}")

# Aurora compatibility check - dimensions should be divisible by patch size
PATCH_SIZE = 4
n_lat = len(batch.metadata.lat)
n_lon = len(batch.metadata.lon)

lat_ok = n_lat % PATCH_SIZE == 0 or (n_lat - 1) % PATCH_SIZE == 0 # aurora can handle 1 extra lat/long
lon_ok = n_lon % PATCH_SIZE == 0 or (n_lon - 1) % PATCH_SIZE == 0 # aurora can handle 1 extra lat/long

print(f"\n📊 Aurora Compatibility Check (patch size = {PATCH_SIZE}):")
print(f"   Latitude or Latitude-1 divisible by {PATCH_SIZE}: {'✓' if lat_ok else '✗'} ({n_lat} / {PATCH_SIZE} = {n_lat / PATCH_SIZE:.2f})")
print(f"   Longitude or Longitude-1 divisible by {PATCH_SIZE}: {'✓' if lon_ok else '✗'} ({n_lon} / {PATCH_SIZE} = {n_lon / PATCH_SIZE:.2f})")

if lat_ok and lon_ok:
    print("\n✓ Batch dimensions are compatible with Aurora!")
else:
    print("\n⚠️  Warning: Dimensions may need adjustment for Aurora inference")

print("\n" + "=" * 60)

## 6. Run Aurora Model Inference

Submit the prepared batch to Microsoft Foundry for inference using the Aurora 0.25° fine-tuned model. The model generates 6-hourly forecasts for the configured duration, with the built-in **Tropical Cyclone Tracker** extracting storm center positions from each prediction.

### 6.1 Run inference and get the predictions
- **Predicted Track**: Latitude, longitude, timestamp for each forecast step
- **Intensity Metrics**: Mean sea level pressure and 10m wind speed at storm center

In [ ]:
# Aurora imports
from aurora import Batch, Metadata
from aurora import Tracker
from aurora.foundry import BlobStorageChannel, FoundryClient, submit

# =============================================================================
# CALCULATE FORECAST DURATION
# =============================================================================
# The forecast duration is calculated based on storm_init_time and storm_end_time
# which were set by find_optimal_aurora_init in Cell 8:
#
# HISTORICAL STORMS:
#   - storm_init_time: Optimal T0 early in track (strong storm, synoptic hour)
#   - storm_end_time: Last track observation
#   - Forecast from T0 to track end
#
# ACTIVE STORMS:
#   - storm_init_time: T0 based on data availability
#   - storm_end_time: T0 + default forecast duration (5 days)
#   - Early termination via pressure threshold may stop sooner
# =============================================================================

MIN_FORECAST_STEPS = 4  # Minimum 24 hours (4 x 6-hour steps)
DEFAULT_FORECAST_HOURS = 120  # 5 days default for active storms

# Calculate forecast duration from storm_init_time to storm_end_time
# Normalize both to tz-naive to handle mixed tz-aware/tz-naive (current vs historical storms)
_init_naive = storm_init_time.tz_localize(None) if storm_init_time.tzinfo is not None else storm_init_time
_end_naive = storm_end_time.tz_localize(None) if storm_end_time.tzinfo is not None else storm_end_time
time_diff = _end_naive - _init_naive
total_hours = time_diff.total_seconds() / 3600

# Ensure minimum forecast duration
# For active storms: use 5-day default if track is too short
# For historical storms: trust the track duration (forecast through actual track end)
if total_hours < 24:
    if is_active_storm:
        print(f"⚠️ Calculated forecast duration too short ({total_hours:.1f}h)")
        print(f"   storm_init_time: {storm_init_time}")
        print(f"   storm_end_time: {storm_end_time}")
        total_hours = DEFAULT_FORECAST_HOURS
        print(f"   Using default: {DEFAULT_FORECAST_HOURS}h (active storm)")
    else:
        # Historical: use actual track duration, minimum 1 step (6h)
        total_hours = max(6, total_hours)
        print(f"📜 Short historical track ({total_hours:.1f}h) - forecasting through track end")
        print(f"   storm_init_time: {storm_init_time}")
        print(f"   storm_end_time: {storm_end_time}")

# For active storms: enforce minimum 24h (4 steps)
# For historical storms: use exact track duration (minimum 1 step)
if is_active_storm:
    NUM_SIX_HOUR_STEPS = max(MIN_FORECAST_STEPS, int(total_hours / 6))
else:
    NUM_SIX_HOUR_STEPS = max(1, int(total_hours / 6))

# Calculate actual end time for display
forecast_end_time = storm_init_time + pd.Timedelta(hours=NUM_SIX_HOUR_STEPS * 6)

START_DATE = storm_init_time.strftime("%Y-%m-%d")
START_TIME = storm_init_time.strftime("%H:%M")
END_DATE = forecast_end_time.strftime("%Y-%m-%d")
END_TIME = forecast_end_time.strftime("%H:%M")

print(f"🌀 Hurricane Forecast Period:")
print(f"   Start (T0): {START_DATE} {START_TIME}")
print(f"   End:        {END_DATE} {END_TIME}")
print(f"   Storm type: {'🔴 ACTIVE' if is_active_storm else '📜 HISTORICAL'}")

print(f"\n📊 Forecast Duration:")
print(f"   storm_init_time (T0):   {storm_init_time}")
print(f"   storm_end_time:         {storm_end_time}")
print(f"   Total forecast hours:   {NUM_SIX_HOUR_STEPS * 6}")
print(f"   Number of 6-hour steps: {NUM_SIX_HOUR_STEPS}")

if is_active_storm:
    print(f"\n   💡 Active storm: Smart termination enabled (MSLP threshold)")

# Aurora configuration
MODEL_NAME = "aurora-0.25-finetuned"

# Initialize Aurora Foundry clients
foundry_client = FoundryClient(endpoint=AURORA_FOUNDRY_ENDPOINT, token=AURORA_FOUNDRY_TOKEN)
channel = BlobStorageChannel(AURORA_BLOB_STORAGE_SAS)

print(f"\n✓ Aurora Foundry client initialized: {AURORA_FOUNDRY_ENDPOINT}")
print(f"✓ Blob storage channel configured")

# Convert longitude from [-180, 180] to [0, 360) format for Aurora
# Aurora uses 0-360 convention, but tropycal uses -180 to 180
init_lon_360 = storm_init_lon + 360 if storm_init_lon < 0 else storm_init_lon
print(f"\n📍 Tracker initialization:")
print(f"   Original init_lon: {storm_init_lon} (tropycal convention)")
print(f"   Converted init_lon: {init_lon_360} (Aurora 0-360 convention)")

# Initialize the Tracker locally
# Match the init_time to your initial_condition time
tracker = Tracker(
    init_lat=storm_init_lat, 
    init_lon=init_lon_360,  # Use converted longitude!
    init_time=storm_init_time
)

# Run Aurora inference with TC tracking
print(f"\n🌀 Starting Aurora inference for Hurricane {storm_name}...")
print(f"   Model: {MODEL_NAME}")
print(f"   Forecast steps: {NUM_SIX_HOUR_STEPS} (6-hour increments)")
print(f"   Total forecast period: {NUM_SIX_HOUR_STEPS * 6} hours")

# We need to capture the task_id to know where predictions are stored
# The submit() generator doesn't expose task_id, so we'll wrap it and extract from logs
import logging

# Set up a custom log handler to capture the task_id
class TaskIdCapture(logging.Handler):
    def __init__(self):
        super().__init__()
        self.task_id = None
    
    def emit(self, record):
        # Look for the log message that contains the task_id
        if "Created task" in record.getMessage() and "at endpoint" in record.getMessage():
            # Extract task_id from message like: "Created task `uuid` at endpoint."
            msg = record.getMessage()
            start = msg.find('`') + 1
            end = msg.find('`', start)
            if start > 0 and end > start:
                self.task_id = msg[start:end]

# Add custom handler to capture task_id
task_id_capture = TaskIdCapture()
aurora_logger = logging.getLogger("aurora.foundry.client.api")
original_level = aurora_logger.level
aurora_logger.setLevel(logging.INFO)
aurora_logger.addHandler(task_id_capture)

preds = []
step_count = 0

# =============================================================================
# INTELLIGENT FORECAST TERMINATION FOR ACTIVE/CURRENT STORMS
# =============================================================================
# For active storms, we apply early termination based on pressure metrics:
#   - Pressure: If MSLP rises above PRESSURE_THRESHOLD_MB for 4 consecutive steps,
#     the storm has lost tropical characteristics
#
# NOTE: Wind speed thresholds are NOT used because Aurora's wind predictions
# are less reliable than pressure predictions for tropical cyclone intensity.
#
# WHY ONLY FOR CURRENT STORMS?
# - Historical storms have complete track data; we forecast to match the known endpoint
# - Current/active storms have no known endpoint; without termination logic, we'd
#   forecast for the full climatological maximum (5-7 days) even if the storm
#   dissipates earlier (e.g., after landfall or over cold water)
# - Example: Tropical Cyclone Luana (2026) lasted ~2.5 days but a fixed 5-day
#   forecast would extend predictions well beyond the storm's actual lifespan
# =============================================================================

# Thresholds for early termination (only applied to active storms)
# Using only pressure threshold since Aurora's pressure predictions are more reliable than wind
PRESSURE_THRESHOLD_MB = 1008.0  # Above this = storm losing organization
CONSECUTIVE_WEAK_STEPS_REQUIRED = 2  # Require 2 consecutive steps above pressure threshold to terminate

# Track consecutive weak pressure steps
consecutive_weak_pressure_steps = 0
early_termination_reason = None

# Check if is_active_storm is defined (set by confirm_selection or auto-select)
if 'is_active_storm' not in dir():
    is_active_storm = False  # Default to False for historical storms

# =============================================================================
# MINIMUM FORECAST DURATION CALCULATION
# =============================================================================
# Since storm prediction init can start back in time (T0 shifted based on data
# availability), we ensure a minimum forecast duration of current_time + 1 day
# This prevents early termination before reaching meaningful forecast coverage
# =============================================================================
import pandas as pd
now_utc = pd.Timestamp.now(tz='UTC')
minimum_forecast_end = now_utc + pd.Timedelta(days=1)
minimum_steps_required = 0  # Will be calculated for active storms

if is_active_storm:
    # Calculate how many 6-hour steps needed to reach minimum_forecast_end
    # Normalize to tz-naive to handle mixed tz-aware/tz-naive timestamps
    _min_end_naive = minimum_forecast_end.tz_localize(None) if minimum_forecast_end.tzinfo is not None else minimum_forecast_end
    _init_naive_min = storm_init_time.tz_localize(None) if storm_init_time.tzinfo is not None else storm_init_time
    hours_to_minimum_end = (_min_end_naive - _init_naive_min).total_seconds() / 3600
    minimum_steps_required = max(0, int(hours_to_minimum_end / 6))
    
    print(f"\n🔴 ACTIVE STORM DETECTED - Intelligent forecast termination ENABLED")
    print(f"   Current time (UTC): {now_utc.strftime('%Y-%m-%d %H:%M')}")
    print(f"   Storm init time: {storm_init_time.strftime('%Y-%m-%d %H:%M')}")
    print(f"   Minimum forecast end: {minimum_forecast_end.strftime('%Y-%m-%d %H:%M')} (current + 1 day)")
    print(f"   Minimum steps required before termination check: {minimum_steps_required}")
    print(f"   Pressure threshold: {PRESSURE_THRESHOLD_MB} mb (require {CONSECUTIVE_WEAK_STEPS_REQUIRED} consecutive steps)")
    print(f"   Note: Wind threshold disabled (Aurora pressure more reliable than wind)")
else:
    print(f"\n📜 HISTORICAL STORM - Using full forecast duration (no early termination)")

try:
    for pred in submit(
        batch,
        model_name=MODEL_NAME,
        num_steps=NUM_SIX_HOUR_STEPS,
        foundry_client=foundry_client,
        channel=channel,
    ):
        preds.append(pred)
        step_count += 1
        
        # Pass each predicted batch from Foundry into the local tracker
        tracker.step(pred)
        
        # Get current intensity from tracker results
        current_track = tracker.results()
        if len(current_track) > 0:
            latest_wind = current_track.iloc[-1]['wind']  # 10m wind in knots
            latest_mslp = current_track.iloc[-1]['msl'] / 100  # Convert Pa to mb
        else:
            latest_wind = None
            latest_mslp = None

        # Log progress every 5 steps
        if step_count % 5 == 0:
            forecast_time = pred.metadata.time[0]
            wind_str = f"{latest_wind:.1f} kt" if latest_wind else "N/A"
            mslp_str = f"{latest_mslp:.1f} mb" if latest_mslp else "N/A"
            print(f"   ✓ Step {step_count}/{NUM_SIX_HOUR_STEPS}: {forecast_time} | Wind: {wind_str} | MSLP: {mslp_str}")
        
        # =============================================================================
        # EARLY TERMINATION CHECK (ACTIVE STORMS ONLY)
        # Using only pressure threshold - Aurora's pressure predictions are more 
        # reliable than wind speed predictions for tropical cyclone intensity
        # 
        # MINIMUM DURATION: Must run until current_time + 2 days before allowing
        # early termination, since T0 may be shifted back in time
        # =============================================================================
        if is_active_storm and latest_mslp is not None:
            # Only allow early termination after reaching minimum forecast duration
            if step_count < minimum_steps_required:
                # Still within minimum forecast window - no termination allowed
                if step_count % 10 == 0:
                    print(f"   📊 Step {step_count}: Within minimum forecast window ({step_count}/{minimum_steps_required} steps)")
            else:
                # Past minimum duration - check pressure threshold (requires 2 consecutive steps above threshold)
                if latest_mslp > PRESSURE_THRESHOLD_MB:
                    consecutive_weak_pressure_steps += 1
                    if consecutive_weak_pressure_steps >= CONSECUTIVE_WEAK_STEPS_REQUIRED:
                        early_termination_reason = f"MSLP ({latest_mslp:.1f} mb) exceeded {PRESSURE_THRESHOLD_MB} mb for {consecutive_weak_pressure_steps} consecutive steps"
                        print(f"\n   ⚠️ EARLY TERMINATION: {early_termination_reason}")
                        print(f"   Storm has likely lost tropical characteristics.")
                        break
                else:
                    # Reset counter if pressure drops back below threshold
                    consecutive_weak_pressure_steps = 0
                
finally:
    # Clean up logging
    aurora_logger.removeHandler(task_id_capture)
    aurora_logger.setLevel(original_level)

# Get the captured task_id
inference_task_id = task_id_capture.task_id

# Get track results
track_df = tracker.results()

# Convert longitude back to -180/180 for easier comparison with ground truth
if len(track_df) > 0:
    track_df['lon_converted'] = track_df['lon'].apply(lambda x: x - 360 if x > 180 else x)

print(f"\n✓ Aurora inference completed!")
print(f"  - Task ID: {inference_task_id}")
print(f"  - Total predictions: {len(preds)}")
print(f"  - Actual forecast steps: {step_count}/{NUM_SIX_HOUR_STEPS}")
if early_termination_reason:
    print(f"  - ⚠️ Early termination: {early_termination_reason}")

if len(preds) > 0:
    print(f"  - Final forecast time: {preds[-1].metadata.time[0]}")
    print(f"\nTropical Cyclone Track DataFrame:")
    print(track_df[['time', 'lat', 'lon', 'lon_converted', 'msl', 'wind']])
else:
    print(f"  - ⚠️ No predictions returned! Check the error logs above.")
    print(f"  - Common issues: grid dimension mismatch, invalid batch data")

print(f"\n" + "="*60)

### 6.2 Upload Results to Azure GeoCatalog

Save the predictions and output for future analysis to Microsoft Planetary Computer Pro

In [ ]:
# ============================================================================
# UPLOAD AURORA FORECAST PREDICTIONS TO PLANETARY COMPUTER PRO
# ============================================================================
# Convert Aurora predictions (Batch objects) to xarray, save as NetCDF,
# upload to blob storage, create STAC collection, and configure visualization.

import xarray as xr
import numpy as np
from pathlib import Path
from datetime import datetime, timezone
import aiohttp
import pystac

print("="*80)
print("📤 UPLOADING AURORA FORECAST PREDICTIONS TO PLANETARY COMPUTER PRO")
print("="*80)

# ============================================================================
# Define Output Collection
# ============================================================================
OUTPUT_COLLECTION = f"hurricane-{storm_name.lower()}-{storm_year}-Forecast-Data"
print(f"\n📁 Output Collection: {OUTPUT_COLLECTION}")
print(f"   Storm: Hurricane {storm_name} ({storm_year})")
print(f"   Total predictions: {len(preds)}")

# ============================================================================
# Helper Function: Convert Aurora Batch to xarray Datasets
# ============================================================================
def batch_to_xarray(pred_batch, include_atmos=True):
    """Convert Aurora Batch prediction to xarray Datasets (surface and atmospheric).
    
    Args:
        pred_batch: Aurora Batch object with predictions
        include_atmos: Whether to include atmospheric variables (default True)
        
    Returns:
        Tuple of (surface_ds, atmos_ds) - atmos_ds may be None if include_atmos=False
        
    Note:
        Aurora uses 0-360 longitude convention, but MPC expects -180 to 180.
        This function converts coordinates and rolls data accordingly.
    """
    metadata = pred_batch.metadata
    
    # Get coordinates
    lat = metadata.lat.numpy() if hasattr(metadata.lat, 'numpy') else np.array(metadata.lat)
    lon_original = metadata.lon.numpy() if hasattr(metadata.lon, 'numpy') else np.array(metadata.lon)
    time_val = metadata.time[0]  # Get the forecast time
    
    # =========================================================================
    # Convert longitude from 0-360 to -180 to 180 for MPC compatibility
    # =========================================================================
    # Check if longitude is in 0-360 range
    if lon_original.max() > 180:
        print(f"   🔄 Converting longitude from 0-360 to -180/180 range")
        # Convert: values > 180 become negative (e.g., 270 -> -90)
        lon = np.where(lon_original > 180, lon_original - 360, lon_original)
        
        # Find the split point where we need to roll the data
        # This is the index where longitude values switch from positive to negative
        split_idx = np.searchsorted(lon_original, 180)
        n_to_roll = len(lon_original) - split_idx
        
        # Sort longitude to be monotonically increasing (-180 to 180)
        sort_idx = np.argsort(lon)
        lon = lon[sort_idx]
        need_roll = True
    else:
        lon = lon_original
        sort_idx = None
        need_roll = False
    
    # Convert time to proper datetime
    if hasattr(time_val, 'to_pydatetime'):
        time_dt = time_val.to_pydatetime()
    elif isinstance(time_val, datetime):
        time_dt = time_val
    else:
        time_dt = datetime.fromisoformat(str(time_val))
    
    # Ensure timezone
    if time_dt.tzinfo is None:
        time_dt = time_dt.replace(tzinfo=timezone.utc)
    
    # Convert to numpy datetime64 for NetCDF serialization
    # Remove timezone info (NetCDF doesn't support timezone-aware datetimes)
    time_dt_naive = time_dt.replace(tzinfo=None)
    time_np = np.datetime64(time_dt_naive, 'ns')
    
    # =========================================================================
    # Surface Variables (2D: lat x lon)
    # =========================================================================
    surface_data_vars = {}
    surface_var_names = {
        'msl': {'long_name': 'Mean sea level pressure', 'units': 'Pa'},
        't2m': {'long_name': '2 metre temperature', 'units': 'K'},
        'u10': {'long_name': '10 metre U wind component', 'units': 'm s**-1'},
        'v10': {'long_name': '10 metre V wind component', 'units': 'm s**-1'},
    }
    
    for var_name, tensor in pred_batch.surf_vars.items():
        # Get 2D data (take last time step if multiple)
        data = tensor.numpy() if hasattr(tensor, 'numpy') else np.array(tensor)
        if data.ndim > 2:
            # Shape is [batch, time, lat, lon] or [batch, lat, lon]
            # Take first batch, last time
            if data.ndim == 4:
                data = data[0, -1, :, :]  # [batch, time, lat, lon] -> [lat, lon]
            elif data.ndim == 3:
                data = data[0, :, :]  # [batch, lat, lon] -> [lat, lon]
        
        # Roll data along longitude axis if needed (to match sorted longitude)
        if need_roll and sort_idx is not None:
            data = data[:, sort_idx]  # Reorder along longitude dimension
        
        # Create DataArray
        attrs = surface_var_names.get(var_name, {'long_name': var_name, 'units': 'unknown'})
        surface_data_vars[var_name] = xr.DataArray(
            data=data,
            dims=['latitude', 'longitude'],
            attrs=attrs
        )
    
    # Create surface dataset
    surface_ds = xr.Dataset(
        data_vars=surface_data_vars,
        coords={
            'latitude': ('latitude', lat, {'units': 'degrees_north', 'axis': 'Y', 'standard_name': 'latitude'}),
            'longitude': ('longitude', lon, {'units': 'degrees_east', 'axis': 'X', 'standard_name': 'longitude'}),
            'time': ('time', [time_np], {'units': 'nanoseconds since 1970-01-01', 'calendar': 'proleptic_gregorian'}),
        },
        attrs={
            'Conventions': 'CF-1.8',
            'title': f'Aurora Forecast - Hurricane {storm_name} {storm_year}',
            'institution': 'Microsoft Research - Aurora AI Weather Model',
            'source': f'Aurora {MODEL_NAME}',
            'history': f'Generated {datetime.now().isoformat()}',
            'forecast_reference_time': str(storm_init_time),
            'forecast_valid_time': str(time_dt),
        }
    )
    
    # =========================================================================
    # Atmospheric Variables (3D: level x lat x lon)
    # =========================================================================
    atmos_ds = None
    if include_atmos and hasattr(pred_batch, 'atmos_vars') and pred_batch.atmos_vars:
        levels = metadata.atmos_levels
        if hasattr(levels, 'numpy'):
            levels = levels.numpy()
        levels = np.array(levels)
        
        atmos_data_vars = {}
        atmos_var_names = {
            't': {'long_name': 'Temperature', 'units': 'K'},
            'u': {'long_name': 'U component of wind', 'units': 'm s**-1'},
            'v': {'long_name': 'V component of wind', 'units': 'm s**-1'},
            'q': {'long_name': 'Specific humidity', 'units': 'kg kg**-1'},
            'z': {'long_name': 'Geopotential', 'units': 'm**2 s**-2'},
        }
        
        for var_name, tensor in pred_batch.atmos_vars.items():
            data = tensor.numpy() if hasattr(tensor, 'numpy') else np.array(tensor)
            if data.ndim > 3:
                # Shape is [batch, time, level, lat, lon] or [batch, level, lat, lon]
                if data.ndim == 5:
                    data = data[0, -1, :, :, :]  # [batch, time, level, lat, lon] -> [level, lat, lon]
                elif data.ndim == 4:
                    data = data[0, :, :, :]  # [batch, level, lat, lon] -> [level, lat, lon]
            
            # Roll data along longitude axis if needed (to match sorted longitude)
            if need_roll and sort_idx is not None:
                data = data[:, :, sort_idx]  # Reorder along longitude dimension (last axis)
            
            attrs = atmos_var_names.get(var_name, {'long_name': var_name, 'units': 'unknown'})
            atmos_data_vars[var_name] = xr.DataArray(
                data=data,
                dims=['isobaricInhPa', 'latitude', 'longitude'],
                attrs=attrs
            )
        
        if atmos_data_vars:
            atmos_ds = xr.Dataset(
                data_vars=atmos_data_vars,
                coords={
                    'latitude': ('latitude', lat, {'units': 'degrees_north', 'axis': 'Y', 'standard_name': 'latitude'}),
                    'longitude': ('longitude', lon, {'units': 'degrees_east', 'axis': 'X', 'standard_name': 'longitude'}),
                    'isobaricInhPa': ('isobaricInhPa', levels, {'units': 'hPa', 'axis': 'Z', 'positive': 'down', 'standard_name': 'air_pressure'}),
                    'time': ('time', [time_np], {'units': 'nanoseconds since 1970-01-01', 'calendar': 'proleptic_gregorian'}),
                },
                attrs={
                    'Conventions': 'CF-1.8',
                    'title': f'Aurora Forecast Atmospheric - Hurricane {storm_name} {storm_year}',
                    'institution': 'Microsoft Research - Aurora AI Weather Model',
                    'source': f'Aurora {MODEL_NAME}',
                    'history': f'Generated {datetime.now().isoformat()}',
                    'forecast_reference_time': str(storm_init_time),
                    'forecast_valid_time': str(time_dt),
                }
            )
    
    return surface_ds, atmos_ds

# ============================================================================
# Convert and Upload All Predictions
# ============================================================================
print(f"\n🔄 Converting {len(preds)} Aurora predictions to xarray datasets...")

forecast_surface_files = []
forecast_atmos_files = []

for i, pred in enumerate(preds):
    forecast_time = pred.metadata.time[0]
    
    # Convert time for filename
    if hasattr(forecast_time, 'strftime'):
        date_str = forecast_time.strftime("%Y%m%d")
        time_str = forecast_time.strftime("%H%M")
    else:
        dt = datetime.fromisoformat(str(forecast_time).replace('Z', '+00:00'))
        date_str = dt.strftime("%Y%m%d")
        time_str = dt.strftime("%H%M")
    
    print(f"\n--- Prediction {i+1}/{len(preds)} ---")
    print(f"   📅 Forecast time: {forecast_time}")
    print(f"   ⏰ Lead time: +{(i+1)*6} hours")
    
    # Convert to xarray
    surface_ds, atmos_ds = batch_to_xarray(pred)
    
    # Clean up time coordinate attributes to avoid conflicts during NetCDF encoding
    # xarray's CF encoder will add these automatically
    for ds in [surface_ds, atmos_ds]:
        if 'time' in ds.coords:
            time_attrs_to_remove = ['units', 'calendar']
            for attr in time_attrs_to_remove:
                if attr in ds['time'].attrs:
                    del ds['time'].attrs[attr]
    
    # Generate unique suffix for this forecast run
    forecast_suffix = f"fcst_{unique_suffix}"
    
    # =========================================================================
    # Save and Upload Surface Data
    # =========================================================================
    surface_filename = f"hurricane_{storm_name.lower()}_forecast_{date_str}_{time_str}_{forecast_suffix}_surface_f{i:03d}.nc"
    surface_path = downloads_dir / surface_filename
    
    print(f"   📊 Surface variables: {list(surface_ds.data_vars)}")
    surface_ds.to_netcdf(surface_path, engine='netcdf4')
    print(f"   ✓ Saved surface data ({surface_path.stat().st_size / (1024*1024):.2f} MB)")
    
    # Upload to blob storage
    blob_url, sas_url = await upload_to_blob_storage(surface_path, surface_filename)
    
    forecast_surface_files.append({
        'index': i,
        'timestamp': forecast_time,
        'lead_hours': (i+1) * 6,
        'blob_url': blob_url,
        'sas_url': sas_url,
        'ds': surface_ds,
        'filename': surface_filename,
        'type': 'surface'
    })
    print(f"   ✅ Uploaded surface: {surface_filename}")
    
    # =========================================================================
    # Save and Upload Atmospheric Data (if available)
    # =========================================================================
    if atmos_ds is not None:
        atmos_filename = f"hurricane_{storm_name.lower()}_forecast_{date_str}_{time_str}_{forecast_suffix}_atmos_f{i:03d}.nc"
        atmos_path = downloads_dir / atmos_filename
        
        print(f"   🌡️ Atmospheric variables: {list(atmos_ds.data_vars)}")
        atmos_ds.to_netcdf(atmos_path, engine='netcdf4')
        print(f"   ✓ Saved atmospheric data ({atmos_path.stat().st_size / (1024*1024):.2f} MB)")
        
        blob_url_atmos, sas_url_atmos = await upload_to_blob_storage(atmos_path, atmos_filename)
        
        forecast_atmos_files.append({
            'index': i,
            'timestamp': forecast_time,
            'lead_hours': (i+1) * 6,
            'blob_url': blob_url_atmos,
            'sas_url': sas_url_atmos,
            'ds': atmos_ds,
            'filename': atmos_filename,
            'type': 'atmospheric',
            'levels': list(atmos_ds.coords['isobaricInhPa'].values)
        })
        print(f"   ✅ Uploaded atmospheric: {atmos_filename}")

print(f"\n" + "="*80)
print(f"✅ UPLOAD COMPLETE")
print(f"   Surface files: {len(forecast_surface_files)}")
print(f"   Atmospheric files: {len(forecast_atmos_files)}")
print("="*80)

# ============================================================================
# Create STAC Collection for Forecast Data
# ============================================================================
print(f"\n🗂️ Creating STAC Collection: {OUTPUT_COLLECTION}")

await ensure_collection_exists(
    OUTPUT_COLLECTION,
    f"Hurricane {storm_name} {storm_year} Forecast Data",
    f"Aurora AI model forecast predictions for Hurricane {storm_name} ({storm_year}). "
    f"Generated from initial conditions at {storm_init_time}. "
    f"Contains {len(preds)} forecast steps at 6-hourly intervals."
)

# ============================================================================
# Create and Ingest STAC Items for Each Forecast Step
# ============================================================================
print(f"\n📥 Ingesting {len(forecast_surface_files)} STAC items...")

forecast_ingested_items = []

for surface_info in forecast_surface_files:
    i = surface_info['index']
    ts = surface_info['timestamp']
    sas_url = surface_info['sas_url']
    surface_ds = surface_info['ds']
    lead_hours = surface_info['lead_hours']
    
    print(f"\n--- Forecast Item {i+1}/{len(forecast_surface_files)} ---")
    print(f"   📅 Forecast time: {ts} (+{lead_hours}h)")
    
    # Ensure timestamp has timezone
    if hasattr(ts, 'tzinfo') and ts.tzinfo is None:
        ts = ts.replace(tzinfo=timezone.utc)
    elif not hasattr(ts, 'tzinfo'):
        ts = datetime.fromisoformat(str(ts).replace('Z', '+00:00'))
        if ts.tzinfo is None:
            ts = ts.replace(tzinfo=timezone.utc)
    
    # Create unique item ID
    if hasattr(ts, 'strftime'):
        ts_str = ts.strftime('%Y%m%d-%H%M')
    else:
        ts_str = str(ts).replace(':', '').replace(' ', '-')[:15]
    
    item_id = f"hurricane-{storm_name.lower()}-forecast-{ts_str}-f{i:03d}-{unique_suffix}"
    print(f"   🆔 Item ID: {item_id}")
    
    # Create STAC item
    stac_item = await create_stac_item(
        item_id=item_id,
        date_time=ts,
        bbox=[-180, -90, 180, 90],
        data_url=sas_url,
        dataset=surface_ds
    )
    
    # Add forecast-specific properties
    stac_item.properties['forecast:reference_time'] = str(storm_init_time)
    stac_item.properties['forecast:lead_time'] = f"PT{lead_hours}H"
    stac_item.properties['forecast:step'] = i + 1
    stac_item.properties['model'] = f"Aurora {MODEL_NAME}"
    
    # Add atmospheric asset if available
    if i < len(forecast_atmos_files):
        atmos_info = forecast_atmos_files[i]
        atmos_sas_url = atmos_info['sas_url']
        
        stac_item.add_asset(
            "atmospheric",
            pystac.Asset(
                href=atmos_sas_url,
                media_type="application/x-netcdf",
                title=f"Aurora Forecast Atmospheric Data (NetCDF)",
                description=f"3D atmospheric forecast variables on pressure levels ({len(atmos_info['levels'])} levels)",
                roles=["data"],
                extra_fields={
                    "type": "application/x-netcdf",
                    "xarray:open_kwargs": {"engine": "h5netcdf"}
                }
            )
        )
        print(f"   🌡️ Added atmospheric asset with {len(atmos_info['levels'])} levels")
    
    # Ingest to GeoCatalog
    await ingest_to_geocatalog(stac_item, OUTPUT_COLLECTION)
    
    forecast_ingested_items.append({
        'item_id': item_id,
        'timestamp': ts,
        'lead_hours': lead_hours,
        'sas_url': sas_url,
        'has_atmospheric': i < len(forecast_atmos_files)
    })
    
    print(f"   ✅ Ingested: {item_id}")

print(f"\n" + "="*80)
print(f"✅ ALL FORECAST ITEMS INGESTED!")
print(f"   Collection: {OUTPUT_COLLECTION}")
print(f"   Total items: {len(forecast_ingested_items)}")
print("="*80)

# Store references for render configuration
forecast_surface_ds = forecast_surface_files[0]['ds'] if forecast_surface_files else None
forecast_atmos_ds = forecast_atmos_files[0]['ds'] if forecast_atmos_files else None
forecast_atmos_levels = forecast_atmos_files[0]['levels'] if forecast_atmos_files else []

### 6.3 Configure Visualization for Forecast Data

Apply render options and mosaic configuration to enable visualization in GeoCatalog Explorer.

In [ ]:
# ============================================================================
# CONFIGURE VISUALIZATION FOR FORECAST DATA
# ============================================================================
# Apply render options and mosaics to enable GeoCatalog Explorer visualization

print("="*70)
print("🚀 CONFIGURING VISUALIZATION FOR FORECAST DATA")
print("="*70)
print(f"\nCollection: {OUTPUT_COLLECTION}")
print(f"Endpoint: {GEOCATALOG_URI}")

# ============================================================================
# Ensure we have references to datasets for render configuration
# ============================================================================
# These should be set from the upload cell, but let's verify and recover if needed
if 'forecast_surface_ds' not in dir() or forecast_surface_ds is None:
    if 'forecast_surface_files' in dir() and forecast_surface_files:
        forecast_surface_ds = forecast_surface_files[0]['ds']
        print("📋 Retrieved forecast_surface_ds from uploaded files")
    else:
        forecast_surface_ds = None
        print("⚠️ No forecast surface dataset available")

if 'forecast_atmos_ds' not in dir() or forecast_atmos_ds is None:
    if 'forecast_atmos_files' in dir() and forecast_atmos_files:
        forecast_atmos_ds = forecast_atmos_files[0]['ds']
        forecast_atmos_levels = forecast_atmos_files[0]['levels']
        print("📋 Retrieved forecast_atmos_ds from uploaded files")
    else:
        forecast_atmos_ds = None
        forecast_atmos_levels = []
        print("⚠️ No forecast atmospheric dataset available")

if 'forecast_atmos_levels' not in dir():
    forecast_atmos_levels = forecast_atmos_files[0]['levels'] if ('forecast_atmos_files' in dir() and forecast_atmos_files) else []

# ============================================================================
# Define Forecast-Specific Render Configuration
# ============================================================================

# Surface variables for forecast visualization
forecast_surface_config = {
    "msl": {"id": "msl-pressure", "name": "Mean Sea Level Pressure (Forecast)", "rescale": "95000,105000", "colormap": "rdylbu_r"},
    "t2m": {"id": "t2m-temp", "name": "2m Temperature (Forecast)", "rescale": "270,310", "colormap": "plasma"},
    "u10": {"id": "u10-wind", "name": "10m U Wind (Forecast)", "rescale": "-50,50", "colormap": "coolwarm"},
    "v10": {"id": "v10-wind", "name": "10m V Wind (Forecast)", "rescale": "-50,50", "colormap": "coolwarm"},
}

# Atmospheric variables for forecast visualization
forecast_atmos_config = {
    "t-850hpa": {"var": "t", "level": 850, "name": "Temperature 850 hPa (Forecast)", "rescale": "260,300", "colormap": "plasma"},
    "t-500hpa": {"var": "t", "level": 500, "name": "Temperature 500 hPa (Forecast)", "rescale": "230,270", "colormap": "plasma"},
    "t-200hpa": {"var": "t", "level": 200, "name": "Temperature 200 hPa (Forecast)", "rescale": "200,240", "colormap": "plasma"},
    "z-850hpa": {"var": "z", "level": 850, "name": "Geopotential 850 hPa (Forecast)", "rescale": "10000,17000", "colormap": "viridis"},
    "z-500hpa": {"var": "z", "level": 500, "name": "Geopotential 500 hPa (Forecast)", "rescale": "50000,60000", "colormap": "viridis"},
    "u-850hpa": {"var": "u", "level": 850, "name": "U Wind 850 hPa (Forecast)", "rescale": "-50,50", "colormap": "coolwarm"},
    "u-200hpa": {"var": "u", "level": 200, "name": "U Wind 200 hPa (Jet, Forecast)", "rescale": "-80,80", "colormap": "coolwarm"},
    "v-850hpa": {"var": "v", "level": 850, "name": "V Wind 850 hPa (Forecast)", "rescale": "-50,50", "colormap": "coolwarm"},
    "v-200hpa": {"var": "v", "level": 200, "name": "V Wind 200 hPa (Jet, Forecast)", "rescale": "-80,80", "colormap": "coolwarm"},
    "q-850hpa": {"var": "q", "level": 850, "name": "Humidity 850 hPa (Forecast)", "rescale": "0,0.02", "colormap": "blues"},
}

print(f"\n📊 Surface variables: {list(forecast_surface_config.keys())}")
print(f"🌡️ Atmospheric variables: {len(forecast_atmos_config)} render options")

# Show current configuration
await show_collection_configuration(pc_client, OUTPUT_COLLECTION)

# ============================================================================
# Configure Render Options for Forecast Collection
# ============================================================================

async def configure_forecast_render_options(client, collection_id, surface_ds, atmos_ds=None, levels=None):
    """Configure render options specifically for forecast data."""
    
    print("\n" + "="*70)
    print("🎨 CONFIGURING FORECAST RENDER OPTIONS")
    print("="*70)
    
    # Get existing render options
    try:
        existing_renders = client.stac.list_render_options(collection_id=collection_id)
        existing_ids = [r.id for r in existing_renders] if existing_renders else []
        print(f"📋 Found {len(existing_ids)} existing render options")
    except Exception as e:
        print(f"⚠️ Could not list existing render options: {e}")
        existing_ids = []
    
    # Delete existing render options
    if existing_ids:
        print(f"\n🗑️ Deleting {len(existing_ids)} existing render options...")
        for render_id in existing_ids:
            try:
                client.stac.delete_render_option(collection_id=collection_id, render_option_id=render_id)
                print(f"   ✓ Deleted: {render_id}")
            except Exception as e:
                print(f"   ⚠️ Failed to delete {render_id}: {e}")
    
    success_count = 0
    failed_count = 0
    
    # Create surface render options
    print(f"\n➕ Creating surface variable render options...")
    for var_name, config in forecast_surface_config.items():
        render_id = config.get("id", f"{var_name}-data")
        
        if surface_ds is None or var_name not in surface_ds.data_vars:
            print(f"   ⏭️ Skipping '{var_name}' - not in dataset")
            continue
        
        options_str = f"assets=data&subdataset_name={var_name}&rescale={config['rescale']}&colormap_name={config['colormap']}"
        
        try:
            render_option = RenderOption(
                id=render_id,
                name=config["name"],
                type=RenderOptionType.RASTER_TILE,
                options=options_str,
                min_zoom=0,
            )
            client.stac.create_render_option(collection_id=collection_id, body=render_option)
            print(f"   ✓ Created: {render_id} - {config['name']}")
            success_count += 1
        except Exception as e:
            print(f"   ⚠️ Failed to create {render_id}: {e}")
            failed_count += 1
    
    # Create atmospheric render options (if available)
    if atmos_ds is not None and levels:
        print(f"\n➕ Creating atmospheric variable render options...")
        for render_id, config in forecast_atmos_config.items():
            var_name = config["var"]
            level = config["level"]
            
            if var_name not in atmos_ds.data_vars:
                print(f"   ⏭️ Skipping '{render_id}' - {var_name} not in dataset")
                continue
            
            if level not in levels:
                print(f"   ⏭️ Skipping '{render_id}' - level {level} not available")
                continue
            
            options_str = f"assets=atmospheric&subdataset_name={var_name}&sel=isobaricInhPa:{level}&rescale={config['rescale']}&colormap_name={config['colormap']}"
            
            try:
                render_option = RenderOption(
                    id=render_id,
                    name=config["name"],
                    type=RenderOptionType.RASTER_TILE,
                    options=options_str,
                    min_zoom=0,
                )
                client.stac.create_render_option(collection_id=collection_id, body=render_option)
                print(f"   ✓ Created: {render_id} - {config['name']}")
                success_count += 1
            except Exception as e:
                print(f"   ⚠️ Failed to create {render_id}: {e}")
                failed_count += 1
    
    print(f"\n📊 Render Options Summary: {success_count} created, {failed_count} failed")
    return success_count > 0

# Configure render options
forecast_render_success = await configure_forecast_render_options(
    pc_client,
    OUTPUT_COLLECTION,
    forecast_surface_ds,
    atmos_ds=forecast_atmos_ds,
    levels=forecast_atmos_levels
)

# Configure mosaics (reuse existing function)
forecast_mosaic_success = await configure_mosaics(pc_client, OUTPUT_COLLECTION)

# Show updated configuration
await show_collection_configuration(pc_client, OUTPUT_COLLECTION)

# Summary
print("\n" + "="*70)
print("📊 FORECAST VISUALIZATION CONFIGURATION SUMMARY")
print("="*70)
print(f"✅ Render Options: {'Configured' if forecast_render_success else 'Failed'}")
print(f"✅ Mosaics: {'Configured' if forecast_mosaic_success else 'Failed'}")

# ============================================================================
# Open Explorer for Forecast Data
# ============================================================================
import webbrowser

# Build Explorer URL for forecast data
forecast_explorer_url = (
    f"{GEOCATALOG_URI}/explorer"
    f"?c=0.0000%2C0.0000"             # Center coordinates
    f"&z=2.00"                        # Zoom level
    f"&v=2"                           # View version
    f"&d={OUTPUT_COLLECTION}"         # Forecast collection
    f"&m=default"                     # Mosaic
    f"&r=msl-pressure"                # Default render (MSL pressure)
    f"&s=false%3A%3A100%3A%3Atrue"    # Settings
    f"&sr=desc"                       # Sort order
    f"&ae=0"                          # Animation
)

print(f"\n🌐 Forecast Explorer URL: {forecast_explorer_url}")

# Wait for backend propagation
import time
print(f"\n⏳ Waiting 30 seconds for render configuration to propagate...")
for remaining in range(30, 0, -10):
    print(f"   {remaining} seconds remaining...")
    time.sleep(10)
print(f"   Done waiting!")

print(f"\n📝 Opening GeoCatalog Explorer to visualize FORECAST data...")
webbrowser.open(forecast_explorer_url)

print(f"\n" + "="*70)
print(f"🎉 FORECAST DATA VISUALIZATION READY!")
print(f"="*70)
print(f"\n📊 Collections Created:")
print(f"   📥 Input Data:    {INPUT_COLLECTION}")
print(f"   📤 Forecast Data: {OUTPUT_COLLECTION}")
print(f"\n🔗 Explorer Links:")
print(f"   Input:    {GEOCATALOG_URI}/explorer?d={INPUT_COLLECTION}")
print(f"   Forecast: {GEOCATALOG_URI}/explorer?d={OUTPUT_COLLECTION}")

## 7. Visualize Hurricane Track

Compare Aurora's predicted storm track against observed/best-track data using Cartopy. The visualization includes:

- **Observed Track** (black) - Historical best-track data from HURDAT/IBTrACS
- **Forecast Track** (blue dashed) - Aurora model predictions
- **Cone of Uncertainty** (orange) - Expanding uncertainty region based on forecast lead time

### 7.1 Visualize the data in matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.lines import Line2D
from matplotlib.patches import Polygon
from matplotlib.collections import PatchCollection
import sys

# Ensure matplotlib outputs inline in notebooks
%matplotlib inline

# Get the track results from the tracker
track = tracker.results()

# Add converted longitude column if not already present
if 'lon_converted' not in track.columns:
    track['lon_converted'] = track['lon'].apply(lambda x: x - 360 if x > 180 else x)

# Prepare observed track data for plotting
if selected_storm_df is not None:
    observed_track = selected_storm_df.copy()
    # Convert longitude to the same convention (-180 to 180)
    observed_track['lon_converted'] = observed_track['lon'].apply(lambda x: x - 360 if x > 180 else x)
    print(f"✓ Observed track loaded with {len(observed_track)} points")
else:
    observed_track = None
    print("⚠️ No observed track data available")

def calculate_cone_of_uncertainty(lats, lons, base_radius=0.5, growth_rate=0.15, end_cap_points=20, end_cap_multiplier=1.5):
    """
    Calculate cone of uncertainty polygon coordinates with a smooth rounded end cap.
    
    Parameters:
    - lats, lons: Track coordinates
    - base_radius: Initial radius in degrees at t=0
    - growth_rate: How much the radius grows per forecast step (degrees)
    - end_cap_points: Number of points to use for the semicircular end cap
    - end_cap_multiplier: How much wider the end cap should be (1.0 = same as last radius, 1.5 = 50% wider)
    
    Returns:
    - cone_lats, cone_lons: Full polygon coordinates including rounded end cap
    """
    n_points = len(lats)
    left_lats, left_lons = [], []
    right_lats, right_lons = [], []
    
    # Store the direction at the last point for the end cap
    last_dx, last_dy = 0, 0
    last_radius = 0
    
    for i in range(n_points):
        # Radius grows with forecast time
        radius = base_radius + (i * growth_rate)
        
        # Calculate perpendicular direction
        if i == 0:
            # First point: use direction to next point
            dx = lons[1] - lons[0]
            dy = lats[1] - lats[0]
        elif i == n_points - 1:
            # Last point: use direction from previous point
            dx = lons[i] - lons[i-1]
            dy = lats[i] - lats[i-1]
        else:
            # Middle points: use average direction
            dx = lons[i+1] - lons[i-1]
            dy = lats[i+1] - lats[i-1]
        
        # Normalize and get perpendicular
        length = np.sqrt(dx**2 + dy**2)
        if length > 0:
            norm_dx = dx / length
            norm_dy = dy / length
            perp_x = -norm_dy
            perp_y = norm_dx
        else:
            norm_dx, norm_dy = 1, 0
            perp_x, perp_y = 0, 1
        
        # Store last point info for end cap
        if i == n_points - 1:
            last_dx = norm_dx
            last_dy = norm_dy
            last_radius = radius
        
        # Calculate left and right edge points
        left_lats.append(lats[i] + radius * perp_y)
        left_lons.append(lons[i] + radius * perp_x)
        right_lats.append(lats[i] - radius * perp_y)
        right_lons.append(lons[i] - radius * perp_x)
    
    # Create semicircular end cap at the last point
    end_cap_lons = []
    end_cap_lats = []
    last_lat = lats[-1]
    last_lon = lons[-1]
    
    # Calculate the angle of the track direction
    track_angle = np.arctan2(last_dy, last_dx)
    
    # Use multiplier to make end cap wider (umbrella shape)
    end_cap_radius = last_radius * end_cap_multiplier
    
    # Generate a full circle at the end point
    for j in range(end_cap_points + 1):
        # Full circle: 0 to 2*pi
        angle = (j * 2 * np.pi / end_cap_points)
        end_cap_lons.append(last_lon + end_cap_radius * np.cos(angle))
        end_cap_lats.append(last_lat + end_cap_radius * np.sin(angle))
    
    # Combine: left edge forward, end cap, right edge backward
    cone_lons = left_lons + end_cap_lons + right_lons[::-1]
    cone_lats = left_lats + end_cap_lats + right_lats[::-1]
    
    return cone_lats, cone_lons, left_lats, left_lons, right_lats, right_lons

# Create the plot with Cartopy projection
fig, ax = plt.subplots(figsize=(14, 10), subplot_kw={'projection': ccrs.PlateCarree()})

# Calculate dynamic map extent based on both storm tracks (observed and forecast)
all_lats = list(selected_storm.lat)
all_lons = list(selected_storm.lon)

# Include forecast track in extent calculation
all_lats.extend(track['lat'].tolist())
all_lons.extend(track['lon_converted'].tolist())

# Calculate bounds with padding
lat_min = min(all_lats) - 5
lat_max = max(all_lats) + 5
lon_min = min(all_lons) - 5
lon_max = max(all_lons) + 5

# Set map extent
try:
    ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
except (AssertionError, KeyError) as e:
    print(f"⚠️ set_extent() failed, using xlim/ylim instead")
    ax.set_xlim(lon_min, lon_max)
    ax.set_ylim(lat_min, lat_max)

print(f"Map extent set to: lon=[{lon_min:.1f}, {lon_max:.1f}], lat=[{lat_min:.1f}, {lat_max:.1f}]")

# Add map features
ax.coastlines(resolution='50m', linewidth=1)
ax.add_feature(cfeature.LAND, facecolor='lightgray')
ax.add_feature(cfeature.OCEAN, facecolor='lightblue')
ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.5)
ax.add_feature(cfeature.STATES, linestyle='-', linewidth=0.3, edgecolor='gray')

# Add gridlines
gl = ax.gridlines(draw_labels=True, linewidth=0.5, color='gray', alpha=0.5, linestyle='--')
gl.top_labels = False
gl.right_labels = False

# Calculate and draw cone of uncertainty using Shapely for smooth geometry
lats = track['lat'].values
lons = track['lon_converted'].values

from shapely.geometry import LineString, Point
from shapely.ops import unary_union

# Create cone by buffering each segment with increasing radius
base_radius = 0.3
growth_rate = 0.12
end_multiplier = 1.5  # Make the end 50% wider
n_points = len(lats)

# Create circles at each point with increasing radius, then take the convex hull
circles = []
for i in range(n_points):
    radius = base_radius + (i * growth_rate)
    
    # Apply extra multiplier for the last few points to make end wider
    if i >= n_points - 3:
        # Gradually increase multiplier for last 3 points
        progress = (i - (n_points - 3)) / 2  # 0, 0.5, 1.0 for last 3 points
        radius *= (1 + progress * (end_multiplier - 1))
    
    point = Point(lons[i], lats[i])
    circle = point.buffer(radius, resolution=32)  # High resolution for smooth circles
    circles.append(circle)

# Union all circles to create the cone shape
cone_shape = unary_union(circles)

# Also add buffered line segments to fill gaps between circles
for i in range(n_points - 1):
    radius_start = base_radius + (i * growth_rate)
    radius_end = base_radius + ((i + 1) * growth_rate)
    avg_radius = (radius_start + radius_end) / 2
    
    segment = LineString([(lons[i], lats[i]), (lons[i+1], lats[i+1])])
    buffered_segment = segment.buffer(avg_radius, cap_style=2, resolution=32)  # cap_style=2 is flat
    circles.append(buffered_segment)

# Final union for complete smooth cone
cone_shape = unary_union(circles)

# Extract coordinates from the shapely polygon
if cone_shape.geom_type == 'Polygon':
    cone_coords = list(cone_shape.exterior.coords)
elif cone_shape.geom_type == 'MultiPolygon':
    # Take the largest polygon
    largest = max(cone_shape.geoms, key=lambda p: p.area)
    cone_coords = list(largest.exterior.coords)

# Draw the cone
cone_polygon = plt.Polygon(
    cone_coords,
    facecolor='orange',
    edgecolor='darkorange',
    linewidth=1.5,
    alpha=0.3,
    transform=ccrs.PlateCarree(),
    zorder=0
)
ax.add_patch(cone_polygon)

# ============================================================================
# Plot the OBSERVED track (from tropycal/HURDAT data)
# ============================================================================
if observed_track is not None:
    # Plot observed track line
    ax.plot(
        observed_track['lon_converted'], 
        observed_track['lat'], 
        color='black', 
        linewidth=3, 
        linestyle='-',
        transform=ccrs.PlateCarree(),
        zorder=1,
        label='Observed Track'
    )
    
    # Plot observed track points
    ax.plot(
        observed_track['lon_converted'], 
        observed_track['lat'], 
        'o',
        color='black',
        markersize=6,
        markeredgecolor='white',
        markeredgewidth=0.5,
        transform=ccrs.PlateCarree(),
        zorder=1
    )
    
    # Mark observed start point
    ax.plot(
        observed_track['lon_converted'].iloc[0], 
        observed_track['lat'].iloc[0], 
        marker='o',
        color='green',
        markersize=12, 
        markeredgecolor='black',
        markeredgewidth=1.5,
        transform=ccrs.PlateCarree(),
        zorder=3
    )
    
    # Mark observed end point (landfall/dissipation)
    ax.plot(
        observed_track['lon_converted'].iloc[-1], 
        observed_track['lat'].iloc[-1], 
        marker='X',
        color='darkred',
        markersize=14, 
        markeredgecolor='black',
        markeredgewidth=1.5,
        transform=ccrs.PlateCarree(),
        zorder=3
    )

# ============================================================================
# Plot the FORECAST track (from Aurora model)
# ============================================================================
# Plot the forecast track line
ax.plot(
    track['lon_converted'], 
    track['lat'], 
    color='blue', 
    linewidth=2.5, 
    linestyle='--',
    transform=ccrs.PlateCarree(),
    zorder=2,
    label='Aurora Forecast'
)

# Plot each forecast point
ax.plot(
    track['lon_converted'], 
    track['lat'], 
    'o',
    color='blue',
    markersize=8,
    markeredgecolor='white',
    markeredgewidth=0.5,
    transform=ccrs.PlateCarree(),
    zorder=2
)

# Mark forecast start point with a special marker
ax.plot(
    track['lon_converted'].iloc[0], 
    track['lat'].iloc[0], 
    marker='*',
    color='lime',
    markersize=18, 
    markeredgecolor='black',
    markeredgewidth=1,
    transform=ccrs.PlateCarree(),
    zorder=4
)

# Mark forecast end point with a special marker
ax.plot(
    track['lon_converted'].iloc[-1], 
    track['lat'].iloc[-1], 
    marker='s',
    color='red',
    markersize=12, 
    markeredgecolor='black',
    markeredgewidth=1,
    transform=ccrs.PlateCarree(),
    zorder=4
)

# Add time labels for forecast track (only once per day at 12Z)
labeled_dates = set()
for i, row in track.iterrows():
    date_str = row['time'].strftime('%Y-%m-%d')
    hour = row['time'].hour
    
    # Label at 12Z (noon) or if it's the first point of the day
    if date_str not in labeled_dates and hour == 12:
        labeled_dates.add(date_str)
        ax.annotate(
            row['time'].strftime('%m/%d'),
            xy=(row['lon_converted'], row['lat']),
            xytext=(8, 8),
            textcoords='offset points',
            fontsize=9,
            fontweight='bold',
            color='blue',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7),
            transform=ccrs.PlateCarree(),
            zorder=5
        )

# If no 12Z points were labeled, label at 00Z
if not labeled_dates:
    for i, row in track.iterrows():
        date_str = row['time'].strftime('%Y-%m-%d')
        hour = row['time'].hour
        if date_str not in labeled_dates and hour == 0:
            labeled_dates.add(date_str)
            ax.annotate(
                row['time'].strftime('%m/%d'),
                xy=(row['lon_converted'], row['lat']),
                xytext=(8, 8),
                textcoords='offset points',
                fontsize=9,
                fontweight='bold',
                color='blue',
                bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7),
                transform=ccrs.PlateCarree(),
                zorder=5
            )

# Create legend with both tracks
legend_elements = [
    Line2D([0], [0], color='black', linewidth=3, linestyle='-', marker='o', markersize=6, 
           markeredgecolor='white', label='Observed Track'),
    Line2D([0], [0], color='blue', linewidth=2.5, linestyle='--', marker='o', markersize=6, 
           markeredgecolor='white', label='Aurora Forecast'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='green', markersize=10, 
           markeredgecolor='black', label='Observed Start'),
    Line2D([0], [0], marker='X', color='w', markerfacecolor='darkred', markersize=10, 
           markeredgecolor='black', label='Observed End'),
    Line2D([0], [0], marker='*', color='w', markerfacecolor='lime', markersize=14, 
           markeredgecolor='black', label='Forecast Start'),
    Line2D([0], [0], marker='s', color='w', markerfacecolor='red', markersize=10, 
           markeredgecolor='black', label='Forecast End'),
    plt.Rectangle((0,0), 1, 1, facecolor='orange', edgecolor='darkorange', alpha=0.3, 
                  label='Cone of Uncertainty'),
]

ax.legend(handles=legend_elements, loc='upper right', fontsize=9, framealpha=0.9)

# Add title
ax.set_title(
    f"Hurricane {selected_storm.name} ({selected_storm.year}) - Observed vs Aurora Model Forecast\n"
    f"Observed: {len(observed_track) if observed_track is not None else 0} points | "
    f"Forecast: {len(track)} points (6-hour intervals)",
    fontsize=14,
    fontweight='bold'
)

plt.tight_layout()

# Display the figure
fig

# Print track summary for both observed and forecast
print("\n" + "="*70)
print("📊 TRACK COMPARISON SUMMARY")
print("="*70)

if observed_track is not None:
    print("\n🌀 OBSERVED TRACK (HURDAT):")
    print(f"   Start: {observed_track['time'].iloc[0]} at ({observed_track['lat'].iloc[0]:.2f}°N, {observed_track['lon_converted'].iloc[0]:.2f}°)")
    print(f"   End:   {observed_track['time'].iloc[-1]} at ({observed_track['lat'].iloc[-1]:.2f}°N, {observed_track['lon_converted'].iloc[-1]:.2f}°)")
    print(f"   Total points: {len(observed_track)}")
    print(f"   Max wind: {observed_track['vmax'].max():.0f} kt")
    print(f"   Min pressure: {observed_track['mslp'].min():.0f} hPa")

print("\n🔮 AURORA FORECAST TRACK:")
print(f"   Start: {track['time'].iloc[0]} at ({track['lat'].iloc[0]:.2f}°N, {track['lon_converted'].iloc[0]:.2f}°)")
print(f"   End:   {track['time'].iloc[-1]} at ({track['lat'].iloc[-1]:.2f}°N, {track['lon_converted'].iloc[-1]:.2f}°)")
print(f"   Total forecast steps: {len(track)}")
print(f"   Max wind: {track['wind'].max():.1f} kt")
print(f"   Min pressure: {track['msl'].min()/100:.1f} hPa")
print("="*70)

In [ ]:
# =============================================================================
# SECTION 7.1b: Asymmetric Intensity-Modulated Impact Swath & Coastal Zones
# =============================================================================
#   Part 1 – Asymmetric intensity-modulated impact swath along forecast track
#   Part 2 – Track-wide coastal impact zones (NHC-style surge / flood warnings)
#   Part 3 – Composed static matplotlib / Cartopy figure
#
#   KEY IMPROVEMENTS (v2):
#     • NHC radii now correctly used as per-side offsets (were halved before)
#     • Pressure-derived wind-extent buffer layered on track uncertainty
#     • Right-of-track asymmetry for Northern Hemisphere cyclones
#     • Morphological polygon smoothing
# =============================================================================

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Polygon as MplPolygon
from matplotlib.lines import Line2D
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cartopy.io.shapereader as shpreader
from shapely.geometry import Point, LineString, box as shapely_box
from shapely.geometry import Polygon as ShapelyPolygon, MultiPolygon
from shapely.ops import unary_union
from shapely.prepared import prep

%matplotlib inline

print("=" * 70)
print("  Asymmetric Intensity-Modulated Impact Swath & Coastal Impact Zones")
print("=" * 70)

# ─────────────────────────────────────────────────────────────────────────────
# 0.  HELPERS
# ─────────────────────────────────────────────────────────────────────────────

def km_to_degrees(km, lat):
    """Convert kilometres → approximate degrees of longitude at *lat*."""
    return km / (111.0 * max(np.cos(np.radians(abs(lat))), 0.01))

# NHC official track-uncertainty radii (km) at each lead hour
_NHC_H  = [0, 6, 12, 24, 36, 48, 72, 96, 120]
_NHC_KM = [0, 40, 65, 100, 140, 175, 240, 325, 410]

def nhc_radius_km(lead_hours):
    """Linearly interpolate NHC uncertainty radius for any lead time."""
    return float(np.interp(lead_hours, _NHC_H, _NHC_KM))

def knaff_zehr(mslp_mb):
    """Knaff-Zehr pressure→wind:  Vmax (kt) from MSLP (hPa)."""
    if np.isnan(mslp_mb):
        return 0.0
    dp = max(1013.0 - mslp_mb, 0.0)
    return 6.3 * np.sqrt(dp) + 0.15 * dp

def ss_cat(vmax):
    """Saffir-Simpson category string from Vmax (kt)."""
    for thresh, label in [(137, 'Cat 5'), (113, 'Cat 4'), (96, 'Cat 3'),
                          (83, 'Cat 2'), (64, 'Cat 1'), (34, 'TS')]:
        if vmax >= thresh:
            return label
    return 'TD'

_SS_CLR = {
    'TD': '#5B9BD5', 'TS': '#00B050', 'Cat 1': '#FFFF00',
    'Cat 2': '#FFC000', 'Cat 3': '#FF0000', 'Cat 4': '#C00000',
    'Cat 5': '#7030A0',
}

def _add_geom(ax, geom, fc, alpha, zorder, ec='none', lw=0):
    """Render a Shapely Polygon/MultiPolygon onto a Cartopy GeoAxes."""
    if geom is None or geom.is_empty:
        return
    if geom.geom_type == 'GeometryCollection':
        for g in geom.geoms:
            _add_geom(ax, g, fc, alpha, zorder, ec, lw)
        return
    if geom.geom_type not in ('Polygon', 'MultiPolygon'):
        return
    polys = [geom] if geom.geom_type == 'Polygon' else list(geom.geoms)
    for p in polys:
        xy = np.array(p.exterior.coords)
        patch = MplPolygon(xy, closed=True, fc=fc, ec=ec, alpha=alpha,
                           lw=lw, transform=ccrs.PlateCarree(), zorder=zorder)
        ax.add_patch(patch)

# ─────────────────────────────────────────────────────────────────────────────
# 1.  EXTRACT FORECAST POSITIONS
# ─────────────────────────────────────────────────────────────────────────────

# Re-use the 'track' DataFrame already computed by the previous cell
_trk = track.copy()
if 'lon_converted' not in _trk.columns:
    _trk['lon_converted'] = _trk['lon'].apply(
        lambda x: x - 360 if x > 180 else x)

# Determine MSLP column – 'msl' (Pa) from Aurora, or 'mslp' (hPa) from obs
if 'msl' in _trk.columns and _trk['msl'].notna().any():
    _trk['_mslp_mb'] = _trk['msl'] / 100.0
elif 'mslp' in _trk.columns and _trk['mslp'].notna().any():
    _trk['_mslp_mb'] = _trk['mslp']
else:
    # Fallback: use 1013 (neutral) if no pressure data available
    print("  ⚠️  No MSLP data found – using 1013 mb fallback")
    _trk['_mslp_mb'] = 1013.0

# Fill any remaining NaNs with column mean
_mslp_mean = _trk['_mslp_mb'].mean()
if np.isnan(_mslp_mean):
    _mslp_mean = 1013.0
_trk['_mslp_mb'] = _trk['_mslp_mb'].fillna(_mslp_mean)

print(f"\n  Track columns     : {list(_trk.columns)}")

_t0 = _trk['time'].iloc[0]
track_points = []
for _, _r in _trk.iterrows():
    track_points.append(dict(
        lat        = float(_r['lat']),
        lon        = float(_r['lon_converted']),
        lead_hours = (_r['time'] - _t0).total_seconds() / 3600.0,
        mslp_mb    = float(_r['_mslp_mb']),
        time       = _r['time'],
    ))

_N       = len(track_points)
_mslps   = [p['mslp_mb'] for p in track_points]
_mslp_lo = min(_mslps)
_mslp_hi = max(_mslps)

print(f"  Forecast positions : {_N}")
print(f"  Lead-time range   : 0 – {track_points[-1]['lead_hours']:.0f} h")
print(f"  MSLP range        : {_mslp_lo:.1f} – {_mslp_hi:.1f} mb")

# ─────────────────────────────────────────────────────────────────────────────
# 2.  ASYMMETRIC INTENSITY-MODULATED IMPACT SWATH  (Part 1)
#
#     Width at each forecast point =
#       NHC track-uncertainty RADIUS        (per-side, NOT diameter)
#     + pressure-derived wind-extent buffer  (Knaff et al. 2013 empirical)
#
#     Northern Hemisphere right-of-track bias applied (Uhlhorn & Nolan 2012).
# ─────────────────────────────────────────────────────────────────────────────

_TIER_FRAC = [0.25, 0.55, 1.00]
_SW_CLR    = ['crimson', 'darkorange', 'gold']
_SW_ALP    = [0.55, 0.40, 0.25]
_SW_LBL    = ['Inner core (25 %)', 'Mid-level (55 %)', 'Outer extent (100 %)']

_left  = [[] for _ in range(3)]
_right = [[] for _ in range(3)]

for i, pt in enumerate(track_points):
    # ── NHC track-uncertainty RADIUS (= half-width of the cone) ──
    # These are per-side offsets, NOT diameters.  Previously the code
    # divided by 2 which halved the swath relative to the NHC cone.
    base_half_km = max(nhc_radius_km(pt['lead_hours']), 40.0)

    # ── Pressure-derived wind-extent buffer ──
    # The NHC cone shows where the *center* might go.  The actual hazard
    # area extends further by the radius of damaging winds.  Deeper
    # pressure → wider destructive wind field.
    # Empirical fit to Knaff et al. (2013) R34/ROCI relationships:
    #   dp=0  → 40 km  (minimal system)
    #   dp=30 → 100 km (Cat 1)
    #   dp=50 → 140 km (Cat 3)
    #   dp=70 → 180 km (Cat 4-5 cap)
    dp = max(1013.0 - pt['mslp_mb'], 0.0)
    wind_extent_km = 40.0 + 2.0 * min(dp, 70.0)

    # Total impact half-width = RSS composition of track uncertainty
    # and wind hazard extent.  These are independent error sources:
    #   • Track uncertainty = where the center could be (NHC cone)
    #   • Wind extent       = how far damaging winds reach from center
    # Linear addition (old) assumes worst-case co-alignment and over-
    # estimates the hazard envelope.  RSS (root-sum-of-squares) is the
    # standard approach for composing independent uncertainties and
    # better matches observed tropical-cyclone damage footprints
    # (Powell & Reinhold 2007; Knaff et al. 2013).
    impact_half_km = np.sqrt(base_half_km**2 + wind_extent_km**2)

    # ── Taper at the start: narrow the swath at early lead hours ──
    # Produces a pointed/narrow origin that smoothly widens, matching
    # the tapered appearance of the NHC cone of uncertainty.
    _TAPER_N = min(4, _N - 1)
    if i < _TAPER_N:
        # Smooth cosine ramp from 0.15 → 1.0 over first _TAPER_N points
        taper = 0.15 + 0.85 * (0.5 - 0.5 * np.cos(np.pi * i / _TAPER_N))
        impact_half_km *= taper

    # Motion vector (central difference where possible)
    if i == 0:
        dx = track_points[min(1, _N-1)]['lon'] - pt['lon']
        dy = track_points[min(1, _N-1)]['lat'] - pt['lat']
    elif i == _N - 1:
        dx = pt['lon'] - track_points[i - 1]['lon']
        dy = pt['lat'] - track_points[i - 1]['lat']
    else:
        dx = track_points[i + 1]['lon'] - track_points[i - 1]['lon']
        dy = track_points[i + 1]['lat'] - track_points[i - 1]['lat']

    mag = np.hypot(dx, dy) or 1e-9
    px, py = -dy / mag, dx / mag                                   # perpendicular (left-hand)

    half = km_to_degrees(impact_half_km, pt['lat'])

    # ── Right-of-track asymmetry ──
    # NH cyclones have stronger winds on the RIGHT side (looking along
    # motion) because storm translation adds to rotational winds there.
    # Ref: Uhlhorn & Nolan (2012), Knaff et al. (2013).
    if pt['lat'] >= 0:          # Northern Hemisphere
        right_factor = 1.25
        left_factor  = 0.80
    else:                       # Southern Hemisphere — mirror
        right_factor = 0.80
        left_factor  = 1.25

    for ti, frac in enumerate(_TIER_FRAC):
        left_off  = half * frac * left_factor
        right_off = half * frac * right_factor
        _left[ti].append(  (pt['lon'] + px * left_off,  pt['lat'] + py * left_off))
        _right[ti].append( (pt['lon'] - px * right_off, pt['lat'] - py * right_off))

swath_polys = []
for ti in range(3):
    # Build closed ring: left edge forward + right edge reversed + close
    ring = _left[ti] + _right[ti][::-1]
    if len(ring) >= 3:
        ring.append(ring[0])                                       # close the ring
        _poly = ShapelyPolygon(ring)
        if not _poly.is_valid:
            _poly = _poly.buffer(0)

        # ── Rounded end cap ──
        # Union a circle at the last forecast position so the swath
        # terminates with a smooth rounded shape (like the cone in 7.1a)
        # instead of a flat cut between the last left/right points.
        _last_pt = track_points[-1]
        _last_l  = _left[ti][-1]
        _last_r  = _right[ti][-1]
        _r_end   = max(
            np.hypot(_last_l[0] - _last_pt['lon'],
                     _last_l[1] - _last_pt['lat']),
            np.hypot(_last_r[0] - _last_pt['lon'],
                     _last_r[1] - _last_pt['lat']),
        ) * 1.00                                                   # exact fit, no oversize
        _end_cap = Point(_last_pt['lon'], _last_pt['lat']).buffer(
            _r_end, resolution=64)
        _poly = _poly.union(_end_cap)

        # Morphological smoothing: symmetric dilate-then-erode to close
        # jagged edges without adding net area (zero-net dilation)
        _poly = _poly.buffer(0.12).buffer(-0.12)
        swath_polys.append(_poly)
    else:
        swath_polys.append(None)

print("  ✓ 3-tier asymmetric swath built (NHC radius + wind extent + NH bias)")

# ─────────────────────────────────────────────────────────────────────────────
# 3.  COASTAL IMPACT ZONES  (Part 2)
# ─────────────────────────────────────────────────────────────────────────────

# Pre-compute rough bounding box for clipping land (performance)
_all_lons_fc = [p['lon'] for p in track_points]
_all_lats_fc = [p['lat'] for p in track_points]
_obs_lons = (observed_track['lon_converted'].tolist()
             if observed_track is not None else [])
_obs_lats = (observed_track['lat'].tolist()
             if observed_track is not None else [])

_rough_pad = 12
_clip_box  = shapely_box(
    min(_all_lons_fc + _obs_lons) - _rough_pad,
    min(_all_lats_fc + _obs_lats) - _rough_pad,
    max(_all_lons_fc + _obs_lons) + _rough_pad,
    max(_all_lats_fc + _obs_lats) + _rough_pad,
)

print("  Loading Natural Earth 50 m land (clipped to AOI)…")
_shp_path   = shpreader.natural_earth('50m', 'physical', 'land')
_land_full  = unary_union(list(shpreader.Reader(_shp_path).geometries()))
_land_clip  = _land_full.intersection(_clip_box)
if not _land_clip.is_valid:
    _land_clip = _land_clip.buffer(0)
_land_boundary = _land_clip.boundary
_land_prep     = prep(_land_clip)
print("  ✓ Land loaded & clipped")

_COAST_KM  = {'core': 200, 'surge': 350}
_CO_CLR    = {'core': 'crimson', 'surge': 'darkorange'}
_CO_ALP    = {'core': 0.50,     'surge': 0.40}
_CO_LBL    = {
    'core':  'Core (~200 km) — hurricane winds / surge',
    'surge': 'Surge (~350 km) — outer bands / rainfall',
}

tier_polys   = {}
coast_strips = {}

for tier, base_km in _COAST_KM.items():
    circles = []
    for pt in track_points:
        dp    = max(1013.0 - pt['mslp_mb'], 0.0)
        scale = 1.0 + 0.4 * min(dp, 75.0) / 75.0
        r_deg = km_to_degrees(base_km * scale, pt['lat'])
        circles.append(Point(pt['lon'], pt['lat']).buffer(r_deg, resolution=48))
    u = unary_union(circles)
    if not u.is_valid:
        u = u.buffer(0)
    tier_polys[tier] = u

    # Intersect with coastline, buffer into drawable strip
    ci = u.intersection(_land_boundary)
    if ci.is_empty:
        coast_strips[tier] = None
    else:
        bw = {'core': 0.18, 'surge': 0.12}[tier]
        coast_strips[tier] = ci.buffer(bw).intersection(u)

_active = sum(1 for v in coast_strips.values() if v and not v.is_empty)
print(f"  ✓ {_active}/2 coastal tiers overlap coastline")

# ─────────────────────────────────────────────────────────────────────────────
# 4.  DETECT LANDFALL CROSSINGS  (ocean → land transitions)
# ─────────────────────────────────────────────────────────────────────────────

_landfalls = []
for i in range(1, _N):
    prev = Point(track_points[i - 1]['lon'], track_points[i - 1]['lat'])
    curr = Point(track_points[i]['lon'],     track_points[i]['lat'])
    if not _land_prep.contains(prev) and _land_prep.contains(curr):
        _landfalls.append(i)

print(f"  Landfall crossings : {len(_landfalls)}")

# ─────────────────────────────────────────────────────────────────────────────
# 5.  COMPUTE PLOT EXTENT
# ─────────────────────────────────────────────────────────────────────────────

_bound_geoms = [g for g in [swath_polys[2], tier_polys.get('surge')]
                if g is not None and not g.is_empty]
if _bound_geoms:
    _bb = unary_union(_bound_geoms).bounds
else:
    _bb = (min(_all_lons_fc), min(_all_lats_fc),
           max(_all_lons_fc), max(_all_lats_fc))

_PAD = 3
_extent = [
    min(_bb[0], *_all_lons_fc, *(_obs_lons or [_bb[0]])) - _PAD,
    max(_bb[2], *_all_lons_fc, *(_obs_lons or [_bb[2]])) + _PAD,
    min(_bb[1], *_all_lats_fc, *(_obs_lats or [_bb[1]])) - _PAD,
    max(_bb[3], *_all_lats_fc, *(_obs_lats or [_bb[3]])) + _PAD,
]

# ─────────────────────────────────────────────────────────────────────────────
# 6.  COMPOSE THE FIGURE  (Part 3)
# ─────────────────────────────────────────────────────────────────────────────
ax.add_feature(cfeature.OCEAN, facecolor='#d6eaf8', zorder=0)
fig, ax = plt.subplots(figsize=(18, 13),
                       subplot_kw={'projection': ccrs.PlateCarree()})
try:
    ax.set_extent(_extent, crs=ccrs.PlateCarree())
except Exception:
    ax.set_xlim(_extent[0], _extent[1])
    ax.set_ylim(_extent[2], _extent[3])

# Base map
ax.add_feature(cfeature.OCEAN, facecolor='#d6eaf8', zorder=0)
ax.add_feature(cfeature.LAND,  facecolor='#f0f0f0', zorder=0)
ax.coastlines('50m', linewidth=0.7, color='#555', zorder=8)
ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.5,
               edgecolor='#777', zorder=8)
ax.add_feature(cfeature.STATES,  linewidth=0.3, edgecolor='#999', zorder=8)
gl = ax.gridlines(draw_labels=True, linewidth=0.3, color='gray',
                  alpha=0.4, linestyle='--')
gl.top_labels = gl.right_labels = False

# ── LAYER 1: Coastal impact tiers (outermost → innermost) ──
for _tk in ['surge', 'core']:
    _add_geom(ax, coast_strips.get(_tk), _CO_CLR[_tk], _CO_ALP[_tk], zorder=1)

# ── LAYER 2: Swath tiers (outermost → innermost) ──
for _ti in [2, 1, 0]:
    _add_geom(ax, swath_polys[_ti], _SW_CLR[_ti], _SW_ALP[_ti], zorder=2,
              ec=_SW_CLR[_ti], lw=0.8)

# ── LAYER 3: Observed track ──
if observed_track is not None:
    ax.plot(observed_track['lon_converted'], observed_track['lat'],
            color='black', linewidth=2.5, linestyle='-',
            transform=ccrs.PlateCarree(), zorder=3)
    ax.scatter(observed_track['lon_converted'], observed_track['lat'],
               c='black', s=25, edgecolors='white', linewidths=0.4,
               transform=ccrs.PlateCarree(), zorder=3)
# ── LAYER 4: Forecast track ──
_fc_lons = [p['lon'] for p in track_points]
_fc_lats = [p['lat'] for p in track_points]
ax.plot(_fc_lons, _fc_lats, color='royalblue', linewidth=2, linestyle='--',
        transform=ccrs.PlateCarree(), zorder=4)

# ── LAYER 5: Position markers coloured by SS category ──
for pt in track_points:
    _v = knaff_zehr(pt['mslp_mb'])
    _c = ss_cat(_v)
    ax.plot(pt['lon'], pt['lat'], 'o', color=_SS_CLR[_c], markersize=7,
            markeredgecolor='black', markeredgewidth=0.6,
            transform=ccrs.PlateCarree(), zorder=5)

# Start marker (lime star)
ax.plot(_fc_lons[0], _fc_lats[0], '*', color='lime', markersize=20,
        markeredgecolor='black', markeredgewidth=1,
        transform=ccrs.PlateCarree(), zorder=6)

# 24 h time labels (+24h, +48h, …)
for pt in track_points:
    if pt['lead_hours'] > 0 and pt['lead_hours'] % 24 == 0:
        ax.annotate(
            f"+{int(pt['lead_hours'])}h",
            xy=(pt['lon'], pt['lat']), xytext=(9, 9),
            textcoords='offset points', fontsize=8, fontweight='bold',
            color='royalblue',
            bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.85),
            transform=ccrs.PlateCarree(), zorder=7,
        )
# ── LAYER 6: Landfall markers & annotation boxes ──
for _li in _landfalls:
    _pt = track_points[_li]
    _v  = knaff_zehr(_pt['mslp_mb'])
    _c  = ss_cat(_v)
    ax.plot(_pt['lon'], _pt['lat'], '^', color='red', markersize=14,
            markeredgecolor='black', markeredgewidth=1.5,
            transform=ccrs.PlateCarree(), zorder=7)
    ax.annotate(
        f"LANDFALL  +{int(_pt['lead_hours'])}h\n"
        f"{_pt['mslp_mb']:.0f} mb  |  {_v:.0f} kt\n{_c}",
        xy=(_pt['lon'], _pt['lat']), xytext=(14, -22),
        textcoords='offset points', fontsize=8, fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.35', fc='#ffe0e0', ec='red',
                  alpha=0.92),
        arrowprops=dict(arrowstyle='->', color='red', lw=1.2),
        transform=ccrs.PlateCarree(), zorder=9,
    )

# ── TWO-COLUMN LEGEND ──
_h_swath = [mpatches.Patch(fc=_SW_CLR[i], alpha=_SW_ALP[i], label=_SW_LBL[i])
            for i in range(3)]
_h_coast = [mpatches.Patch(fc=_CO_CLR[k], alpha=_CO_ALP[k], label=_CO_LBL[k])
            for k in ['core', 'surge']]
_h_track = [
    Line2D([], [], color='black', lw=2.5, label='Observed track'),
    Line2D([], [], color='royalblue', lw=2, ls='--', label='Forecast track'),
    Line2D([], [], marker='*', ls='', mfc='lime', ms=12,
           mec='black', label='Forecast start'),
    Line2D([], [], marker='^', ls='', mfc='red', ms=10,
           mec='black', label='Landfall crossing'),
]

leg1 = ax.legend(handles=_h_swath + _h_coast, loc='upper left', fontsize=7.5,
                 framealpha=0.92, title='Impact Tiers', title_fontsize=8.5,
                 ncol=2)
ax.add_artist(leg1)
ax.legend(handles=_h_track, loc='lower left', fontsize=7.5,
          framealpha=0.92, title='Track Elements', title_fontsize=8.5)

# Title
_slbl = (f"Hurricane {selected_storm.name} ({selected_storm.year})"
         if hasattr(selected_storm, 'name') else "Storm")
ax.set_title(
    f"{_slbl}\n"
    "Asymmetric Intensity-Modulated Impact Swath  &  Coastal Impact Zones\n"
    f"Aurora forecast  {track_points[0]['time'].strftime('%Y-%m-%d %HZ')} → "
    f"+{track_points[-1]['lead_hours']:.0f} h   |   "
    f"MSLP {_mslp_lo:.0f}–{_mslp_hi:.0f} mb",
    fontsize=13, fontweight='bold', linespacing=1.4,
)

plt.tight_layout()
plt.show()

# ─────────────────────────────────────────────────────────────────────────────
# 7.  SUMMARY
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "=" * 70)
print("  IMPACT SWATH & COASTAL ZONE SUMMARY")
print("=" * 70)
for i, lbl in enumerate(_SW_LBL):
    area = swath_polys[i].area if swath_polys[i] else 0.0
    print(f"  Swath {lbl} : {area:.2f} sq-deg")
for tk in ['core', 'surge']:
    s = coast_strips.get(tk)
    tag = "coastline overlap ✓" if (s and not s.is_empty) else "no coastline overlap"
    print(f"  Coastal {_CO_LBL[tk]} : {tag}")

print(f"\n  Landfall crossings : {len(_landfalls)}")
for li in _landfalls:
    p = track_points[li]
    v = knaff_zehr(p['mslp_mb'])
    print(f"    +{p['lead_hours']:.0f} h   {p['mslp_mb']:.0f} mb   "
          f"{v:.0f} kt   {ss_cat(v)}   "
          f"({p['lat']:.2f}°N, {p['lon']:.2f}°)")
print("=" * 70)

### 7.2 Upload Track & Impact Swath data to MPC GeoCatalog

We will now upload the track and impact swath vector data to Geocatalog


In [ ]:
# ============================================================================
# Upload Track Data to MPC GeoCatalog
# ============================================================================
# This section creates:
# 1. A Cloud-Optimized GeoTIFF (COG) visualization of the hurricane track data
# 2. A STAC collection and item for the track data in GeoCatalog
# 
# The GeoTIFF contains a rendered map with:
# - Observed track (black line with dots)
# - Predicted/Forecast track (blue dashed line with dots)
# - Impact swath tiers (inner core / mid-level / outer extent)

# ============================================================================

import numpy as np
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend for saving figures
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as MplPolygon
from matplotlib.collections import PatchCollection
from matplotlib.lines import Line2D
import rasterio
from rasterio.transform import from_bounds
from rasterio.crs import CRS
import io
from PIL import Image
import os
from datetime import datetime, timezone

# Output directory for GeoTIFF
OUTPUTS_DIR = os.path.join(os.getcwd(), "outputs")
os.makedirs(OUTPUTS_DIR, exist_ok=True)

# Collection naming convention
collection_id = f"hurricane-{storm_name.lower()}-{storm_year}-track-data"
item_id = f"{storm_name.lower()}-{storm_year}-combined-track"

print(f"📊 Creating GeoTIFF Visualization for Hurricane {storm_name} ({storm_year})")
print(f"   Collection ID: {collection_id}")
print(f"   Item ID: {item_id}")
print("="*70)

# ============================================================================
# Step 1: Calculate bounds from all track data
# ============================================================================

# Gather all coordinates to determine the bounding box
all_lats = []
all_lons = []

# Add observed track coordinates
if 'observed_track' in dir() and observed_track is not None and len(observed_track) > 0:
    all_lats.extend(observed_track['lat'].tolist())
    all_lons.extend(observed_track['lon_converted'].tolist())
    print(f"✓ Observed track: {len(observed_track)} points")
elif 'selected_storm_df' in dir() and selected_storm_df is not None and len(selected_storm_df) > 0:
    # Use selected_storm_df if observed_track not available
    obs_track = selected_storm_df.copy()
    obs_track['lon_converted'] = obs_track['lon'].apply(lambda x: x - 360 if x > 180 else x)
    all_lats.extend(obs_track['lat'].tolist())
    all_lons.extend(obs_track['lon_converted'].tolist())
    print(f"✓ Observed track (from selected_storm_df): {len(obs_track)} points")
else:
    obs_track = None
    print("⚠️ No observed track data available")

# Add forecast track coordinates
if 'track' in dir() and track is not None and len(track) > 0:
    forecast_track = track.copy()
    if 'lon_converted' not in forecast_track.columns:
        forecast_track['lon_converted'] = forecast_track['lon'].apply(lambda x: x - 360 if x > 180 else x)
    all_lats.extend(forecast_track['lat'].tolist())
    all_lons.extend(forecast_track['lon_converted'].tolist())
    print(f"✓ Forecast track: {len(forecast_track)} points")

# Add swath polygon bounds (swath_polys from Section 7.1b)
if 'swath_polys' in dir() and swath_polys is not None:
    for _sp_idx, _sp in enumerate(swath_polys):
        if _sp is not None and not _sp.is_empty:
            _sp_bounds = _sp.bounds  # (minx, miny, maxx, maxy)
            all_lons.extend([_sp_bounds[0], _sp_bounds[2]])
            all_lats.extend([_sp_bounds[1], _sp_bounds[3]])
    _n_valid = sum(1 for sp in swath_polys if sp and not sp.is_empty)
    print(f"✓ Impact swath: {_n_valid} tier(s)")


# Calculate bounds with padding
padding = 2.0  # degrees
lon_min = min(all_lons) - padding
lon_max = max(all_lons) + padding
lat_min = min(all_lats) - padding
lat_max = max(all_lats) + padding

print(f"\n📍 Bounding Box (with {padding}° padding):")
print(f"   Latitude:  {lat_min:.2f}° to {lat_max:.2f}°")
print(f"   Longitude: {lon_min:.2f}° to {lon_max:.2f}°")

# ============================================================================
# Step 2: Create the map visualization using matplotlib
# ============================================================================

# High resolution for the GeoTIFF (adjust as needed)
DPI = 150
fig_width = 16  # inches
fig_height = 12  # inches

# Calculate pixel dimensions
width_px = int(fig_width * DPI)
height_px = int(fig_height * DPI)

print(f"\n🎨 Creating visualization...")
print(f"   Figure size: {fig_width}\" x {fig_height}\" at {DPI} DPI")
print(f"   Output resolution: {width_px} x {height_px} pixels")

# Create figure with exact pixel dimensions
fig, ax = plt.subplots(figsize=(fig_width, fig_height), dpi=DPI)

# Set axis limits to match our bounds
ax.set_xlim(lon_min, lon_max)
ax.set_ylim(lat_min, lat_max)

# Remove axes, ticks, and labels for clean GeoTIFF
ax.set_axis_off()

# Set background to light blue (ocean)
ax.set_facecolor('#E6F3FF')

# =============================================================================
# Draw the impact swath tiers (bottom layer — outermost to innermost)
# =============================================================================
_SW_CLR_T = ['crimson', 'darkorange', 'gold']
_SW_ALP_T = [0.55, 0.40, 0.25]
_SW_LBL_T = ['Inner core (25 %)', 'Mid-level (55 %)', 'Outer extent (100 %)']

if 'swath_polys' in dir() and swath_polys is not None:
    for _ti in [2, 1, 0]:  # outermost → innermost so inner layers draw on top
        if _ti < len(swath_polys) and swath_polys[_ti] is not None and not swath_polys[_ti].is_empty:
            _geom = swath_polys[_ti]
            _polys_to_draw = []
            if _geom.geom_type == 'Polygon':
                _polys_to_draw = [_geom]
            elif _geom.geom_type == 'MultiPolygon':
                _polys_to_draw = list(_geom.geoms)

            for _pg in _polys_to_draw:
                _patch = MplPolygon(
                    list(_pg.exterior.coords),
                    facecolor=_SW_CLR_T[_ti],
                    edgecolor=_SW_CLR_T[_ti],
                    linewidth=0.8,
                    alpha=_SW_ALP_T[_ti],
                    zorder=1
                )
                ax.add_patch(_patch)
    _n_drawn = sum(1 for sp in swath_polys if sp and not sp.is_empty)
    print(f"   ✓ Impact swath drawn ({_n_drawn} tier(s))")


# ============================================================================
# Draw the observed track (middle layer)
# ============================================================================
obs_track_data = None
if 'observed_track' in dir() and observed_track is not None and len(observed_track) > 0:
    obs_track_data = observed_track
elif 'selected_storm_df' in dir() and selected_storm_df is not None and len(selected_storm_df) > 0:
    obs_track_data = selected_storm_df.copy()
    obs_track_data['lon_converted'] = obs_track_data['lon'].apply(lambda x: x - 360 if x > 180 else x)

if obs_track_data is not None:
    # Draw observed track line
    ax.plot(
        obs_track_data['lon_converted'],
        obs_track_data['lat'],
        color='black',
        linewidth=3,
        linestyle='-',
        zorder=2,
        label='Observed Track'
    )
    
    # Draw observed track points
    ax.scatter(
        obs_track_data['lon_converted'],
        obs_track_data['lat'],
        c='black',
        s=40,
        edgecolors='white',
        linewidths=0.5,
        zorder=3
    )
    print("   ✓ Observed track drawn")

# ============================================================================
# Draw the forecast track (top layer)
# ============================================================================
if 'track' in dir() and track is not None and len(track) > 0:
    forecast_data = track.copy()
    if 'lon_converted' not in forecast_data.columns:
        forecast_data['lon_converted'] = forecast_data['lon'].apply(lambda x: x - 360 if x > 180 else x)
    
    # Draw forecast track line
    ax.plot(
        forecast_data['lon_converted'],
        forecast_data['lat'],
        color='#1E90FF',  # Dodger blue
        linewidth=3,
        linestyle='--',
        zorder=4,
        label='Forecast Track'
    )
    
    # Draw forecast track points
    ax.scatter(
        forecast_data['lon_converted'],
        forecast_data['lat'],
        c='#1E90FF',
        s=50,
        edgecolors='white',
        linewidths=1,
        zorder=5,
        marker='o'
    )
    print("   ✓ Forecast track drawn")

# ============================================================================
# Add legend
# ============================================================================
legend_elements = [
    Line2D([0], [0], color='black', linewidth=3, linestyle='-', label='Observed Track'),
    Line2D([0], [0], color='#1E90FF', linewidth=3, linestyle='--', label='Forecast Track'),
]
# Add swath tier legend patches
for _li in range(3):
    legend_elements.append(
        MplPolygon([(0, 0)], facecolor=_SW_CLR_T[_li], edgecolor=_SW_CLR_T[_li],
                   alpha=_SW_ALP_T[_li], label=_SW_LBL_T[_li])
    )
ax.legend(handles=legend_elements, loc='upper left', fontsize=11, framealpha=0.9)

# Add title
ax.set_title(
    f"Hurricane {storm_name} ({storm_year}) - Track & Impact Swath",
    fontsize=16,
    fontweight='bold',
    pad=10
)


# Tight layout to minimize whitespace
plt.tight_layout()

# ============================================================================
# Step 3: Save figure to PNG buffer
# ============================================================================
print("\n💾 Saving visualization...")

# Save to buffer as PNG
png_buffer = io.BytesIO()
fig.savefig(png_buffer, format='png', dpi=DPI, bbox_inches='tight', pad_inches=0.1)
png_buffer.seek(0)

# Load PNG into PIL for processing
img = Image.open(png_buffer)
img_array = np.array(img)

print(f"   PNG dimensions: {img_array.shape}")

# ============================================================================
# Step 4: Create GeoTIFF with georeference information
# ============================================================================
geotiff_filename = f"hurricane_{storm_name.lower()}_{storm_year}_track.tif"
geotiff_path = os.path.join(OUTPUTS_DIR, geotiff_filename)

# Calculate transform from bounds
# The transform maps pixel coordinates to geographic coordinates
height, width = img_array.shape[:2]
transform = from_bounds(lon_min, lat_min, lon_max, lat_max, width, height)

# Handle different channel configurations
if len(img_array.shape) == 2:
    # Grayscale
    count = 1
    data = img_array[np.newaxis, :, :]
elif img_array.shape[2] == 3:
    # RGB
    count = 3
    data = np.transpose(img_array, (2, 0, 1))
elif img_array.shape[2] == 4:
    # RGBA
    count = 4
    data = np.transpose(img_array, (2, 0, 1))
else:
    raise ValueError(f"Unexpected image shape: {img_array.shape}")

# Create GeoTIFF
with rasterio.open(
    geotiff_path,
    'w',
    driver='GTiff',
    height=height,
    width=width,
    count=count,
    dtype=data.dtype,
    crs=CRS.from_epsg(4326),  # WGS84
    transform=transform,
    compress='lzw'  # Compression for smaller file size
) as dst:
    dst.write(data)

file_size_mb = os.path.getsize(geotiff_path) / 1024 / 1024
print(f"   ✓ GeoTIFF saved: {geotiff_path}")
print(f"   ✓ File size: {file_size_mb:.2f} MB")
print(f"   ✓ CRS: EPSG:4326 (WGS84)")
print(f"   ✓ Bounds: [{lon_min:.4f}, {lat_min:.4f}, {lon_max:.4f}, {lat_max:.4f}]")

# Clean up
plt.close(fig)
png_buffer.close()

print("\n✅ GeoTIFF visualization created successfully!")

In [ ]:
# Fallback: ensure URL variables are defined (empty string if earlier cells didn't run)
explorer_url = locals().get('explorer_url', '')
forecast_explorer_url = locals().get('forecast_explorer_url', '')
track_explorer_url = locals().get('track_explorer_url', '')
infra_explorer_url_safe = locals().get('infra_explorer_url_safe', locals().get('infra_explorer_url', ''))

# ============================================================================
# Upload Track Data to MPC GeoCatalog
# ============================================================================
# This cell creates:
# 1. A GeoTIFF STAC item for visualization (raster)
# 2. A GeoJSON STAC item for vector track data
# 3. Configures render options for both items
# ============================================================================

import aiohttp
import pystac
from pathlib import Path
from datetime import datetime, timezone
import json
from shapely.geometry import LineString, mapping

# Collection configuration for Track Data
TRACK_COLLECTION = f"hurricane-{storm_name.lower()}-{storm_year}-track-data"
GEOTIFF_ITEM_ID = f"{storm_name.lower()}-{storm_year}-track-visualization"
GEOJSON_ITEM_ID = f"{storm_name.lower()}-{storm_year}-track-vectors"

print("="*70)
print("🚀 UPLOADING TRACK DATA TO MPC GEOCATALOG")
print("="*70)
print(f"\nCollection: {TRACK_COLLECTION}")
print(f"GeoTIFF Item: {GEOTIFF_ITEM_ID}")
print(f"GeoJSON Item: {GEOJSON_ITEM_ID}")

# ============================================================================
# Step 1: Upload GeoTIFF to Blob Storage
# ============================================================================

geotiff_blob_name = f"track-data/{GEOTIFF_ITEM_ID}.tif"
print(f"\n📤 Uploading GeoTIFF to blob storage...")
print(f"   Blob name: {geotiff_blob_name}")

geotiff_blob_url, geotiff_sas_url = await upload_to_blob_storage(
    Path(geotiff_path), 
    geotiff_blob_name
)
print(f"   ✅ Uploaded: {geotiff_blob_url}")

# ============================================================================
# Step 2: Create GeoJSON with track vectors and upload
# ============================================================================

print(f"\n📤 Creating and uploading GeoJSON...")

# Build GeoJSON FeatureCollection with all track layers
geojson_features = []

# Feature 1: Observed Track (LineString)
if 'observed_track' in dir() and observed_track is not None and len(observed_track) > 0:
    obs_coords = list(zip(
        observed_track['lon_converted'].tolist(),
        observed_track['lat'].tolist()
    ))
    obs_line = LineString(obs_coords)
    geojson_features.append({
        "type": "Feature",
        "properties": {
            "layer": "observed_track",
            "name": f"Hurricane {storm_name} Observed Track",
            "color": "#000000",
            "stroke_width": 3,
            "points": len(observed_track),
            "start_time": observed_track['time'].iloc[0].strftime("%Y-%m-%dT%H:%M:%SZ"),
            "end_time": observed_track['time'].iloc[-1].strftime("%Y-%m-%dT%H:%M:%SZ"),
            "max_wind_kt": float(observed_track['vmax'].max()) if 'vmax' in observed_track.columns else None,
            "min_pressure_mb": float(observed_track['mslp'].min()) if 'mslp' in observed_track.columns else None
        },
        "geometry": mapping(obs_line)
    })
    print(f"   ✓ Added observed track: {len(obs_coords)} points")

# Feature 2: Forecast Track (LineString)
if 'track' in dir() and track is not None and len(track) > 0:
    forecast_coords = list(zip(
        track['lon_converted'].tolist(),
        track['lat'].tolist()
    ))
    forecast_line = LineString(forecast_coords)
    geojson_features.append({
        "type": "Feature",
        "properties": {
            "layer": "forecast_track",
            "name": f"Hurricane {storm_name} Aurora Forecast",
            "color": "#0000FF",
            "stroke_width": 2.5,
            "stroke_dasharray": "5,5",
            "points": len(track),
            "start_time": track['time'].iloc[0].strftime("%Y-%m-%dT%H:%M:%SZ"),
            "end_time": track['time'].iloc[-1].strftime("%Y-%m-%dT%H:%M:%SZ"),
            "max_wind_kt": float(track['wind'].max()) if 'wind' in track.columns else None,
            "min_pressure_mb": float(track['msl'].min() / 100) if 'msl' in track.columns else None
        },
        "geometry": mapping(forecast_line)
    })
    print(f"   ✓ Added forecast track: {len(forecast_coords)} points")

# Feature 3+: Impact Swath Tiers (Polygons)
_swath_colors = ['#DC143C', '#FF8C00', '#FFD700']   # crimson, darkorange, gold
_swath_opacities = [0.55, 0.40, 0.25]
_swath_labels = ['Inner core (25 %)', 'Mid-level (55 %)', 'Outer extent (100 %)']

if 'swath_polys' in dir() and swath_polys is not None:
    for ti in range(len(swath_polys)):
        sp = swath_polys[ti]
        if sp is not None and not sp.is_empty:
            geojson_features.append({
                "type": "Feature",
                "properties": {
                    "layer": f"impact_swath_tier_{ti}",
                    "name": f"Hurricane {storm_name} Impact Swath - {_swath_labels[ti]}",
                    "fill_color": _swath_colors[ti],
                    "fill_opacity": _swath_opacities[ti],
                    "stroke_color": _swath_colors[ti],
                    "stroke_width": 0.8,
                    "tier": ti,
                    "tier_label": _swath_labels[ti],
                    "area_sq_deg": sp.area
                },
                "geometry": mapping(sp)
            })
    _n_swath = sum(1 for sp in swath_polys if sp and not sp.is_empty)
    print(f"   ✓ Added {_n_swath} impact swath tier(s)")


# Create FeatureCollection
geojson_data = {
    "type": "FeatureCollection",
    "features": geojson_features,
    "properties": {
        "storm_name": storm_name,
        "storm_year": storm_year,
        "created": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
    }
}

# Save GeoJSON locally
geojson_path = os.path.join(OUTPUTS_DIR, f"{GEOJSON_ITEM_ID}.geojson")
with open(geojson_path, 'w') as f:
    json.dump(geojson_data, f, indent=2)

geojson_size_kb = os.path.getsize(geojson_path) / 1024
print(f"   ✓ GeoJSON saved: {geojson_path} ({geojson_size_kb:.1f} KB)")

# Upload GeoJSON to blob storage
geojson_blob_name = f"track-data/{GEOJSON_ITEM_ID}.geojson"
geojson_blob_url, geojson_sas_url = await upload_to_blob_storage(
    Path(geojson_path),
    geojson_blob_name
)
print(f"   ✅ Uploaded: {geojson_blob_url}")

# ============================================================================
# Step 3: Ensure Track Data Collection exists (create if not)
# ============================================================================

async def ensure_track_collection_exists_v2(collection_id: str):
    """Create track data collection if it doesn't exist."""
    print(f"\n📂 Checking collection '{collection_id}'...")
    
    headers = get_bearer_token()
    params = {"api-version": API_VERSION}
    
    timeout = aiohttp.ClientTimeout(total=300)
    async with aiohttp.ClientSession(timeout=timeout) as session:
        collection_url = join_url(STAC_API_URL, "collections", collection_id)
        
        # Check if collection exists
        async with session.get(collection_url, headers=headers, params=params) as response:
            if response.status == 200:
                print(f"   ✓ Collection already exists, will add items to it")
                return  # Collection exists, just use it
        
        # Collection doesn't exist, create it
        print(f"   Creating new collection...")
        
        # Get track time bounds
        track_start = storm_init_time
        track_end = storm_end_time
        
        if 'observed_track' in dir() and observed_track is not None and len(observed_track) > 0:
            track_start = observed_track['time'].min()
            track_end = observed_track['time'].max()
        if 'track' in dir() and track is not None and len(track) > 0:
            track_end = max(track_end, track['time'].max())
        
        track_start_str = track_start.strftime("%Y-%m-%dT%H:%M:%SZ")
        track_end_str = track_end.strftime("%Y-%m-%dT%H:%M:%SZ")
        
        # Define item_assets for the collection
        item_assets = {
            "visual": {
                "type": "image/tiff; application=geotiff; profile=cloud-optimized",
                "roles": ["data", "visual"],
                "title": "Track Visualization (COG)",
                "description": "Cloud-Optimized GeoTIFF visualization of hurricane track"
            },
            "data": {
                "type": "application/geo+json",
                "roles": ["data"],
                "title": "Track Vector Data (GeoJSON)",
                "description": "Vector track data including observed track, forecast track, and impact swath tiers"

            }
        }
        
        collection_dict = {
            "id": collection_id,
            "type": "Collection",
            "stac_version": "1.0.0",
            "title": f"Hurricane {storm_name} {storm_year} - Track Data",
            "description": (
                f"Track data for Hurricane {storm_name} ({storm_year}). "
                f"Contains observed track, Aurora AI forecast track, and impact swath tiers."
            ),

            "license": "proprietary",
            "keywords": ["hurricane", "tropical cyclone", "forecast", "track", storm_name.lower(), str(storm_year)],
            "extent": {
                "spatial": {"bbox": [[lon_min, lat_min, lon_max, lat_max]]},
                "temporal": {"interval": [[track_start_str, track_end_str]]}
            },
            "item_assets": item_assets,
            "links": []
        }
        
        collections_url = join_url(STAC_API_URL, "collections")
        
        async with session.post(
            collections_url,
            json=collection_dict,
            headers={**headers, "Content-Type": "application/json"},
            params=params
        ) as response:
            if response.status >= 400:
                error_text = await response.text()
                print(f"   ❌ Failed: {response.status} - {error_text}")
                raise Exception(f"Failed to create collection: {response.status}")
            
            print(f"   ✓ Collection created (status {response.status})")
            
            if response.status == 202:
                print(f"   ⏳ Waiting for availability...")
                for attempt in range(10):
                    await asyncio.sleep(3)
                    async with session.get(collection_url, headers=headers, params=params) as check:
                        if check.status == 200:
                            print(f"   ✓ Collection available")
                            break

import asyncio
await ensure_track_collection_exists_v2(TRACK_COLLECTION)

# ============================================================================
# Step 4: Create and Ingest GeoTIFF STAC Item
# ============================================================================

print(f"\n📝 Creating GeoTIFF STAC item...")

# Determine the time to use for the item
item_datetime = storm_init_time
if 'observed_track' in dir() and observed_track is not None and len(observed_track) > 0:
    item_datetime = observed_track['time'].iloc[0]

# Create the GeoTIFF STAC item
geotiff_item = pystac.Item(
    id=GEOTIFF_ITEM_ID,
    geometry={
        "type": "Polygon",
        "coordinates": [[
            [lon_min, lat_min],
            [lon_max, lat_min],
            [lon_max, lat_max],
            [lon_min, lat_max],
            [lon_min, lat_min]
        ]]
    },
    bbox=[lon_min, lat_min, lon_max, lat_max],
    datetime=item_datetime.to_pydatetime() if hasattr(item_datetime, 'to_pydatetime') else item_datetime,
    properties={
        "title": f"Hurricane {storm_name} {storm_year} Track Visualization",
        "description": f"Combined observed and forecast track visualization for Hurricane {storm_name}",
        "storm_name": storm_name,
        "storm_year": storm_year,
        "content_type": "image/tiff"
    }
)

# Add asset
geotiff_item.add_asset(
    "visual",
    pystac.Asset(
        href=geotiff_sas_url,
        media_type="image/tiff; application=geotiff; profile=cloud-optimized",
        roles=["data", "visual"],
        title="Track Visualization"
    )
)

# Ingest the item
print(f"   Ingesting item '{GEOTIFF_ITEM_ID}'...")

async def ingest_track_item(collection_id: str, item: pystac.Item):
    """Ingest a track item into the collection."""
    headers = get_bearer_token()
    params = {"api-version": API_VERSION}
    
    item_dict = item.to_dict()
    item_dict["collection"] = collection_id
    
    # Add required collection link (STAC spec requires this when collection property is set)
    collection_link = {
        "rel": "collection",
        "href": join_url(STAC_API_URL, "collections", collection_id),
        "type": "application/json"
    }
    
    # Add parent link as well
    parent_link = {
        "rel": "parent",
        "href": join_url(STAC_API_URL, "collections", collection_id),
        "type": "application/json"
    }
    
    # Ensure links array exists and add required links
    if "links" not in item_dict:
        item_dict["links"] = []
    
    # Remove any existing collection/parent links to avoid duplicates
    item_dict["links"] = [l for l in item_dict["links"] if l.get("rel") not in ["collection", "parent"]]
    
    # Add the required links
    item_dict["links"].append(collection_link)
    item_dict["links"].append(parent_link)
    
    timeout = aiohttp.ClientTimeout(total=300)
    async with aiohttp.ClientSession(timeout=timeout) as session:
        items_url = join_url(STAC_API_URL, "collections", collection_id, "items")
        
        async with session.post(
            items_url,
            json=item_dict,
            headers={**headers, "Content-Type": "application/json"},
            params=params
        ) as response:
            if response.status >= 400:
                error_text = await response.text()
                # Check if item already exists
                if response.status == 409 or "already exists" in error_text.lower():
                    print(f"   ℹ️  Item already exists, updating...")
                    item_url = join_url(items_url, item.id)
                    async with session.put(
                        item_url,
                        json=item_dict,
                        headers={**headers, "Content-Type": "application/json"},
                        params=params
                    ) as update_response:
                        if update_response.status >= 400:
                            update_error = await update_response.text()
                            print(f"   ⚠️  Update failed: {update_response.status} - {update_error}")
                        else:
                            print(f"   ✓ Item updated")
                else:
                    print(f"   ❌ Failed: {response.status} - {error_text}")
            else:
                print(f"   ✓ Item ingested (status {response.status})")

await ingest_track_item(TRACK_COLLECTION, geotiff_item)

# ============================================================================
# Step 5: Create and Ingest GeoJSON STAC Item
# ============================================================================

print(f"\n📝 Creating GeoJSON STAC item...")

# Create the GeoJSON STAC item
geojson_item = pystac.Item(
    id=GEOJSON_ITEM_ID,
    geometry={
        "type": "Polygon",
        "coordinates": [[
            [lon_min, lat_min],
            [lon_max, lat_min],
            [lon_max, lat_max],
            [lon_min, lat_max],
            [lon_min, lat_min]
        ]]
    },
    bbox=[lon_min, lat_min, lon_max, lat_max],
    datetime=item_datetime.to_pydatetime() if hasattr(item_datetime, 'to_pydatetime') else item_datetime,
    properties={
        "title": f"Hurricane {storm_name} {storm_year} Track Vectors",
        "description": f"Vector track data for Hurricane {storm_name} including observed track, forecast, and impact swath",

        "storm_name": storm_name,
        "storm_year": storm_year,
        "content_type": "application/geo+json",
        "feature_count": len(geojson_features)
    }
)

# Add asset
geojson_item.add_asset(
    "data",
    pystac.Asset(
        href=geojson_sas_url,
        media_type="application/geo+json",
        roles=["data"],
        title="Track Vector Data"
    )
)

# Ingest the item
print(f"   Ingesting item '{GEOJSON_ITEM_ID}'...")
await ingest_track_item(TRACK_COLLECTION, geojson_item)

# ============================================================================
# Summary
# ============================================================================

print(f"\n" + "="*70)
print(f"✅ TRACK DATA UPLOAD COMPLETE")
print(f"="*70)
print(f"\n📊 Collection: {TRACK_COLLECTION}")
print(f"\n📁 Items created:")
print(f"   1. {GEOTIFF_ITEM_ID} (GeoTIFF visualization)")
print(f"   2. {GEOJSON_ITEM_ID} (GeoJSON vectors)")

# Build explorer URL
track_explorer_url = f"{GEOCATALOG_URI}/explorer?d={TRACK_COLLECTION}"
print(f"\n🌐 View in GeoCatalog Explorer:")
print(f"   {track_explorer_url}")


## 8. Infrastructure Impact Analysis

Analyze power infrastructure within the storm's **asymmetric impact swath** and **coastal impact zones** using OpenStreetMap data via the Overpass API. This section uses the intensity-modulated swath and coastal flood/surge tiers computed in Section 7.1b — *not* the earlier cone of uncertainty.

1. Builds a unified impact zone polygon from the 3-tier swath and coastal flood tiers
2. Queries OpenStreetMap for power infrastructure (transmission lines, substations, power plants)
3. Filters infrastructure to those intersecting the impact zone
4. Generates an interactive Folium map with swath visualization, coastal zones, and categorized infrastructure layers

**Data Sources**
- **Section 7.1b** — Asymmetric impact swath polygons (`swath_polys`) and coastal zone polygons (`tier_polys`, `coast_strips`)
- **OpenStreetMap** via Overpass API — Power infrastructure data
- **OpenInfraMap** tile overlay — Visual reference for power grid

### 8.1 Install Infrastructure Analysis Packages

In [ ]:
# Install required packages for OpenStreetMap infrastructure data
import subprocess
import sys

# Install packages if not already installed
packages = ['osmnx', 'folium', 'geopandas']
for pkg in packages:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
        
print("✓ Package installation check complete")

In [ ]:
# import libraries
import osmnx as ox
import geopandas as gpd
import folium
from folium.plugins import MarkerCluster
from shapely.geometry import Point, LineString, Polygon, box
from shapely.ops import unary_union
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Configure osmnx
ox.settings.use_cache = True
ox.settings.log_console = False

print("✓ Libraries loaded successfully")

### 8.2 Create Storm Cone of Uncertainty

Generate a smooth polygon representing the forecast uncertainty region using Shapely's buffer and union operations.

In [ ]:
# ==============================================================================
# Build unified impact zone from Section 7.1b swath & coastal tiers
# ==============================================================================
# swath_polys[0..2] = inner / mid / outer swath tiers  (from cell 50)
# coast_strips      = coastline-intersected tier strips (for rendering & filtering)
#
# The IMPACT ZONE for infrastructure queries is the union of:
#   - The swath polygons (forecast uncertainty cone)
#   - The coastal strip overlaps (coast_strips values)
# This captures the direct-path hazard area AND the actual coastal
# impact zones, without extending hundreds of km beyond the swath.
# ==============================================================================

from shapely.geometry import LineString, Point, Polygon
from shapely.ops import unary_union

# -- Verify that Section 7.1b outputs are available --
assert 'swath_polys' in dir() and swath_polys is not None, \
    "Run Section 7.1b first - swath_polys not found"
assert 'coast_strips' in dir() and coast_strips is not None, \
    "Run Section 7.1b first - coast_strips not found"

# -- Collect non-empty geometries --
_zone_parts = []

# Include swath polygons (the forecast uncertainty cone)
for sp in swath_polys:
    if sp is not None and not sp.is_empty:
        _zone_parts.append(sp)

# Include coast_strips (coastline-intersected tiers) instead of
# tier_polys (which are huge 200-750 km radius circles)
for key in ('surge', 'core'):
    cs = coast_strips.get(key)
    if cs is not None and not cs.is_empty:
        _zone_parts.append(cs)

assert _zone_parts, "No valid swath or coastal-strip polygons available"

# -- Union into a single impact-zone polygon --
impact_zone = unary_union(_zone_parts)
if not impact_zone.is_valid:
    impact_zone = impact_zone.buffer(0)

# -- Bounding box for Overpass queries --
iz_bounds = impact_zone.bounds  # (minx, miny, maxx, maxy)
buffer_deg = 0.5
lat_min_infra = iz_bounds[1] - buffer_deg
lat_max_infra = iz_bounds[3] + buffer_deg
lon_min_infra = iz_bounds[0] - buffer_deg
lon_max_infra = iz_bounds[2] + buffer_deg

bbox = (lon_min_infra, lat_min_infra, lon_max_infra, lat_max_infra)

print("Impact Zone (swath + coastal strips):")
print(f"  Geometry type : {impact_zone.geom_type}")
print(f"  Area          : {impact_zone.area:.4f} sq degrees")
print(f"  Components    : {len(_zone_parts)} polygons merged")
print(f"\nQuery Bounds (with {buffer_deg}° buffer):")
print(f"  Latitude  : {lat_min_infra:.2f}° to {lat_max_infra:.2f}°")
print(f"  Longitude : {lon_min_infra:.2f}° to {lon_max_infra:.2f}°")


### 8.3 Query Power Infrastructure from OpenStreetMap

Download substations, power plants, and transmission lines from OpenStreetMap using tiled queries to avoid API timeouts. Results are filtered to the **impact zone** (swath + coastal tiers).

#### 8.3.1 Configure Overpass API

In [ ]:
# Setup: Overpass API helper functions
import requests
import pandas as pd
from datetime import datetime
import time
import hashlib

print(f"Query bounds: {lat_min_infra:.2f}°N to {lat_max_infra:.2f}°N, {lon_min_infra:.2f}°W to {lon_max_infra:.2f}°W")

# Overpass API servers (with fallback)
OVERPASS_SERVERS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://maps.mail.ru/osm/tools/overpass/api/interpreter"
]

# Infrastructure cache directory
INFRA_CACHE_DIR = os.path.join(os.getcwd(), "cache", "infra_data")
os.makedirs(INFRA_CACHE_DIR, exist_ok=True)

def get_query_cache_path(query, cache_name=None):
    """Generate cache file path for an Overpass query.
    
    Args:
        query: The Overpass API query string (used for hash fallback)
        cache_name: Optional human-readable name for the cache file.
                    If provided, creates filename like: substations_25.0_-90.0_35.0_-80.0.json
                    If None, falls back to SHA1 hash of query.
    """
    if cache_name:
        # Sanitize the cache name (replace invalid chars)
        safe_name = cache_name.replace(" ", "_").replace("/", "_").replace("\\", "_")
        return os.path.join(INFRA_CACHE_DIR, f"{safe_name}.json")
    else:
        # Fallback to hash-based naming
        query_hash = hashlib.sha1(query.encode()).hexdigest()
        return os.path.join(INFRA_CACHE_DIR, f"{query_hash}.json")

def query_overpass(query, timeout=180, use_cache=True, cache_name=None):
    """Execute Overpass query with fallback servers and local caching.
    
    Args:
        query: The Overpass API query string
        timeout: Request timeout in seconds
        use_cache: If True, check cache first and save results to cache
        cache_name: Optional human-readable name for cache file (e.g., 'substations_25.0_-90.0_35.0_-80.0')
    """
    # Check cache first
    cache_path = get_query_cache_path(query, cache_name)
    if use_cache and os.path.exists(cache_path):
        try:
            with open(cache_path, 'r') as f:
                cache_filename = os.path.basename(cache_path)
                print(f"  ✓ Loaded from cache: {cache_filename}")
                return json.load(f)
        except Exception as e:
            print(f"  ⚠ Cache read error: {e}")
    
    # Query from API
    for server in OVERPASS_SERVERS:
        try:
            print(f"  Trying {server.split('/')[2]}...")
            response = requests.post(server, data={'data': query}, timeout=timeout)
            if response.status_code == 200:
                print(f"  ✓ Success!")
                result = response.json()
                
                # Save to cache
                if use_cache:
                    try:
                        with open(cache_path, 'w') as f:
                            json.dump(result, f)
                        cache_filename = os.path.basename(cache_path)
                        print(f"  💾 Cached: {cache_filename}")
                    except Exception as e:
                        print(f"  ⚠ Cache write error: {e}")
                
                return result
            elif response.status_code == 429:
                print(f"  Rate limited, trying next server...")
                time.sleep(2)
            else:
                print(f"  HTTP {response.status_code}, trying next...")
        except requests.exceptions.Timeout:
            print(f"  Timeout, trying next server...")
        except Exception as e:
            print(f"  Error: {str(e)[:60]}...")
            time.sleep(1)
    return None

def categorize_voltage(voltage_str):
    """Categorize voltage into high/medium/low"""
    if voltage_str == 'Unknown':
        return 'unknown', 0
    try:
        v = int(str(voltage_str).replace(' ', '').replace('kV', '000').split(';')[0])
        if v >= 345000:
            return 'high', v
        elif v >= 115000:
            return 'medium-high', v
        elif v >= 69000:
            return 'medium', v
        else:
            return 'low', v
    except:
        return 'unknown', 0

print("✓ Helper functions defined")

#### 8.3.2 Download Substations and Power Plants

In [ ]:
# Download Substations and Power Plants (split into tiles for reliability)
print(f"[{datetime.now().strftime('%H:%M:%S')}] Downloading substations and power plants...")
print(f"Full query area: {lat_min_infra:.2f}°N to {lat_max_infra:.2f}°N, {lon_min_infra:.2f}° to {lon_max_infra:.2f}°")

# Split large area into smaller tiles (5x5 degree tiles)
TILE_SIZE = 5.0  # degrees

def get_tiles(south, west, north, east, tile_size=5.0):
    """Split bounding box into smaller tiles"""
    tiles = []
    lat = south
    while lat < north:
        lon = west
        while lon < east:
            tile_north = min(lat + tile_size, north)
            tile_east = min(lon + tile_size, east)
            tiles.append((lat, lon, tile_north, tile_east))
            lon += tile_size
        lat += tile_size
    return tiles

tiles = get_tiles(lat_min_infra, lon_min_infra, lat_max_infra, lon_max_infra, TILE_SIZE)
print(f"Split into {len(tiles)} tiles of {TILE_SIZE}° each")

all_substations = []
all_plants = []
seen_ids = set()  # Avoid duplicates

for i, (tile_south, tile_west, tile_north, tile_east) in enumerate(tiles):
    print(f"\n  Tile {i+1}/{len(tiles)}: ({tile_south:.1f},{tile_west:.1f}) to ({tile_north:.1f},{tile_east:.1f})")
    
    query = f"""
    [out:json][timeout:60];
    (
      node["power"="substation"]({tile_south},{tile_west},{tile_north},{tile_east});
      way["power"="substation"]({tile_south},{tile_west},{tile_north},{tile_east});
      node["power"="plant"]({tile_south},{tile_west},{tile_north},{tile_east});
      way["power"="plant"]({tile_south},{tile_west},{tile_north},{tile_east});
    );
    out center;
    """
    
    # Human-readable cache name: facilities_<south>_<west>_<north>_<east>
    cache_name = f"facilities_{tile_south:.1f}_{tile_west:.1f}_{tile_north:.1f}_{tile_east:.1f}"
    result = query_overpass(query, timeout=90, cache_name=cache_name)
    
    if result and 'elements' in result:
        print(f"    Found {len(result['elements'])} elements")
        
        for el in result['elements']:
            el_id = el.get('id')
            if el_id in seen_ids:
                continue
            seen_ids.add(el_id)
            
            power_type = el.get('tags', {}).get('power', '')
            lat = el.get('lat') or (el.get('center', {}).get('lat'))
            lon = el.get('lon') or (el.get('center', {}).get('lon'))
            
            if not lat or not lon:
                continue
                
            tags = el.get('tags', {})
            
            if power_type == 'substation':
                all_substations.append({
                    'id': el_id,
                    'lat': lat,
                    'lon': lon,
                    'name': tags.get('name', tags.get('ref', f"Substation #{el_id}")),
                    'operator': tags.get('operator', tags.get('owner', 'Unknown')),
                    'voltage': tags.get('voltage', 'Unknown'),
                    'type': 'substation'
                })
            elif power_type == 'plant':
                all_plants.append({
                    'id': el_id,
                    'lat': lat,
                    'lon': lon,
                    'name': tags.get('name', tags.get('ref', f"Plant #{el_id}")),
                    'operator': tags.get('operator', 'Unknown'),
                    'output': tags.get('plant:output:electricity', 'Unknown'),
                    'source': tags.get('plant:source', 'Unknown'),
                    'type': power_type
                })
    else:
        print(f"    ⚠ No data for this tile")
    
    # Small delay between requests to avoid rate limiting
    time.sleep(1)

# Filter to impact zone (swath + coastal tiers)
substations_in_zone = []
for sub in all_substations:
    point = Point(sub['lon'], sub['lat'])
    if impact_zone.contains(point):
        substations_in_zone.append(sub)

plants_in_zone = []
for plant in all_plants:
    point = Point(plant['lon'], plant['lat'])
    if impact_zone.contains(point):
        plants_in_zone.append(plant)

print(f"\n{'='*50}")
print("Substations & Power Plants:")
print(f"  Total substations found: {len(all_substations)}")
print(f"  Substations in impact zone: {len(substations_in_zone)}")
print(f"  Total plants found: {len(all_plants)}")
print(f"  Plants in impact zone: {len(plants_in_zone)}")
print(f"{'='*50}")

#### 8.3.3 Download Transmission Lines

Query transmission lines in smaller tiles (3° × 3°) due to higher data density. Lines are filtered to those intersecting the **impact zone**.

In [ ]:
# Download Power Lines (split into tiles for reliability)
print(f"[{datetime.now().strftime('%H:%M:%S')}] Downloading transmission lines...")
print("(Using tiled approach - this may take a few minutes)")

# Use smaller tiles for lines (more data per tile)
LINE_TILE_SIZE = 3.0  # degrees

line_tiles = get_tiles(lat_min_infra, lon_min_infra, lat_max_infra, lon_max_infra, LINE_TILE_SIZE)
print(f"Split into {len(line_tiles)} tiles of {LINE_TILE_SIZE}° each")

all_lines = []
seen_line_ids = set()

for i, (tile_south, tile_west, tile_north, tile_east) in enumerate(line_tiles):
    print(f"\n  Tile {i+1}/{len(line_tiles)}: ({tile_south:.1f},{tile_west:.1f}) to ({tile_north:.1f},{tile_east:.1f})")
    
    query = f"""
    [out:json][timeout:120];
    (
      way["power"="line"]({tile_south},{tile_west},{tile_north},{tile_east});
    );
    out body geom;
    """
    
    # Human-readable cache name: transmission_<south>_<west>_<north>_<east>
    cache_name = f"transmission_{tile_south:.1f}_{tile_west:.1f}_{tile_north:.1f}_{tile_east:.1f}"
    result = query_overpass(query, timeout=150, cache_name=cache_name)
    
    if result and 'elements' in result:
        new_count = 0
        for el in result['elements']:
            el_id = el.get('id')
            if el_id in seen_line_ids:
                continue
            seen_line_ids.add(el_id)
            
            if 'geometry' in el:
                tags = el.get('tags', {})
                all_lines.append({
                    'id': el_id,
                    'geometry': el['geometry'],
                    'name': tags.get('name', tags.get('ref', '')),
                    'voltage': tags.get('voltage', 'Unknown'),
                    'operator': tags.get('operator', 'Unknown'),
                    'cables': tags.get('cables', 'Unknown'),
                    'type': 'line'
                })
                new_count += 1
        print(f"    Found {new_count} new lines (total: {len(all_lines)})")
    else:
        print(f"    ⚠ No data for this tile")
    
    # Delay between requests
    time.sleep(2)

# Filter lines to impact zone (swath + coastal tiers)
print(f"\nFiltering {len(all_lines)} lines to impact zone...")
lines_in_zone = []
for line in all_lines:
    if 'geometry' in line:
        coords = [(node['lon'], node['lat']) for node in line['geometry']]
        if len(coords) >= 2:
            line_geom = LineString(coords)
            if impact_zone.intersects(line_geom):
                lines_in_zone.append(line)

# Categorize by voltage
high_voltage_lines = []
medium_high_lines = []
medium_lines = []
low_voltage_lines = []
unknown_voltage_lines = []

for line in lines_in_zone:
    category, v = categorize_voltage(line['voltage'])
    if category == 'high':
        high_voltage_lines.append(line)
    elif category == 'medium-high':
        medium_high_lines.append(line)
    elif category == 'medium':
        medium_lines.append(line)
    elif category == 'low':
        low_voltage_lines.append(line)
    else:
        unknown_voltage_lines.append(line)

print(f"\n{'='*50}")
print("Transmission Lines:")
print(f"  Total lines found: {len(all_lines)}")
print(f"  Lines in impact zone: {len(lines_in_zone)}")
print(f"\nBy voltage category:")
print(f"  ≥345kV (High):     {len(high_voltage_lines)}")
print(f"  ≥115kV (Med-High): {len(medium_high_lines)}")
print(f"  ≥69kV (Medium):    {len(medium_lines)}")
print(f"  <69kV (Low):       {len(low_voltage_lines)}")
print(f"  Unknown:           {len(unknown_voltage_lines)}")
print(f"{'='*50}")

In [ ]:
# Summary of all infrastructure data
print(f"{'='*60}")
print("INFRASTRUCTURE DATA SUMMARY")
print(f"{'='*60}")
print(f"\nFacilities in Impact Zone (swath + coastal tiers):")
print(f"  Substations:    {len(substations_in_zone)}")
print(f"  Power Plants:   {len(plants_in_zone)}")
print(f"\nTransmission Lines in Impact Zone:")
print(f"  ≥345kV (High):     {len(high_voltage_lines)}")
print(f"  ≥115kV (Med-High): {len(medium_high_lines)}")
print(f"  ≥69kV (Medium):    {len(medium_lines)}")
print(f"  <69kV (Low):       {len(low_voltage_lines)}")
print(f"  Unknown:           {len(unknown_voltage_lines)}")
print(f"  TOTAL LINES:       {len(lines_in_zone)}")
print(f"{'='*60}")

### 8.4 Ingest Infra data to MPC Pro Geocatalog

We will now ingest the vector data to GeoCatalog

In [ ]:
# ============================================================================
# Upload Infrastructure Impact Analysis to MPC GeoCatalog
# ============================================================================
# This cell creates a new GeoCatalog collection with:
# 1. A GeoTIFF visualization showing swath tiers, coastal zones & infrastructure
# 2. A GeoJSON with all vector data (tracks, swath, coastal zones, power lines, facilities)
# ============================================================================

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as MplPolygon
from matplotlib.lines import Line2D
from matplotlib.collections import LineCollection
import matplotlib.patches as mpatches
import rasterio
from rasterio.transform import from_bounds
from rasterio.crs import CRS
import io
from PIL import Image
from shapely.geometry import LineString, Point, mapping
from datetime import datetime, timezone
import json
import pystac
import aiohttp
import asyncio

# Collection and Item IDs
INFRA_COLLECTION = f"hurricane-{storm_name.lower()}-{storm_year}-infra-impact"
INFRA_GEOTIFF_ITEM = f"{storm_name.lower()}-{storm_year}-infra-visualization"
INFRA_GEOJSON_ITEM = f"{storm_name.lower()}-{storm_year}-infra-vectors"

print("="*70)
print("  CREATING INFRASTRUCTURE IMPACT GEOCATALOG")
print("="*70)
print(f"\nCollection: {INFRA_COLLECTION}")
print(f"GeoTIFF Item: {INFRA_GEOTIFF_ITEM}")
print(f"GeoJSON Item: {INFRA_GEOJSON_ITEM}")

# ============================================================================
# Step 1: Calculate bounds from impact zone
# ============================================================================

infra_lon_min = lon_min_infra
infra_lon_max = lon_max_infra
infra_lat_min = lat_min_infra
infra_lat_max = lat_max_infra

print(f"\n  Bounding Box:")
print(f"   Latitude:  {infra_lat_min:.2f}° to {infra_lat_max:.2f}°")
print(f"   Longitude: {infra_lon_min:.2f}° to {infra_lon_max:.2f}°")

# ============================================================================
# Step 2: Create GeoTIFF visualization with swath + coastal + infrastructure
# ============================================================================

print(f"\n  Creating infrastructure visualization...")

DPI = 150
fig_width = 18
fig_height = 14

fig, ax = plt.subplots(figsize=(fig_width, fig_height), dpi=DPI)
ax.set_xlim(infra_lon_min, infra_lon_max)
ax.set_ylim(infra_lat_min, infra_lat_max)
ax.set_facecolor('#E6F3FF')

# ── Swath tier colors / labels (matching Section 7.1b) ──
_SW_CLR_G  = ['crimson', 'darkorange', 'gold']
_SW_ALP_G  = [0.55, 0.40, 0.25]
_SW_LBL_G  = ['Inner core (25 %)', 'Mid-level (55 %)', 'Outer extent (100 %)']

_CO_CLR_G  = {'core': 'crimson', 'surge': 'darkorange'}
_CO_ALP_G  = {'core': 0.50,     'surge': 0.40}
_CO_LBL_G  = {
    'core':  'Core (~200 km) — hurricane winds / surge',
    'surge': 'Surge (~350 km) — outer bands / rainfall',
}

def _draw_geom_mpl(ax, geom, fc, alpha, zorder, ec='none', lw=0):
    """Render a Shapely Polygon/MultiPolygon onto a matplotlib Axes."""
    if geom is None or geom.is_empty:
        return
    if geom.geom_type == 'GeometryCollection':
        for g in geom.geoms:
            _draw_geom_mpl(ax, g, fc, alpha, zorder, ec, lw)
        return
    if geom.geom_type not in ('Polygon', 'MultiPolygon'):
        return
    polys = [geom] if geom.geom_type == 'Polygon' else list(geom.geoms)
    for p in polys:
        xy = np.array(p.exterior.coords)
        patch = MplPolygon(xy, closed=True, fc=fc, ec=ec, alpha=alpha, lw=lw, zorder=zorder)
        ax.add_patch(patch)

# ----- Draw Coastal Impact Zones (bottom layer) -----
for _tk in ['surge', 'core']:
    _draw_geom_mpl(ax, coast_strips.get(_tk), _CO_CLR_G[_tk], _CO_ALP_G[_tk], zorder=1)
print("   ✓ Coastal impact zones")

# ----- Draw Swath Tiers (above coastal) -----
for _ti in [2, 1, 0]:
    if _ti < len(swath_polys) and swath_polys[_ti] is not None:
        _draw_geom_mpl(ax, swath_polys[_ti], _SW_CLR_G[_ti], _SW_ALP_G[_ti],
                        zorder=2, ec=_SW_CLR_G[_ti], lw=0.8)
print("   ✓ Impact swath tiers")

# ----- Draw Power Lines by voltage category -----
def draw_lines(line_list, color, linewidth, label, alpha=0.7, zorder=3):
    if not line_list:
        return
    segments = []
    for line in line_list:
        if 'geometry' in line:
            coords = [(node['lon'], node['lat']) for node in line['geometry']]
            if len(coords) >= 2:
                segments.append(coords)
    if segments:
        lc = LineCollection(segments, colors=color, linewidths=linewidth, alpha=alpha, zorder=zorder)
        ax.add_collection(lc)
    print(f"   ✓ {label}: {len(line_list)} lines")

draw_lines(unknown_voltage_lines, '#999999', 0.5, 'Unknown voltage', alpha=0.4, zorder=3)
draw_lines(low_voltage_lines, '#90EE90', 0.8, 'Low voltage (<69kV)', alpha=0.5, zorder=4)
draw_lines(medium_lines, '#FFD700', 1.0, 'Medium voltage (69-114kV)', alpha=0.6, zorder=5)
draw_lines(medium_high_lines, '#FF8C00', 1.5, 'Medium-high voltage (115-344kV)', alpha=0.7, zorder=6)
draw_lines(high_voltage_lines, '#FF0000', 2.5, 'High voltage (≥345kV)', alpha=0.9, zorder=7)

# ----- Draw Substations -----
if substations_in_zone:
    sub_lons = [s['lon'] for s in substations_in_zone]
    sub_lats = [s['lat'] for s in substations_in_zone]
    ax.scatter(sub_lons, sub_lats, c='blue', s=50, marker='s',
               edgecolors='white', linewidths=0.5, zorder=9, label='Substations')
    print(f"   ✓ Substations: {len(substations_in_zone)}")

# ----- Draw Power Plants -----
if plants_in_zone:
    plant_lons = [p['lon'] for p in plants_in_zone]
    plant_lats = [p['lat'] for p in plants_in_zone]
    ax.scatter(plant_lons, plant_lats, c='purple', s=80, marker='^',
               edgecolors='white', linewidths=0.5, zorder=10, label='Power Plants')
    print(f"   ✓ Power plants: {len(plants_in_zone)}")

# ----- Draw Observed Track -----
if observed_track is not None and len(observed_track) > 0:
    ax.plot(observed_track['lon_converted'], observed_track['lat'],
            color='black', linewidth=3, linestyle='-', zorder=11)
    ax.scatter(observed_track['lon_converted'], observed_track['lat'],
               c='black', s=30, edgecolors='white', linewidths=0.5, zorder=12)
    print("   ✓ Observed track")

# ----- Draw Forecast Track -----
if track is not None and len(track) > 0:
    ax.plot(track['lon_converted'], track['lat'],
            color='blue', linewidth=2.5, linestyle='--', zorder=13)
    ax.scatter(track['lon_converted'], track['lat'],
               c='blue', s=25, edgecolors='white', linewidths=0.5, zorder=14)
    print("   ✓ Forecast track")

# Title
ax.set_title(
    f"Hurricane {storm_name} ({storm_year}) - Infrastructure Impact Analysis\n"
    f"Impact Swath + Coastal Zones  |  "
    f"Substations: {len(substations_in_zone)} | Power Plants: {len(plants_in_zone)} | "
    f"Transmission Lines: {len(lines_in_zone)}",
    fontsize=14, fontweight='bold', pad=10
)

# Legend
legend_elements = [
    Line2D([0], [0], color='black', linewidth=3, label='Observed Track'),
    Line2D([0], [0], color='blue', linewidth=2.5, linestyle='--', label='Aurora Forecast'),
    mpatches.Patch(fc='crimson', alpha=0.55, label='Swath Inner Core'),
    mpatches.Patch(fc='darkorange', alpha=0.40, label='Swath Mid-Level'),
    mpatches.Patch(fc='gold', alpha=0.25, label='Swath Outer Extent'),
    mpatches.Patch(fc='lightskyblue', alpha=0.25, label='Coastal Flood Zone'),
    Line2D([0], [0], color='#FF0000', linewidth=2.5, label='High Voltage (≥345kV)'),
    Line2D([0], [0], color='#FF8C00', linewidth=1.5, label='Med-High (115-344kV)'),
    Line2D([0], [0], color='#FFD700', linewidth=1.0, label='Medium (69-114kV)'),
    Line2D([0], [0], marker='s', color='w', markerfacecolor='blue', markersize=8, label='Substations'),
    Line2D([0], [0], marker='^', color='w', markerfacecolor='purple', markersize=10, label='Power Plants'),
]
ax.legend(handles=legend_elements, loc='upper right', fontsize=8, framealpha=0.9, ncol=2)

plt.tight_layout()

# Save to buffer
buf = io.BytesIO()
fig.savefig(buf, format='png', dpi=DPI, bbox_inches='tight', pad_inches=0.1,
            facecolor='white', edgecolor='none')
buf.seek(0)
img = Image.open(buf)
img_array = np.array(img)
plt.close(fig)

infra_actual_height, infra_actual_width = img_array.shape[:2]
print(f"\n  Image: {infra_actual_width} x {infra_actual_height} pixels")

# Save as GeoTIFF
infra_geotiff_path = os.path.join(OUTPUTS_DIR, f"{INFRA_GEOTIFF_ITEM}.tif")
transform = from_bounds(infra_lon_min, infra_lat_min, infra_lon_max, infra_lat_max,
                        infra_actual_width, infra_actual_height)

count = img_array.shape[2] if len(img_array.shape) == 3 else 1
bands = [img_array[:, :, i] for i in range(count)] if count > 1 else [img_array]

with rasterio.open(
    infra_geotiff_path, 'w', driver='GTiff',
    height=infra_actual_height, width=infra_actual_width,
    count=count, dtype=img_array.dtype,
    crs=CRS.from_epsg(4326), transform=transform,
    compress='LZW', tiled=True, blockxsize=256, blockysize=256,
    photometric='RGB' if count >= 3 else 'MINISBLACK'
) as dst:
    for i, band in enumerate(bands, 1):
        dst.write(band, i)

file_size_mb = os.path.getsize(infra_geotiff_path) / (1024 * 1024)
print(f"   ✅ GeoTIFF saved: {infra_geotiff_path} ({file_size_mb:.2f} MB)")
buf.close()

# ============================================================================
# Step 3: Create GeoJSON with all vector data
# ============================================================================

print(f"\n  Creating GeoJSON with all vector data...")

infra_features = []

# Feature: Swath tiers
_sw_labels = ['inner_core', 'mid_level', 'outer_extent']
_sw_colors = ['#DC143C', '#FF8C00', '#FFD700']
for ti in range(3):
    if ti < len(swath_polys) and swath_polys[ti] is not None and not swath_polys[ti].is_empty:
        infra_features.append({
            "type": "Feature",
            "properties": {
                "layer": "swath_tier",
                "tier": _sw_labels[ti],
                "name": f"Impact Swath — {_SW_LBL_G[ti]}",
                "fill_color": _sw_colors[ti],
                "fill_opacity": _SW_ALP_G[ti]
            },
            "geometry": mapping(swath_polys[ti])
        })
print(f"   ✓ Swath tiers: {sum(1 for sp in swath_polys if sp and not sp.is_empty)}")

# Feature: Coastal impact zones
for tier_key in ['core', 'surge']:
    cs = coast_strips.get(tier_key)
    if cs is not None and not cs.is_empty:
        infra_features.append({
            "type": "Feature",
            "properties": {
                "layer": "coastal_zone",
                "tier": tier_key,
                "name": f"Coastal Zone — {_CO_LBL_G[tier_key]}",
                "fill_color": _CO_CLR_G[tier_key],
                "fill_opacity": _CO_ALP_G[tier_key]
            },
            "geometry": mapping(cs)
        })
_cz_count = sum(1 for k in ['core', 'surge']
                if coast_strips.get(k) is not None and not coast_strips[k].is_empty)
print(f"   ✓ Coastal zones: {_cz_count}")

# Feature: Observed Track
if observed_track is not None and len(observed_track) > 0:
    obs_coords = list(zip(observed_track['lon_converted'].tolist(), observed_track['lat'].tolist()))
    infra_features.append({
        "type": "Feature",
        "properties": {
            "layer": "observed_track",
            "name": f"Hurricane {storm_name} Observed Track",
            "color": "#000000",
            "stroke_width": 3,
            "points": len(observed_track)
        },
        "geometry": mapping(LineString(obs_coords))
    })
    print(f"   ✓ Observed track: {len(obs_coords)} points")

# Feature: Forecast Track
if track is not None and len(track) > 0:
    fc_coords = list(zip(track['lon_converted'].tolist(), track['lat'].tolist()))
    infra_features.append({
        "type": "Feature",
        "properties": {
            "layer": "forecast_track",
            "name": f"Hurricane {storm_name} Aurora Forecast",
            "color": "#0000FF",
            "stroke_width": 2.5,
            "points": len(track)
        },
        "geometry": mapping(LineString(fc_coords))
    })
    print(f"   ✓ Forecast track: {len(fc_coords)} points")

# Features: Power Lines
def add_line_features(line_list, voltage_category, color):
    for line in line_list:
        if 'geometry' in line:
            coords = [(node['lon'], node['lat']) for node in line['geometry']]
            if len(coords) >= 2:
                infra_features.append({
                    "type": "Feature",
                    "properties": {
                        "layer": "power_line",
                        "voltage_category": voltage_category,
                        "name": line.get('name', ''),
                        "voltage": line.get('voltage', 'Unknown'),
                        "operator": line.get('operator', 'Unknown'),
                        "color": color,
                        "osm_id": line.get('id')
                    },
                    "geometry": mapping(LineString(coords))
                })

add_line_features(high_voltage_lines, "high", "#FF0000")
add_line_features(medium_high_lines, "medium-high", "#FF8C00")
add_line_features(medium_lines, "medium", "#FFD700")
add_line_features(low_voltage_lines, "low", "#90EE90")
add_line_features(unknown_voltage_lines, "unknown", "#999999")
print(f"   ✓ Power lines: {len(lines_in_zone)}")

# Features: Substations
for sub in substations_in_zone:
    infra_features.append({
        "type": "Feature",
        "properties": {
            "layer": "substation",
            "name": sub.get('name', 'Unknown'),
            "operator": sub.get('operator', 'Unknown'),
            "voltage": sub.get('voltage', 'Unknown'),
            "osm_id": sub.get('id'),
            "marker_color": "#0000FF"
        },
        "geometry": mapping(Point(sub['lon'], sub['lat']))
    })
print(f"   ✓ Substations: {len(substations_in_zone)}")

# Features: Power Plants
for plant in plants_in_zone:
    infra_features.append({
        "type": "Feature",
        "properties": {
            "layer": "power_plant",
            "name": plant.get('name', 'Unknown'),
            "operator": plant.get('operator', 'Unknown'),
            "output": plant.get('output', 'Unknown'),
            "source": plant.get('source', 'Unknown'),
            "osm_id": plant.get('id'),
            "marker_color": "#800080"
        },
        "geometry": mapping(Point(plant['lon'], plant['lat']))
    })
print(f"   ✓ Power plants: {len(plants_in_zone)}")

# Create FeatureCollection
infra_geojson_data = {
    "type": "FeatureCollection",
    "features": infra_features,
    "properties": {
        "storm_name": storm_name,
        "storm_year": storm_year,
        "analysis_method": "asymmetric_impact_swath_v3",
        "total_substations": len(substations_in_zone),
        "total_plants": len(plants_in_zone),
        "total_lines": len(lines_in_zone),
        "high_voltage_lines": len(high_voltage_lines),
        "medium_high_lines": len(medium_high_lines),
        "medium_lines": len(medium_lines),
        "low_voltage_lines": len(low_voltage_lines),
        "created": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
    }
}

# Save GeoJSON
infra_geojson_path = os.path.join(OUTPUTS_DIR, f"{INFRA_GEOJSON_ITEM}.geojson")
with open(infra_geojson_path, 'w') as f:
    json.dump(infra_geojson_data, f)

geojson_size_kb = os.path.getsize(infra_geojson_path) / 1024
print(f"   ✅ GeoJSON saved: {infra_geojson_path} ({geojson_size_kb:.1f} KB)")
print(f"   Total features: {len(infra_features)}")

# ============================================================================
# Step 4: Upload files to Blob Storage
# ============================================================================

print(f"\n  Uploading to blob storage...")

infra_geotiff_blob = f"infra-data/{INFRA_GEOTIFF_ITEM}.tif"
infra_geotiff_blob_url, infra_geotiff_sas_url = await upload_to_blob_storage(
    Path(infra_geotiff_path), infra_geotiff_blob
)
print(f"   ✅ GeoTIFF uploaded")

infra_geojson_blob = f"infra-data/{INFRA_GEOJSON_ITEM}.geojson"
infra_geojson_blob_url, infra_geojson_sas_url = await upload_to_blob_storage(
    Path(infra_geojson_path), infra_geojson_blob
)
print(f"   ✅ GeoJSON uploaded")

# ============================================================================
# Step 5: Create GeoCatalog Collection
# ============================================================================

async def ensure_infra_collection_exists(collection_id: str):
    """Create infrastructure collection if it doesn't exist."""
    print(f"\n  Checking collection '{collection_id}'...")
    
    headers = get_bearer_token()
    params = {"api-version": API_VERSION}
    
    timeout = aiohttp.ClientTimeout(total=300)
    async with aiohttp.ClientSession(timeout=timeout) as session:
        collection_url = join_url(STAC_API_URL, "collections", collection_id)
        
        async with session.get(collection_url, headers=headers, params=params) as response:
            if response.status == 200:
                print(f"   ✓ Collection already exists")
                return
        
        print(f"   Creating new collection...")
        
        item_assets = {
            "visual": {
                "type": "image/tiff; application=geotiff; profile=cloud-optimized",
                "roles": ["data", "visual"],
                "title": "Infrastructure Impact Visualization (COG)"
            },
            "data": {
                "type": "application/geo+json",
                "roles": ["data"],
                "title": "Infrastructure Vector Data (GeoJSON)"
            }
        }
        
        collection_dict = {
            "id": collection_id,
            "type": "Collection",
            "stac_version": "1.0.0",
            "title": f"Hurricane {storm_name} {storm_year} - Infrastructure Impact (Swath Analysis)",
            "description": (
                f"Infrastructure impact analysis for Hurricane {storm_name} ({storm_year}). "
                f"Uses asymmetric intensity-modulated impact swath and coastal flood/surge zones "
                f"to identify at-risk power infrastructure (lines, substations, plants)."
            ),
            "license": "proprietary",
            "keywords": ["hurricane", "infrastructure", "power grid", "impact swath",
                         "coastal zones", storm_name.lower()],
            "extent": {
                "spatial": {"bbox": [[infra_lon_min, infra_lat_min, infra_lon_max, infra_lat_max]]},
                "temporal": {"interval": [[storm_init_time.strftime("%Y-%m-%dT%H:%M:%SZ"),
                                           storm_end_time.strftime("%Y-%m-%dT%H:%M:%SZ")]]}
            },
            "item_assets": item_assets,
            "summaries": {
                "substations": [len(substations_in_zone)],
                "power_plants": [len(plants_in_zone)],
                "transmission_lines": [len(lines_in_zone)]
            },
            "links": []
        }
        
        collections_url = join_url(STAC_API_URL, "collections")
        async with session.post(
            collections_url, json=collection_dict,
            headers={**headers, "Content-Type": "application/json"},
            params=params
        ) as response:
            if response.status >= 400:
                error_text = await response.text()
                print(f"   ❌ Failed: {response.status} - {error_text}")
                raise Exception(f"Failed to create collection")
            
            print(f"   ✓ Collection created (status {response.status})")
            if response.status == 202:
                print(f"   ⏳ Waiting for availability...")
                for attempt in range(10):
                    await asyncio.sleep(3)
                    async with session.get(collection_url, headers=headers, params=params) as check:
                        if check.status == 200:
                            print(f"   ✓ Collection available")
                            break

await ensure_infra_collection_exists(INFRA_COLLECTION)

# ============================================================================
# Step 6: Create and Ingest STAC Items
# ============================================================================

print(f"\n  Creating STAC items...")

item_datetime = storm_init_time
if hasattr(item_datetime, 'tzinfo') and item_datetime.tzinfo is None:
    item_datetime = item_datetime.replace(tzinfo=timezone.utc)

geometry = {
    "type": "Polygon",
    "coordinates": [[[infra_lon_min, infra_lat_min], [infra_lon_max, infra_lat_min],
                     [infra_lon_max, infra_lat_max], [infra_lon_min, infra_lat_max],
                     [infra_lon_min, infra_lat_min]]]
}

# GeoTIFF Item
infra_geotiff_item = pystac.Item(
    id=INFRA_GEOTIFF_ITEM,
    geometry=geometry,
    bbox=[infra_lon_min, infra_lat_min, infra_lon_max, infra_lat_max],
    datetime=item_datetime,
    properties={
        "title": f"Hurricane {storm_name} ({storm_year}) - Infrastructure Impact (Swath)",
        "storm_name": storm_name,
        "storm_year": storm_year,
        "asset_type": "raster",
        "analysis_method": "asymmetric_impact_swath_v3",
        "substations": len(substations_in_zone),
        "power_plants": len(plants_in_zone),
        "transmission_lines": len(lines_in_zone),
        "proj:epsg": 4326
    },
    stac_extensions=[
        "https://stac-extensions.github.io/projection/v1.1.0/schema.json",
        "https://stac-extensions.github.io/raster/v1.1.0/schema.json"
    ]
)

infra_geotiff_item.add_asset(
    "visual",
    pystac.Asset(
        href=infra_geotiff_sas_url,
        media_type="image/tiff; application=geotiff; profile=cloud-optimized",
        title=f"Infrastructure Impact Visualization (COG)",
        roles=["data", "visual"],
        extra_fields={
            "proj:epsg": 4326,
            "proj:shape": [infra_actual_height, infra_actual_width],
            "raster:bands": [
                {"name": "red", "data_type": "uint8"},
                {"name": "green", "data_type": "uint8"},
                {"name": "blue", "data_type": "uint8"},
                {"name": "alpha", "data_type": "uint8"}
            ]
        }
    )
)

# GeoJSON Item
infra_geojson_item = pystac.Item(
    id=INFRA_GEOJSON_ITEM,
    geometry=geometry,
    bbox=[infra_lon_min, infra_lat_min, infra_lon_max, infra_lat_max],
    datetime=item_datetime,
    properties={
        "title": f"Hurricane {storm_name} ({storm_year}) - Infrastructure Vector Data (Swath)",
        "storm_name": storm_name,
        "storm_year": storm_year,
        "asset_type": "vector",
        "analysis_method": "asymmetric_impact_swath_v3",
        "feature_count": len(infra_features),
        "substations": len(substations_in_zone),
        "power_plants": len(plants_in_zone),
        "transmission_lines": len(lines_in_zone)
    }
)

infra_geojson_item.add_asset(
    "data",
    pystac.Asset(
        href=infra_geojson_sas_url,
        media_type="application/geo+json",
        title=f"Infrastructure Impact Data (GeoJSON)",
        roles=["data"]
    )
)

# Ingest items
async def ingest_infra_item(item: pystac.Item, collection_id: str):
    headers = get_bearer_token()
    params = {"api-version": API_VERSION}
    timeout = aiohttp.ClientTimeout(total=300)
    
    item_url = join_url(STAC_API_URL, "collections", collection_id, "items", item.id)
    
    async with aiohttp.ClientSession(timeout=timeout) as session:
        async with session.get(item_url, headers=headers, params=params) as response:
            if response.status == 200:
                print(f"   Replacing existing item '{item.id}'...")
                await session.delete(item_url, headers=headers, params=params)
                await asyncio.sleep(2)
        
        items_url = join_url(STAC_API_URL, "collections", collection_id, "items")
        async with session.post(
            items_url, json=item.to_dict(),
            headers={**headers, "Content-Type": "application/json"},
            params=params
        ) as response:
            if response.status >= 400:
                error_text = await response.text()
                print(f"   ❌ Failed: {item.id}")
            else:
                print(f"   ✅ Ingested '{item.id}'")

print(f"\n  Ingesting items...")
await ingest_infra_item(infra_geotiff_item, INFRA_COLLECTION)
await ingest_infra_item(infra_geojson_item, INFRA_COLLECTION)

# ============================================================================
# Step 7: Configure Render Options
# ============================================================================

print(f"\n  Configuring render options...")

try:
    existing_renders = pc_client.stac.list_render_options(collection_id=INFRA_COLLECTION)
    if existing_renders:
        for r in existing_renders:
            try:
                pc_client.stac.delete_render_option(collection_id=INFRA_COLLECTION, render_option_id=r.id)
            except:
                pass
    
    render_option = RenderOption(
        id="infra-visual",
        name="Infrastructure Impact Visualization",
        type=RenderOptionType.RASTER_TILE,
        options="assets=visual&bidx=1&bidx=2&bidx=3",
        min_zoom=0
    )
    
    pc_client.stac.create_render_option(collection_id=INFRA_COLLECTION, body=render_option)
    print(f"   ✅ Created render option: infra-visual")
except Exception as e:
    print(f"   ⚠️ Render config: {e}")

# ============================================================================
# Step 8: Generate Explorer URL and Launch
# ============================================================================

center_lon = (infra_lon_min + infra_lon_max) / 2
center_lat = (infra_lat_min + infra_lat_max) / 2

collection_api_url = f"{STAC_API_URL}/collections/{INFRA_COLLECTION}"

infra_explorer_url = (
    f"{GEOCATALOG_URI}/explorer"
    f"?c={center_lon:.4f}%2C{center_lat:.4f}"
    f"&z=5.00"
    f"&v=2"
    f"&d={INFRA_COLLECTION}"
    f"&m=default"
    f"&r=infra-visual"
    f"&s=false%3A%3A100%3A%3Atrue"
    f"&sr=desc"
    f"&ae=0"
)

print("\n" + "="*70)
print("✅ INFRASTRUCTURE IMPACT DATA UPLOADED TO GEOCATALOG")
print("="*70)
print(f"\n  Collection: {INFRA_COLLECTION}")
print(f"\n  Infrastructure Summary (Impact Swath + Coastal Zones):")
print(f"   Substations:    {len(substations_in_zone)}")
print(f"   Power Plants:   {len(plants_in_zone)}")
print(f"   Power Lines:    {len(lines_in_zone)}")
print(f"   Total Features: {len(infra_features)}")
print(f"\n  STAC Items (2):")
print(f"   1. {INFRA_GEOTIFF_ITEM} (GeoTIFF visualization)")
print(f"   2. {INFRA_GEOJSON_ITEM} (GeoJSON vectors)")
print(f"\n  Collection API:")
print(f"   {collection_api_url}")
print(f"\n  Explorer URL:")
print(f"   {infra_explorer_url}")
print("="*70)

import webbrowser
print(f"\n  Launching Explorer in browser...")
webbrowser.open(infra_explorer_url)

### 8.5 Generate Animated Storm & Infrastructure Map

Create an interactive Folium map with animated storm tracking, **impact swath tiers**, **coastal flood/surge zones**, and toggleable layers for different voltage levels and facility types. The map includes playback controls for storm progression and is saved as an HTML file for sharing.

In [ ]:
# Fallback: ensure URL variables are defined
# If earlier cells didn't run, rebuild URLs from GEOCATALOG_URI
_gc_base = GEOCATALOG_URI if 'GEOCATALOG_URI' in dir() else ''

# Always rebuild explorer_url for the INPUT collection (cell 52 previously overwrote it)
if _gc_base and 'INPUT_COLLECTION' in dir():
    explorer_url = f"{_gc_base}/explorer?d={INPUT_COLLECTION}"
else:
    explorer_url = locals().get('explorer_url', '') or '#'

forecast_explorer_url = locals().get('forecast_explorer_url', '') or (
    f"{_gc_base}/explorer?d={OUTPUT_COLLECTION}" if _gc_base and 'OUTPUT_COLLECTION' in dir() else '#')
track_explorer_url = locals().get('track_explorer_url', '') or (
    f"{_gc_base}/explorer?d={TRACK_COLLECTION}" if _gc_base and 'TRACK_COLLECTION' in dir() else '#')
infra_explorer_url_safe = (
    locals().get('infra_explorer_url_safe', '')
    or locals().get('infra_explorer_url', '')
    or (f"{_gc_base}/explorer?d={INFRA_COLLECTION}" if _gc_base and 'INFRA_COLLECTION' in dir() else '#'))


# =============================================================================
# Interactive Folium Map — Impact Swath + Coastal Zones + Infrastructure
# =============================================================================
import folium
from folium import FeatureGroup
import pandas as pd
from datetime import datetime
import json
from shapely.geometry import mapping as shapely_mapping

print(f"[{datetime.now().strftime('%H:%M:%S')}] Creating animated power infrastructure map...")

# ── Fix track times ──
print("\n" + "="*60)
print("Preparing track data with corrected times")
print("="*60)

init_time = track_df.iloc[0]['time']
print(f"Initial forecast time: {init_time}")

corrected_times = []
for i in range(len(track_df)):
    corrected_time = pd.Timestamp(init_time) + pd.Timedelta(hours=i*6)
    corrected_times.append(corrected_time)

track_df_fixed = track_df.copy()
track_df_fixed['time_corrected'] = corrected_times

print(f"\nCorrected times (first 5 points):")
for i in range(min(5, len(track_df_fixed))):
    row = track_df_fixed.iloc[i]
    print(f"  Point {i}: {row['time_corrected']} (was: {row['time']})")
print("="*60 + "\n")

# ── Map bounds from impact zone ──
iz_bnds = impact_zone.bounds
iz_west, iz_south, iz_east, iz_north = iz_bnds
center_lat = (iz_south + iz_north) / 2
center_lon = (iz_west + iz_east) / 2

print(f"Impact zone bounds: {iz_south:.2f}°N to {iz_north:.2f}°N, {iz_west:.2f}° to {iz_east:.2f}°")
print(f"\nUsing pre-filtered infrastructure data:")
print(f"  Substations in impact zone: {len(substations_in_zone)}")
print(f"  Power plants in impact zone: {len(plants_in_zone)}")
print(f"  High voltage lines (≥345kV): {len(high_voltage_lines)}")
print(f"  Medium-high lines (≥115kV): {len(medium_high_lines)}")
print(f"  Medium lines (≥69kV): {len(medium_lines)}")
print(f"  Low voltage lines (<69kV): {len(low_voltage_lines)}")
print(f"  Unknown voltage lines: {len(unknown_voltage_lines)}")

# ── Create base map ──
m = folium.Map(location=[center_lat, center_lon], zoom_start=6, tiles=None)
folium.TileLayer('OpenStreetMap', control=False).add_to(m)



# =========================================================================
# LAYER: COASTAL IMPACT ZONES  (bottom — toggle-able)
# =========================================================================
_folium_co_clr = {'core': '#DC143C', 'surge': '#FF8C00'}
_folium_co_alp = {'core': 0.30,     'surge': 0.28}
_folium_co_lbl = {
    'core':  '🌊 Coastal Core (hurricane winds / surge)',
    'surge': '🌊 Coastal Surge (outer bands / rainfall)',
}

for tier_key in ['surge', 'core']:
    cs = coast_strips.get(tier_key)
    if cs is None or cs.is_empty:
        continue
    fg_coast = FeatureGroup(name=_folium_co_lbl[tier_key], show=True)
    geojson_data = shapely_mapping(cs)
    folium.GeoJson(
        geojson_data,
        interactive=False,
        style_function=lambda feature, clr=_folium_co_clr[tier_key], alp=_folium_co_alp[tier_key]: {
            'fillColor': clr,
            'color': clr,
            'weight': 1,
            'fillOpacity': alp,
            'opacity': 0.5,
        },
        name=_folium_co_lbl[tier_key],
    ).add_to(fg_coast)
    fg_coast.add_to(m)
print("   ✓ Coastal impact zones added")

# =========================================================================
# LAYER: IMPACT SWATH TIERS  (above coastal — toggle-able)
# =========================================================================
_folium_sw_clr = ['#DC143C', '#FF8C00', '#FFD700']
_folium_sw_alp = [0.40, 0.28, 0.18]
_folium_sw_lbl = [
    '🔴 Swath Inner Core (25 %)',
    '🟠 Swath Mid-Level (55 %)',
    '🟡 Swath Outer Extent (100 %)',
]

for ti in [2, 1, 0]:
    sp = swath_polys[ti] if ti < len(swath_polys) else None
    if sp is None or sp.is_empty:
        continue
    fg_swath = FeatureGroup(name=_folium_sw_lbl[ti], show=True)
    geojson_data = shapely_mapping(sp)
    folium.GeoJson(
        geojson_data,
        interactive=False,
        style_function=lambda feature, clr=_folium_sw_clr[ti], alp=_folium_sw_alp[ti]: {
            'fillColor': clr,
            'color': clr,
            'weight': 1.5,
            'fillOpacity': alp,
            'opacity': 0.6,
        },
        name=_folium_sw_lbl[ti],
    ).add_to(fg_swath)
    fg_swath.add_to(m)
print("   ✓ Impact swath tiers added")

# =========================================================================
# LAYER: NHC CONE OF UNCERTAINTY  (classic symmetric cone — toggle-able)
# =========================================================================
# Build a simple symmetric cone using only NHC track-uncertainty radii
# (no wind-extent buffer, no right-of-track asymmetry).
from shapely.geometry import Point as _ConePoint
_cone_left = []
_cone_right = []
_cone_N = len(track_points)

for _ci, _cpt in enumerate(track_points):
    _c_half_km = max(nhc_radius_km(_cpt['lead_hours']), 40.0)

    # Taper at start (same as swath)
    _c_taper_n = min(4, _cone_N - 1)
    if _ci < _c_taper_n:
        _c_half_km *= 0.15 + 0.85 * (0.5 - 0.5 * np.cos(np.pi * _ci / _c_taper_n))

    # Motion vector
    if _ci == 0:
        _cdx = track_points[min(1, _cone_N - 1)]['lon'] - _cpt['lon']
        _cdy = track_points[min(1, _cone_N - 1)]['lat'] - _cpt['lat']
    elif _ci == _cone_N - 1:
        _cdx = _cpt['lon'] - track_points[_ci - 1]['lon']
        _cdy = _cpt['lat'] - track_points[_ci - 1]['lat']
    else:
        _cdx = track_points[_ci + 1]['lon'] - track_points[_ci - 1]['lon']
        _cdy = track_points[_ci + 1]['lat'] - track_points[_ci - 1]['lat']

    _cmag = np.hypot(_cdx, _cdy) or 1e-9
    _cpx, _cpy = -_cdy / _cmag, _cdx / _cmag

    _c_half_deg = km_to_degrees(_c_half_km, _cpt['lat'])
    _cone_left.append((_cpt['lon'] + _cpx * _c_half_deg, _cpt['lat'] + _cpy * _c_half_deg))
    _cone_right.append((_cpt['lon'] - _cpx * _c_half_deg, _cpt['lat'] - _cpy * _c_half_deg))

# Build the cone polygon
_cone_ring = _cone_left + _cone_right[::-1]
if len(_cone_ring) >= 3:
    _cone_ring.append(_cone_ring[0])
    _cone_poly = ShapelyPolygon(_cone_ring).buffer(0)
    # End cap
    _cone_last = track_points[-1]
    _cone_r_end = km_to_degrees(max(nhc_radius_km(_cone_last['lead_hours']), 40.0), _cone_last['lat'])
    _cone_cap = _ConePoint(_cone_last['lon'], _cone_last['lat']).buffer(_cone_r_end, resolution=64)
    _cone_poly = _cone_poly.union(_cone_cap)
    # Smooth
    _cone_poly = _cone_poly.buffer(0.12).buffer(-0.12)

    fg_cone = FeatureGroup(name='\U0001f535 NHC Cone of Uncertainty (classic)', show=False)
    folium.GeoJson(
        shapely_mapping(_cone_poly),
        interactive=False,
        style_function=lambda feature: {
            'fillColor': '#4A90D9',
            'color': '#2563EB',
            'weight': 2,
            'fillOpacity': 0.15,
            'opacity': 0.7,
        },
        name='NHC Cone of Uncertainty',
    ).add_to(fg_cone)
    fg_cone.add_to(m)
    print("   \u2713 NHC Cone of Uncertainty layer added (toggle in layer control)")
else:
    print("   \u26a0 Not enough points to build cone polygon")

# =========================================================================
# ACTUAL / OBSERVED TRACK
# =========================================================================
actual_track_data = []
actual_track_data = []
fg_actual_track = FeatureGroup(name='📍 Observed Track (HURDAT)', show=False)

if selected_storm_df is not None and len(selected_storm_df) > 0:
    print(f"\n  Adding observed track from tropycal ({len(selected_storm_df)} points)...")
    for idx, row in selected_storm_df.iterrows():
        lat = row['lat']
        lon = row['lon']
        if lon > 180:
            lon = lon - 360
        time_val = row['time']
        time_str = time_val.strftime('%Y-%m-%d %H:%M UTC') if hasattr(time_val, 'strftime') else str(time_val)
        vmax = row.get('vmax', None) if 'vmax' in row else None
        mslp = row.get('mslp', None) if 'mslp' in row else None
        actual_track_data.append({
            'lat': float(lat), 'lon': float(lon), 'time': time_str,
            'vmax': vmax, 'mslp': mslp
        })
    
    actual_track_coords = [(t['lat'], t['lon']) for t in actual_track_data]
    folium.PolyLine(
        actual_track_coords, weight=4, color='#1F2937', opacity=0.9,
        popup=f'Hurricane {storm_name} Actual Track (HURDAT/IBTrACS)'
    ).add_to(fg_actual_track)
    
    for i, tdata in enumerate(actual_track_data):
        popup_content = (f"<b>Observed Point {i+1}</b><br><b>{tdata['time']}</b><br>"
                         f"Lat: {tdata['lat']:.2f}° | Lon: {tdata['lon']:.2f}°")
        if tdata['vmax'] is not None:
            popup_content += f"<br>Wind: {tdata['vmax']:.0f} kt"
        if tdata['mslp'] is not None:
            popup_content += f"<br>Pressure: {tdata['mslp']:.0f} mb"
        folium.CircleMarker(
            location=[tdata['lat'], tdata['lon']], radius=4,
            color='#1F2937', fill=True, fillColor='#374151',
            fillOpacity=0.8, popup=popup_content
        ).add_to(fg_actual_track)
    print(f"   ✓ Actual track added: {len(actual_track_data)} points")
else:
    print("\n  ⚠️ No observed track data available")

fg_actual_track.add_to(m)

# =========================================================================
# FORECAST TRACK
# =========================================================================
track_data = []
for idx, row in track_df_fixed.iterrows():
    lat = row['lat']
    lon = row['lon']
    if lon > 180:
        lon = lon - 360
    time_val = row['time_corrected']
    time_str = time_val.strftime('%Y-%m-%d %H:%M UTC')
    forecast_hours = idx * 6
    track_data.append({
        'lat': float(lat), 'lon': float(lon),
        'time': time_str, 'forecast_hr': forecast_hours
    })

print(f"\nTrack data prepared: {len(track_data)} points")
track_coords = [(t['lat'], t['lon']) for t in track_data]

folium.PolyLine(
    track_coords, weight=4, color='#4F46E5', opacity=0.9,
    dash_array='5, 10',
    popup=f'Hurricane {storm_name} Predicted Track (Aurora Model)'
).add_to(m)

# Start / End markers
start_data = track_data[0]
folium.Marker(
    location=[start_data['lat'], start_data['lon']],
    popup=(f"<b>FORECAST START</b><br>Hurricane {storm_name}<br>"
           f"<b>{start_data['time']}</b><br>T+0h"),
    icon=folium.Icon(color='green', icon='play', prefix='fa')
).add_to(m)

end_data = track_data[-1]
folium.Marker(
    location=[end_data['lat'], end_data['lon']],
    popup=(f"<b>FORECAST END</b><br>Hurricane {storm_name}<br>"
           f"<b>{end_data['time']}</b><br>T+{end_data['forecast_hr']}h"),
    icon=folium.Icon(color='red', icon='stop', prefix='fa')
).add_to(m)

for i, tdata in enumerate(track_data[1:-1], 1):
    folium.CircleMarker(
        location=[tdata['lat'], tdata['lon']], radius=6,
        color='#4F46E5', fill=True, fillColor='#6366F1', fillOpacity=0.8,
        popup=(f"<b>Track Point {i+1}</b><br><b>{tdata['time']}</b><br>"
               f"T+{tdata['forecast_hr']}h")
    ).add_to(m)

# =========================================================================
# POWER LINES (toggle-able feature groups)
# =========================================================================
fg_high = FeatureGroup(name='⚡ ≥345kV Transmission (High)', show=True)
for line in high_voltage_lines:
    if 'geometry' in line:
        coords = [(node['lat'], node['lon']) for node in line['geometry']]
        folium.PolyLine(
            coords, weight=5, color='#DC2626', opacity=0.9,
            popup=f"<b>{line.get('name','')}</b><br>Voltage: {line.get('voltage','')}"
        ).add_to(fg_high)
fg_high.add_to(m)

fg_med_high = FeatureGroup(name='⚡ ≥115kV (Medium-High)', show=False)
for line in medium_high_lines:
    if 'geometry' in line:
        coords = [(node['lat'], node['lon']) for node in line['geometry']]
        folium.PolyLine(
            coords, weight=4, color='#F97316', opacity=0.85,
            popup=f"<b>{line.get('name','')}</b><br>Voltage: {line.get('voltage','')}"
        ).add_to(fg_med_high)
fg_med_high.add_to(m)

fg_med = FeatureGroup(name='⚡ ≥69kV (Medium)', show=False)
for line in medium_lines:
    if 'geometry' in line:
        coords = [(node['lat'], node['lon']) for node in line['geometry']]
        folium.PolyLine(
            coords, weight=3, color='#16A34A', opacity=0.85,
            popup=f"<b>{line.get('name','')}</b><br>Voltage: {line.get('voltage','')}"
        ).add_to(fg_med)
fg_med.add_to(m)

fg_low = FeatureGroup(name='⚡ <69kV (Low)', show=False)
for line in low_voltage_lines:
    if 'geometry' in line:
        coords = [(node['lat'], node['lon']) for node in line['geometry']]
        folium.PolyLine(coords, weight=2, color='#2563EB', opacity=0.8).add_to(fg_low)
fg_low.add_to(m)

fg_unknown = FeatureGroup(name='⚡ Unknown Voltage', show=False)
for line in unknown_voltage_lines:
    if 'geometry' in line:
        coords = [(node['lat'], node['lon']) for node in line['geometry']]
        folium.PolyLine(coords, weight=2, color='#64748B', opacity=0.7).add_to(fg_unknown)
fg_unknown.add_to(m)

# =========================================================================
# SUBSTATIONS & POWER PLANTS
# =========================================================================
fg_substations = FeatureGroup(name='🔌 Substations', show=False)
for sub in substations_in_zone:
    voltage_str = sub.get('voltage', 'Unknown')
    if voltage_str != 'Unknown':
        try:
            v_int = int(str(voltage_str).replace(' ', '').split(';')[0])
            voltage_str = f"{v_int/1000:.0f} kV" if v_int >= 1000 else f"{v_int} V"
        except:
            pass
    folium.CircleMarker(
        location=[sub['lat'], sub['lon']], radius=8,
        color='#0E7490', fill=True, fillColor='#06B6D4',
        fillOpacity=0.9, weight=2,
        popup=f"<b>⚡ {sub['name']}</b><br>Operator: {sub['operator']}<br>Voltage: {voltage_str}"
    ).add_to(fg_substations)
fg_substations.add_to(m)

fg_plants = FeatureGroup(name='🏭 Power Plants', show=False)
for plant in plants_in_zone:
    folium.CircleMarker(
        location=[plant['lat'], plant['lon']], radius=10,
        color='#047857', fill=True, fillColor='#10B981',
        fillOpacity=0.9, weight=2,
        popup=(f"<b>🏭 {plant['name']}</b><br>Operator: {plant['operator']}<br>"
               f"Output: {plant['output']}<br>Source: {plant['source']}")
    ).add_to(fg_plants)
fg_plants.add_to(m)

# Layer control
folium.LayerControl(collapsed=False).add_to(m)

# (z-ordering handled by custom Leaflet panes: zones/infra/tracks)

# =========================================================================
# LEGEND — swath tiers, coastal zones, and cone of uncertainty

# =========================================================================
total_forecast_hours = (len(track_data) - 1) * 6
legend_html = f'''
<div id="map-legend" style="position: fixed; bottom: 10px; left: 10px; background: rgba(255,255,255,0.97);
            padding: 14px; border-radius: 8px; box-shadow: 0 3px 12px rgba(0,0,0,0.25);
            z-index: 9999; font-size: 11px; max-width: 310px; border: 1.5px solid #cbd5e1;">

    <div style="font-size: 13px; font-weight: bold; color: #0f172a; margin-bottom: 2px;">
        🌀 Hurricane {selected_storm.name} ({selected_storm.year})
    </div>
    <div style="color: #475569; font-size: 13px; margin-bottom: 2px;">
        Aurora Model {total_forecast_hours}h Forecast
    </div>
    <div style="color: #64748b; font-size: 11px; margin-bottom: 12px;">
        Init: {track_data[0]['time']}
    </div>

    <!-- Impact Swath Tiers -->
    <div style="border-top: 1px solid #e2e8f0; padding-top: 6px; margin-bottom: 6px;">
        <div style="font-weight: bold; color: #1e293b; margin-bottom: 4px; font-size: 11px;">
            🎯 Impact Swath
        </div>
        <table style="width: 100%; border-collapse: collapse; font-size: 12px;">
            <tr style="height: 20px;">
                <td style="width: 50px;">
                    <svg width="40" height="14"><rect x="0" y="0" width="40" height="14" fill="#DC143C" fill-opacity="0.55" rx="3"/></svg>
                </td>
                <td style="color: #334155;">Inner Core (25 %)</td>
            </tr>
            <tr style="height: 20px;">
                <td>
                    <svg width="40" height="14"><rect x="0" y="0" width="40" height="14" fill="#FF8C00" fill-opacity="0.40" rx="3"/></svg>
                </td>
                <td style="color: #334155;">Mid-Level (55 %)</td>
            </tr>
            <tr style="height: 20px;">
                <td>
                    <svg width="40" height="14"><rect x="0" y="0" width="40" height="14" fill="#FFD700" fill-opacity="0.25" rx="3"/></svg>
                </td>
                <td style="color: #334155;">Outer Extent (100 %)</td>
            </tr>
        </table>
    </div>

    <!-- Coastal Impact Zones -->
    <div style="border-top: 1px solid #e2e8f0; padding-top: 6px; margin-bottom: 6px;">
        <div style="font-weight: bold; color: #1e293b; margin-bottom: 4px; font-size: 11px;">
            🌊 Coastal Impact Zones
        </div>
        <table style="width: 100%; border-collapse: collapse; font-size: 12px;">
            <tr style="height: 20px;">
                <td style="width: 50px;">
                    <svg width="40" height="14"><rect x="0" y="0" width="40" height="14" fill="#DC143C" fill-opacity="0.30" rx="3"/></svg>
                </td>
                <td style="color: #334155;">Core — hurricane winds / surge</td>
            </tr>
            <tr style="height: 20px;">
                <td>
                    <svg width="40" height="14"><rect x="0" y="0" width="40" height="14" fill="#FF8C00" fill-opacity="0.28" rx="3"/></svg>
                </td>
                <td style="color: #334155;">Surge — outer bands / rainfall</td>
            </tr>
        </table>
    </div>
    <!-- Cone of Uncertainty -->
    <div style="border-top: 1px solid #e2e8f0; padding-top: 6px; margin-bottom: 6px;">
        <div style="font-weight: bold; color: #1e293b; margin-bottom: 4px; font-size: 11px;">
            🔵 Cone of Uncertainty

        </div>
        <table style="width: 100%; border-collapse: collapse; font-size: 12px;">
            <tr style="height: 20px;">
                <td style="width: 50px;">
                    <svg width="40" height="14"><rect x="0" y="0" width="40" height="14" fill="#4A90D9" fill-opacity="0.15" stroke="#2563EB" stroke-width="1.5" rx="3"/></svg>
                </td>
                <td style="color: #334155;">Classic symmetric cone</td>
            </tr>
        </table>
    </div>


    <!-- Power Lines -->
    <div style="border-top: 1px solid #e2e8f0; padding-top: 6px; margin-bottom: 6px;">
        <div style="font-weight: bold; color: #1e293b; margin-bottom: 4px; font-size: 11px;">
            ⚡ Power Lines
        </div>
        <table style="width: 100%; border-collapse: collapse; font-size: 12px;">
            <tr style="height: 20px;">
                <td style="width: 50px;">
                    <svg width="40" height="8"><line x1="0" y1="4" x2="40" y2="4" stroke="#DC2626" stroke-width="5"/></svg>
                </td>
                <td style="color: #334155;">≥345kV High Voltage</td>
                <td style="color: #64748b; text-align: right;">({len(high_voltage_lines)})</td>
            </tr>
            <tr style="height: 20px;">
                <td>
                    <svg width="40" height="8"><line x1="0" y1="4" x2="40" y2="4" stroke="#F97316" stroke-width="4"/></svg>
                </td>
                <td style="color: #334155;">≥115kV Medium-High</td>
                <td style="color: #64748b; text-align: right;">({len(medium_high_lines)})</td>
            </tr>
            <tr style="height: 20px;">
                <td>
                    <svg width="40" height="8"><line x1="0" y1="4" x2="40" y2="4" stroke="#16A34A" stroke-width="3"/></svg>
                </td>
                <td style="color: #334155;">≥69kV Medium</td>
                <td style="color: #64748b; text-align: right;">({len(medium_lines)})</td>
            </tr>
            <tr style="height: 20px;">
                <td>
                    <svg width="40" height="8"><line x1="0" y1="4" x2="40" y2="4" stroke="#2563EB" stroke-width="2"/></svg>
                </td>
                <td style="color: #334155;">&lt;69kV Low Voltage</td>
                <td style="color: #64748b; text-align: right;">({len(low_voltage_lines)})</td>
            </tr>
            <tr style="height: 20px;">
                <td>
                    <svg width="40" height="8"><line x1="0" y1="4" x2="40" y2="4" stroke="#64748B" stroke-width="2" stroke-dasharray="4,3"/></svg>
                </td>
                <td style="color: #334155;">Unknown Voltage</td>
                <td style="color: #64748b; text-align: right;">({len(unknown_voltage_lines)})</td>
            </tr>
        </table>
    </div>

    <!-- Storm Tracks -->
    <div style="border-top: 1px solid #e2e8f0; padding-top: 6px; margin-bottom: 6px;">
        <div style="font-weight: bold; color: #1e293b; margin-bottom: 4px; font-size: 11px;">
            📍 Storm Tracks
        </div>
        <table style="width: 100%; border-collapse: collapse; font-size: 12px;">
            <tr style="height: 20px;">
                <td style="width: 50px; text-align: center;">
                    <svg width="40" height="10"><line x1="0" y1="5" x2="40" y2="5" stroke="#1F2937" stroke-width="4"/></svg>
                </td>
                <td style="color: #334155;">Actual Path (HURDAT)</td>
                <td style="color: #64748b; text-align: right;">({len(actual_track_data)})</td>
            </tr>
            <tr style="height: 20px;">
                <td style="text-align: center;">
                    <svg width="40" height="10"><line x1="0" y1="5" x2="40" y2="5" stroke="#4F46E5" stroke-width="3" stroke-dasharray="5,5"/></svg>
                </td>
                <td style="color: #334155;">Predicted Path (Aurora)</td>
                <td style="color: #64748b; text-align: right;">({len(track_data)})</td>
            </tr>
            <tr style="height: 20px;">
                <td style="text-align: center;">
                    <svg width="40" height="20"><polygon points="10,2 30,10 10,18" fill="#22c55e"/></svg>
                </td>
                <td colspan="2" style="color: #334155;">Forecast Start (T+0h)</td>
            </tr>
            <tr style="height: 20px;">
                <td style="text-align: center;">
                    <svg width="40" height="20"><rect x="10" y="2" width="16" height="16" fill="#ef4444"/></svg>
                </td>
                <td colspan="2" style="color: #334155;">Forecast End (T+{total_forecast_hours}h)</td>
            </tr>
            <tr style="height: 20px;">
                <td style="text-align: center;">
                    <span style="font-size: 22px;">🌀</span>
                </td>
                <td colspan="2" style="color: #334155;">Animated Storm Position</td>
            </tr>
        </table>
    </div>

    <!-- Facilities -->
    <div style="border-top: 1px solid #e2e8f0; padding-top: 12px;">
        <div style="font-weight: bold; color: #1e293b; margin-bottom: 4px; font-size: 11px;">
            🏢 Facilities
        </div>
        <table style="width: 100%; border-collapse: collapse; font-size: 12px;">
            <tr style="height: 24px;">
                <td style="width: 50px; text-align: center;">
                    <svg width="40" height="24"><circle cx="20" cy="12" r="8" fill="#06B6D4" stroke="#0E7490" stroke-width="2"/></svg>
                </td>
                <td style="color: #334155; font-weight: 500;">Substations</td>
                <td style="color: #64748b; text-align: right;">({len(substations_in_zone)})</td>
            </tr>
            <tr style="height: 24px;">
                <td style="text-align: center;">
                    <svg width="40" height="28"><circle cx="20" cy="14" r="10" fill="#10B981" stroke="#047857" stroke-width="2"/></svg>
                </td>
                <td style="color: #334155; font-weight: 500;">Power Plants</td>
                <td style="color: #64748b; text-align: right;">({len(plants_in_zone)})</td>
            </tr>
        </table>
    </div>
</div>
'''
m.get_root().html.add_child(folium.Element(legend_html))

# Responsive overrides for the legend panel
legend_responsive_css = f'''
<style>
#map-legend table {{ font-size: 10px !important; }}
#map-legend tr {{ height: 20px !important; }}
#map-legend svg {{ transform: scale(0.85); transform-origin: left center; }}
@media (max-width: 700px) {{
    #map-legend {{ max-width: 220px !important; padding: 10px !important;
                   font-size: 10px !important; bottom: 6px !important; left: 6px !important; }}
}}
@media (max-height: 600px) {{
    #map-legend {{ max-height: calc(100vh - 160px); overflow-y: auto; bottom: 6px !important; }}
}}
</style>
'''
m.get_root().html.add_child(folium.Element(legend_responsive_css))

# =========================================================================
# GROUPED LAYER CONTROL PANEL (replaces default flat LayerControl)
# =========================================================================
grouped_panel_html = """
<style>
.leaflet-control-layers { display: none !important; }
/* ---- right-column shared width ---- */
:root { --rc-w: 240px; --rc-r: 10px; }
#layer-panel {
    position: fixed; top: 10px; right: var(--rc-r);
    background: rgba(255,255,255,0.97);
    border-radius: 8px;
    box-shadow: 0 3px 12px rgba(0,0,0,0.22);
    z-index: 10000;
    font-family: 'Segoe UI', Arial, sans-serif;
    font-size: 11px;
    width: var(--rc-w);
    border: 1.5px solid #cbd5e1;
    max-height: calc(100vh - 220px);
    overflow-y: auto;
}
#layer-panel .lp-title {
    padding: 8px 10px 6px;
    font-size: 12px; font-weight: 700;
    color: #0f172a;
    border-bottom: 1.5px solid #e2e8f0;
}
#layer-panel .lp-section {
    border-bottom: 1px solid #e2e8f0;
}
#layer-panel .lp-section:last-child { border-bottom: none; }
#layer-panel .lp-header {
    display: flex; justify-content: space-between; align-items: center;
    padding: 6px 10px; cursor: pointer;
    font-weight: 600; color: #1e293b; font-size: 11px;
    transition: background 0.15s;
    user-select: none;
}
#layer-panel .lp-header:hover { background: #f1f5f9; }
#layer-panel .lp-arrow {
    font-size: 9px; transition: transform 0.25s; color: #94a3b8;
}
#layer-panel .lp-section.collapsed .lp-arrow { transform: rotate(-90deg); }
#layer-panel .lp-section.collapsed .lp-body { display: none; }
#layer-panel .lp-body {
    padding: 2px 10px 6px;
}
#layer-panel .lp-body label {
    display: flex; align-items: center; gap: 5px;
    padding: 2px 0; cursor: pointer; color: #334155;
    font-size: 11px; line-height: 1.3; transition: color 0.15s;
}
#layer-panel .lp-body label:hover { color: #0f172a; }
#layer-panel .lp-body input[type="checkbox"] {
    width: 13px; height: 13px; cursor: pointer;
    accent-color: #4F46E5; flex-shrink: 0;
}
#layer-panel .lp-body .lp-sub {
    font-size: 9px; color: #94a3b8; font-weight: 600;
    text-transform: uppercase; letter-spacing: 0.5px;
    margin: 4px 0 1px; padding-top: 3px;
    border-top: 1px solid #f1f5f9;
}
#layer-panel .lp-body .lp-sub:first-child { border-top: none; margin-top: 0; }
/* ---- responsive: small viewports ---- */
@media (max-width: 700px) {
    :root { --rc-w: 190px; --rc-r: 6px; }
    #layer-panel .lp-body label { font-size: 10px; }
    #layer-panel .lp-header { font-size: 10px; padding: 5px 8px; }
}
@media (max-height: 600px) {
    #layer-panel { max-height: calc(100vh - 180px); }
}
</style>

<div id="layer-panel">
    <div class="lp-title">&#x1F5FA;&#xFE0F; Map Layers</div>

    <!-- Section 1: Impact Zones -->
    <div class="lp-section" id="sec-impact">
        <div class="lp-header" onclick="lpToggle('sec-impact')">
            &#x1F3AF; Impact Zones <span class="lp-arrow">&#x25BC;</span>
        </div>
        <div class="lp-body">
            <div class="lp-sub">Coastal</div>
            <label><input type="checkbox" data-layer="coastal-core" checked onchange="lpCheck(this)"> &#x1F30A; Coastal Core</label>
            <label><input type="checkbox" data-layer="coastal-surge" checked onchange="lpCheck(this)"> &#x1F30A; Coastal Surge</label>
            <div class="lp-sub">Swath</div>
            <label><input type="checkbox" data-layer="swath-inner" checked onchange="lpCheck(this)"> &#x1F534; Swath Inner Core</label>
            <label><input type="checkbox" data-layer="swath-mid" checked onchange="lpCheck(this)"> &#x1F7E0; Swath Mid-Level</label>
            <label><input type="checkbox" data-layer="swath-outer" checked onchange="lpCheck(this)"> &#x1F7E1; Swath Outer Extent</label>
        </div>
    </div>

    <!-- Section 2: Storm Track -->
    <div class="lp-section" id="sec-track">
        <div class="lp-header" onclick="lpToggle('sec-track')">
            &#x1F300; Storm Track <span class="lp-arrow">&#x25BC;</span>
        </div>
        <div class="lp-body">
            <label><input type="checkbox" data-layer="observed-track" onchange="lpCheck(this)"> &#x1F4CD; Observed Track</label>
            <label><input type="checkbox" data-layer="cone-uncertainty" onchange="lpCheck(this)"> &#x1F535; Cone of Uncertainty</label>
        </div>
    </div>

    <!-- Section 3: Infrastructure -->
    <div class="lp-section" id="sec-infra">
        <div class="lp-header" onclick="lpToggle('sec-infra')">
            &#x26A1; Infrastructure <span class="lp-arrow">&#x25BC;</span>
        </div>
        <div class="lp-body">
            <div class="lp-sub">Transmission Lines</div>
            <label><input type="checkbox" data-layer="tx-high" checked onchange="lpCheck(this)"> &#x26A1; &ge;345kV (High)</label>
            <label><input type="checkbox" data-layer="tx-med-high" onchange="lpCheck(this)"> &#x26A1; &ge;115kV (Med-High)</label>
            <label><input type="checkbox" data-layer="tx-med" onchange="lpCheck(this)"> &#x26A1; &ge;69kV (Medium)</label>
            <label><input type="checkbox" data-layer="tx-low" onchange="lpCheck(this)"> &#x26A1; &lt;69kV (Low)</label>
            <label><input type="checkbox" data-layer="tx-unknown" onchange="lpCheck(this)"> &#x26A1; Unknown Voltage</label>
            <div class="lp-sub">Facilities</div>
            <label><input type="checkbox" data-layer="substations" onchange="lpCheck(this)"> &#x1F50C; Substations</label>
            <label><input type="checkbox" data-layer="power-plants" onchange="lpCheck(this)"> &#x1F3ED; Power Plants</label>
        </div>
    </div>
</div>

<script>
/* ---- Unique substrings to locate each overlay in the native Leaflet
       LayerControl (hidden via CSS but still functional) ---- */
var LP_MATCH = {
    'coastal-core':     'Coastal Core',
    'coastal-surge':    'Coastal Surge',
    'swath-inner':      'Swath Inner Core',
    'swath-mid':        'Swath Mid-Level',
    'swath-outer':      'Swath Outer Extent',
    'observed-track':   'Observed Track',
    'cone-uncertainty': 'Cone of Uncertainty',
    'tx-high':          '345kV Transmission',
    'tx-med-high':      '115kV (Medium',
    'tx-med':           '69kV (Medium)',
    'tx-low':           '69kV (Low)',
    'tx-unknown':       'Unknown Voltage',
    'substations':      'Substations',
    'power-plants':     'Power Plants'
};

/* Find the hidden native checkbox for a given layer key */
function lpFindNative(key) {
    var match = LP_MATCH[key];
    if (!match) return null;
    var labels = document.querySelectorAll(
        '.leaflet-control-layers-overlays label');
    for (var i = 0; i < labels.length; i++) {
        var txt = labels[i].textContent || '';
        if (txt.indexOf(match) !== -1)
            return labels[i].querySelector('input');
    }
    return null;
}

/* Collapse / expand a section */
function lpToggle(sectionId) {
    var sec = document.getElementById(sectionId);
    if (sec) sec.classList.toggle('collapsed');
}

/* Guard against re-entrant calls during cone logic */
var _lpBusy = false;

/* Infrastructure keys and impact-zone keys */
var _infraKeys = ['tx-high','tx-med-high','tx-med','tx-low','tx-unknown',
                  'substations','power-plants'];
var _impactZoneKeys = ['coastal-core','coastal-surge',
                       'swath-inner','swath-mid','swath-outer'];
var _infraFirstUse = false;

/* Handle a custom-panel checkbox change */
function lpCheck(cb) {
    if (_lpBusy) return;
    var key  = cb.getAttribute('data-layer');
    var isCone = (key === 'cone-uncertainty');
    var isObs  = (key === 'observed-track');
    var isInfra = (_infraKeys.indexOf(key) !== -1);

    /* 0. First-ever infrastructure checkbox ON ➜ select all impact zones */
    if (isInfra && cb.checked && _infraFirstUse) {
        _infraFirstUse = false;
        _lpBusy = true;
        for (var z = 0; z < _impactZoneKeys.length; z++) {
            var zCb = document.querySelector(
                '#layer-panel input[data-layer="' + _impactZoneKeys[z] + '"]');
            if (zCb && !zCb.checked) {
                zCb.checked = true;
                var nz = lpFindNative(_impactZoneKeys[z]);
                if (nz && !nz.checked) nz.click();
            }
        }
        _lpBusy = false;
    }

    /* 1. Sync with the hidden native LayerControl */
    var nat = lpFindNative(key);
    if (nat && nat.checked !== cb.checked) nat.click();

    /* helper: turn on swath-outer so the map isn't empty */
    function _lpEnableSwathOuter() {
        var sw = document.querySelector(
            '#layer-panel input[data-layer="swath-outer"]');
        if (sw && !sw.checked) {
            sw.checked = true;
            var n = lpFindNative('swath-outer');
            if (n && !n.checked) n.click();
        }
    }

    /* 2. Cone of Uncertainty special behaviour */
    if (isCone && cb.checked) {
        /* Cone turned ON  ➜  deselect everything EXCEPT
           Observed Track (force ON) and Cone itself */
        _lpBusy = true;
        var all = document.querySelectorAll(
            '#layer-panel input[data-layer]');
        for (var i = 0; i < all.length; i++) {
            var k = all[i].getAttribute('data-layer');
            if (k === 'cone-uncertainty') continue;
            if (k === 'observed-track') {
                /* ensure observed track is ON */
                if (!all[i].checked) {
                    all[i].checked = true;
                    var n = lpFindNative(k);
                    if (n && !n.checked) n.click();
                }
            } else if (all[i].checked) {
                all[i].checked = false;
                var n = lpFindNative(k);
                if (n && n.checked) n.click();
            }
        }
        _lpBusy = false;
        return;
    }

    /* 2b. Cone manually deselected  ➜  restore swath outer extent */
    if (isCone && !cb.checked) {
        _lpBusy = true;
        _lpEnableSwathOuter();
        _lpBusy = false;
        return;
    }

    /* 3. Any non-observed, non-cone layer turned ON  ➜  deselect Cone
          and restore swath outer extent */
    if (!isCone && !isObs && cb.checked) {
        var coneCb = document.querySelector(
            '#layer-panel input[data-layer="cone-uncertainty"]');
        if (coneCb && coneCb.checked) {
            _lpBusy = true;
            coneCb.checked = false;
            var n = lpFindNative('cone-uncertainty');
            if (n && n.checked) n.click();
            _lpEnableSwathOuter();
            _lpBusy = false;
        }
    }
}
</script>
"""
m.get_root().html.add_child(folium.Element(grouped_panel_html))

# =========================================================================
# STORM ANIMATION (unchanged logic, updated labels)
# =========================================================================
track_json = json.dumps([{
    "lat": t["lat"], "lon": t["lon"], "time": t["time"], "hr": t["forecast_hr"]
} for t in track_data])

track_explorer_url = (f"{GEOCATALOG_URI}/explorer?d={TRACK_COLLECTION}"
                      if 'TRACK_COLLECTION' in dir() else "#")


animation_html = f'''
<script>
var trackPoints = {track_json};
var stormMarker = null;
var animationInterval = null;
var currentIndex = 0;
var isPlaying = false;
var animationSpeed = 500;
var mapReady = false;

function initStormAnimation() {{
    if (mapReady) return;
    var map = null;
    var mapDiv = document.querySelector('.folium-map');
    if (mapDiv && mapDiv._leaflet_id) {{
        for (var key in window) {{
            try {{
                if (window[key] && typeof window[key] === 'object' &&
                    window[key]._container === mapDiv) {{ map = window[key]; break; }}
            }} catch(e) {{}}
        }}
    }}
    if (!map) {{
        for (var key in window) {{
            try {{
                if (window[key] && window[key]._leaflet_id &&
                    typeof window[key].getCenter === 'function') {{ map = window[key]; break; }}
            }} catch(e) {{}}
        }}
    }}
    if (!map) {{ setTimeout(initStormAnimation, 500); return; }}
    mapReady = true;
    var stormIcon = L.divIcon({{
        className: 'storm-marker',
        html: '<span class="storm-icon">🌀</span>',
        iconSize: [50, 50], iconAnchor: [25, 25]
    }});
    stormMarker = L.marker([trackPoints[0].lat, trackPoints[0].lon], {{
        icon: stormIcon, zIndexOffset: 1000
    }}).addTo(map);
    stormMarker.bindPopup('<b>Hurricane {storm_name}</b><br>Ready to animate');
    updateTimeDisplay(0);
    document.getElementById('storm-status').textContent = '✓ Ready';
}}

function updateTimeDisplay(idx) {{
    var point = trackPoints[idx];
    var progress = (idx / (trackPoints.length - 1)) * 100;
    document.getElementById('progress-fill').style.width = progress + '%';
    document.getElementById('time-display').innerHTML = point.time + ' (T+' + point.hr + 'h)';
}}

function playAnimation() {{
    if (!mapReady || !stormMarker) return;
    if (isPlaying) return;
    isPlaying = true;
    document.getElementById('play-btn').style.background = '#4CAF50';
    document.getElementById('play-btn').style.color = 'white';
    document.getElementById('pause-btn').style.background = '#f0f0f0';
    document.getElementById('pause-btn').style.color = 'black';
    animationInterval = setInterval(function() {{
        if (currentIndex >= trackPoints.length - 1) currentIndex = 0;
        var point = trackPoints[currentIndex];
        stormMarker.setLatLng([point.lat, point.lon]);
        stormMarker.setPopupContent(
            '<b>🌀 Hurricane {storm_name}</b><br><b>' + point.time + '</b><br>T+' + point.hr + 'h');
        updateTimeDisplay(currentIndex);
        currentIndex++;
    }}, animationSpeed);
}}

function pauseAnimation() {{
    isPlaying = false;
    if (animationInterval) clearInterval(animationInterval);
    document.getElementById('play-btn').style.background = '#f0f0f0';
    document.getElementById('play-btn').style.color = 'black';
    document.getElementById('pause-btn').style.background = '#4CAF50';
    document.getElementById('pause-btn').style.color = 'white';
}}

function resetAnimation() {{
    pauseAnimation();
    currentIndex = 0;
    if (stormMarker) stormMarker.setLatLng([trackPoints[0].lat, trackPoints[0].lon]);
    updateTimeDisplay(0);
}}

function setSpeed(speed) {{
    animationSpeed = speed;
    if (isPlaying) {{ pauseAnimation(); isPlaying = false; playAnimation(); }}
}}

if (document.readyState === 'complete') {{ setTimeout(initStormAnimation, 1000); }}
else {{ window.addEventListener('load', function() {{ setTimeout(initStormAnimation, 1000); }}); }}
</script>

<style>
.storm-icon {{
    font-size: 45px;
    animation: pulse 1s ease-in-out infinite, spin 2s linear infinite;
    filter: drop-shadow(0 0 8px #ffffff) drop-shadow(0 0 15px #06B6D4)
            drop-shadow(0 0 25px #0891B2) drop-shadow(0 0 35px #0E7490);
}}
@keyframes pulse {{
    0%, 100% {{ transform: scale(1); opacity: 1; }}
    50% {{ transform: scale(1.3); opacity: 0.95; }}
}}
@keyframes spin {{
    0% {{ transform: rotate(0deg) scale(1); }}
    25% {{ transform: rotate(90deg) scale(1.15); }}
    50% {{ transform: rotate(180deg) scale(1.3); }}
    75% {{ transform: rotate(270deg) scale(1.15); }}
    100% {{ transform: rotate(360deg) scale(1); }}
}}
.storm-marker {{ background: transparent !important; border: none !important; }}
#storm-control {{
    position: fixed; bottom: 10px; right: var(--rc-r, 10px);
    padding: 10px; width: var(--rc-w, 240px); box-sizing: border-box;
    background: rgba(255,255,255,0.96); border-radius: 8px;
    box-shadow: 0 3px 12px rgba(0,0,0,0.22); z-index: 9999;
    font-family: 'Segoe UI', Arial, sans-serif; font-size: 11px;
    border: 1.5px solid #e2e8f0;
}}
#storm-control b {{ font-size: 12px; }}
#storm-control button {{
    margin: 2px; padding: 5px 9px; cursor: pointer; font-size: 11px;
    border: 1px solid #ccc; border-radius: 4px; background: #f0f0f0;
}}
#storm-control button:hover {{ background: #e0e0e0; }}
#storm-control select {{ font-size: 11px; padding: 3px 4px; }}
#progress-bar {{
    width: 100%; height: 6px; background: #ddd;
    border-radius: 3px; margin: 6px 0;
}}
#progress-fill {{
    height: 100%; background: linear-gradient(90deg, #7C3AED, #EC4899);
    border-radius: 3px; width: 0%; transition: width 0.2s;
}}
#time-display {{ font-weight: bold; text-align: center; margin-top: 3px; font-size: 11px; }}
#storm-status {{ font-size: 9px; color: #666; text-align: center; }}
.data-sources-wrapper {{
    margin-top: 6px; border-top: 1px solid #e2e8f0; padding-top: 6px;
}}
.data-sources-btn {{
    display: flex; justify-content: space-between; align-items: center;
    cursor: pointer; padding: 5px 7px;
    background: linear-gradient(135deg, #0078D4 0%, #106EBE 100%);
    color: white; border-radius: 4px; font-size: 10px; font-weight: 500;
    transition: all 0.2s;
}}
.data-sources-btn:hover {{
    background: linear-gradient(135deg, #106EBE 0%, #005A9E 100%);
    box-shadow: 0 2px 8px rgba(0,120,212,0.3);
}}
.expand-icon {{ font-size: 8px; transition: transform 0.3s; }}
.data-sources-wrapper.expanded .expand-icon {{ transform: rotate(180deg); }}
.data-sources-panel {{
    max-height: 0; overflow: hidden; transition: max-height 0.3s ease-out, padding 0.3s;
    background: #f8fafc; border-radius: 0 0 4px 4px; margin-top: 0;
}}
.data-sources-wrapper.expanded .data-sources-panel {{
    max-height: 350px; padding: 8px;
    border: 1px solid #e2e8f0; border-top: none;
}}
.source-header {{ font-size: 10px; font-weight: 600; color: #0078D4; margin-bottom: 6px; text-align: center; }}
.source-subheader {{ font-size: 9px; font-weight: 600; color: #64748b; margin-bottom: 5px; }}
.source-links {{ display: flex; flex-direction: column; gap: 3px; }}
.source-link {{
    display: flex; align-items: center; gap: 5px; padding: 4px 6px;
    background: white; border: 1px solid #e2e8f0; border-radius: 3px;
    color: #334155; text-decoration: none; font-size: 9px; transition: all 0.2s;
}}
.source-link:hover {{
    background: #0078D4; color: white; border-color: #0078D4; transform: translateX(2px);
}}
.source-icon {{ font-size: 11px; }}
.source-divider {{ height: 1px; background: #e2e8f0; margin: 6px 0; }}
@media (max-width: 700px) {{
    #storm-control {{ width: 190px; right: 6px; padding: 8px; font-size: 10px; }}
    #storm-control button {{ padding: 4px 7px; font-size: 10px; }}
}}
@media (max-height: 600px) {{
    #storm-control {{ bottom: 6px; padding: 8px; }}
}}
</style>

<div id="storm-control">
    <b>🌀 Storm Animation</b><br>
    <div style="margin: 4px 0;">
        <button id="play-btn" onclick="playAnimation()">▶ Play</button>
        <button id="pause-btn" onclick="pauseAnimation()">⏸ Pause</button>
        <button onclick="resetAnimation()">⏮ Reset</button>
    </div>
    <div>
        <label>Speed: </label>
        <select onchange="setSpeed(parseInt(this.value))">
            <option value="1000">Slow</option>
            <option value="500" selected>Normal</option>
            <option value="250">Fast</option>
            <option value="100">Very Fast</option>
        </select>
    </div>
    <div id="progress-bar"><div id="progress-fill"></div></div>
    <div id="time-display">Loading...</div>
    <div id="storm-status">Initializing...</div>
    <div class="data-sources-wrapper">
        <div class="data-sources-btn" onclick="toggleDataSources()">
            <span>ℹ️ Data Sources</span>
            <span class="expand-icon">▼</span>
        </div>
        <div id="data-sources-panel" class="data-sources-panel">
            <div class="source-header">Powered by Microsoft Planetary Computer Pro</div>
            <div class="source-links">
                <a href="{explorer_url}" target="_blank" class="source-link">
                    <span class="source-icon">📥</span><span>Hurricane Input Collection</span>
                </a>
                <a href="{forecast_explorer_url}" target="_blank" class="source-link">
                    <span class="source-icon">🌀</span><span>Hurricane Forecast Collection</span>
                </a>
                <a href="{track_explorer_url}" target="_blank" class="source-link">
                    <span class="source-icon">🗺️</span><span>Track Data GeoCatalog</span>
                </a>
                <a href="{infra_explorer_url_safe}" target="_blank" class="source-link">
                    <span class="source-icon">⚡</span><span>Infrastructure Impact GeoCatalog</span>
                </a>
            </div>
            <div class="source-divider"></div>
            <div class="source-subheader">Data Sources Used:</div>
            <div class="source-links">
                <a href="https://planetarycomputer.microsoft.com/dataset/ecmwf-forecast" target="_blank" class="source-link">
                    <span class="source-icon">🌤️</span><span>ECMWF Forecasts (via Planetary Computer)</span>
                </a>
                <a href="https://www.openstreetmap.org/" target="_blank" class="source-link">
                    <span class="source-icon">🗺️</span><span>OpenStreetMap (via Overpass API)</span>
                </a>
                <a href="https://www.ncei.noaa.gov/products/international-best-track-archive" target="_blank" class="source-link">
                    <span class="source-icon">🌊</span><span>IBTrACS (Best Track Archive)</span>
                </a>
                <a href="https://www.nhc.noaa.gov/" target="_blank" class="source-link">
                    <span class="source-icon">🇺🇸</span><span>NHC (National Hurricane Center)</span>
                </a>
                <a href="https://www.metoc.navy.mil/jtwc/jtwc.html" target="_blank" class="source-link">
                    <span class="source-icon">⚓</span><span>JTWC (Joint Typhoon Warning Center)</span>
                </a>
            </div>
        </div>
    </div>
</div>

<script>
function toggleDataSources() {{
    var panel = document.getElementById('data-sources-panel');
    var wrapper = panel.parentElement;
    wrapper.classList.toggle('expanded');
}}
</script>
'''
m.get_root().html.add_child(folium.Element(animation_html))

# ── Save and open ──
outputs_dir = Path("outputs")
outputs_dir.mkdir(exist_ok=True)
html_file = outputs_dir / f"hurricane_{storm_name.lower()}_{storm_year}_infrastructure_impact.html"
m.save(str(html_file))
print(f"\n✓ Map saved to: {html_file}")

import webbrowser
webbrowser.open(str(html_file))
print("✓ Opening in browser...")

print(f"\n{'='*60}")
print(f"Infrastructure Impact Summary (Swath + Coastal Zones)")
print(f"{'='*60}")
print(f"  Aurora forecast: T+0h to T+{total_forecast_hours}h ({len(track_data)} points)")
print(f"  Time range: {track_data[0]['time']} to {track_data[-1]['time']}")
print(f"\n  Power Lines in Impact Zone:")
print(f"    High voltage (≥345kV): {len(high_voltage_lines)} - VISIBLE")
print(f"    Medium-high (≥115kV):  {len(medium_high_lines)} - Toggle")
print(f"    Medium (≥69kV):        {len(medium_lines)} - Toggle")
print(f"    Low (<69kV):           {len(low_voltage_lines)} - Toggle")
print(f"    Unknown voltage:       {len(unknown_voltage_lines)} - Toggle")
print(f"\n  Facilities in Impact Zone:")
print(f"    Substations:           {len(substations_in_zone)} - Toggle")
print(f"    Power Plants:          {len(plants_in_zone)} - Toggle")
print(f"\n  GeoCatalog Collections:")
print(f"    Track Data:            {track_explorer_url[:80]}...")
print(f"    Infrastructure Impact: {infra_explorer_url_safe[:80] if infra_explorer_url_safe != '#' else 'N/A'}...")
print(f"{'='*60}")